<a href="https://colab.research.google.com/github/AhmedHamedAttia/COMPASS/blob/main/Cyber_Confidence_260930a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Human–Agent Cyber Confidence–Risk–Behavior (CRB)


**Purpose.** This notebook implements the experimental extension of the CRB paper as a **restart-safe, Google-Drive-backed** pipeline.

It validates the work in two complementary modules:

1. **Actual Cyber Risk Engine** — learns/calibrates technical phishing risk on the real **PhiUSIIL** benchmark (235,795 URLs; 54 features; associated with *Computers & Security*, 2024).
2. **Human Cyber-Calibration / Overconfidence Engine** — evaluates whether mismatch between **objective cyber risk** and **human perceived risk**, together with decision confidence, predicts unsafe behavior using a public human phishing dataset with task-level judgments.

It also includes:
- current/recent competitor registry;
- reproducible dataset acquisition;
- dataset/schema auditing;
- corrected calibration and overconfidence formulation;
- original-draft terms as baselines;
- grouped/repeated cross-validation to avoid participant leakage;
- Logistic Regression, Random Forest, HistGradientBoosting, and XGBoost baselines;
- AUROC, AUPRC, F1, MCC, balanced accuracy, sensitivity, specificity, Brier score, log-loss, ECE;
- participant-cluster bootstrap confidence intervals;
- GEE inferential statistics with robust repeated-measure handling;
- signal detection metrics (d′ and criterion);
- calibration/reliability analyses;
- adaptive-intervention **simulation** (explicitly not empirical intervention evidence);
- manuscript-ready CSV/LaTeX tables and PNG/PDF figures;
- a prospective real-user experiment template;
- PhiUSIIL shortcut-risk / leakage audit;
- URL-only robust feature benchmark;
- current external PhishTank + Tranco validation;
- task-aware CRB with training-only scenario priors;
- double-cold-start evaluation (unseen users + unseen tasks);
- personalized CRB onboarding simulation.

## Important scientific distinction

The public human datasets validate the **human calibration/behavior layer**. PhiUSIIL validates the **technical risk-estimation layer**. They are different samples and must **not** be merged row-by-row. End-to-end validation of the complete agentic system requires a future controlled user study in which the agent computes risk on the same tasks that participants judge.

## Main corrected formulation

Let \(R_t\in[0,1]\) be objective/agent-estimated cyber risk and \(P_t\in[0,1]\) the user's perceived risk.

\[
CE_t = |R_t-P_t|
\]

\[
OC_t = \max(0,R_t-P_t)
\]

\[
UC_t = \max(0,P_t-R_t)
\]

A hazardous overconfidence interaction is:

\[
HOC_t = OC_t \times Q_t
\]

where \(Q_t\) is the user's confidence in the decision. This separates **risk under-perception** from generic decision confidence.

---

### Key literature used to design the experiment

- Chen, Zahedi, Abbasi & Dobolyi — trust calibration in phishing-detection tools, *Information & Management*, 2021, DOI: 10.1016/j.im.2020.103394.
- Gallo et al. — Spamley / real user phishing behavior, *Computers & Security*, 2024, DOI: 10.1016/j.cose.2023.103671.
- Baltuttis & Teubner — visual risk indicators and trust/behavior, *Computers & Security*, 2024, DOI: 10.1016/j.cose.2024.103940.
- Prasad & Chandra — PhiUSIIL phishing risk benchmark, *Computers & Security*, 2024, DOI: 10.1016/j.cose.2023.103545.
- Security awareness, decision style, knowledge, and phishing detection, *Computers & Security*, 2025, DOI: 10.1016/j.cose.2024.104129.
- Bera & Kim — mindfulness, affect, information processing and phishing identification, *Information & Management*, 2025, DOI: 10.1016/j.im.2025.104110.
- Lu, Jiang, Head & Yang — phishing detection under multitasking; detection quality includes confidence, *European Journal of Information Systems*, Vol. 35 (2026), DOI: 10.1080/0960085X.2025.2548543.
- Malatji — Human–AI teaming in cybersecurity, *Human-Centric Intelligent Systems*, 2025, DOI: 10.1007/s44230-025-00103-8.
- Jamshidi — Human-Informed Neural Networks / trust-adaptive human supervision, *Human-Centric Intelligent Systems*, 2026, DOI: 10.1007/s44230-026-00173-2.

### Datasets

- **Primary human calibration dataset:** Kleitman, Law & Kay (2018), Figshare article **7205903**. It contains task-level phishing/genuine decisions, perceived maliciousness and decision confidence; the paper states Q1–Q20 are phishing and Q21–Q40 are genuine.
- **Technical risk dataset:** UCI **PhiUSIIL**, dataset id **967**.
- **Secondary current robustness source:** Threat Terminal / Human Detection of AI-Generated Phishing (2026; Zenodo record **19410549**) if a machine-readable table is present in the record at run time.
- **Additional real-world phishing simulation source:** Zenodo **18701713** if its table contains usable behavioral fields.
- **Spamley:** valuable external dataset but the *Computers & Security* article states data are available **on request**. The notebook supports an optional manually supplied Spamley table in the project `raw/spamley/` folder.


## Q1-strengthening protocol added in v5

This version deliberately separates **high in-dataset performance** from **deployment-relevant generalization**.

The technical risk layer is now evaluated under four progressively harder conditions:

\[
\text{Full PhiUSIIL}
\rightarrow
\text{Shortcut-risk ablation}
\rightarrow
\text{URL-only lexical model}
\rightarrow
\text{Current external PhishTank + Tranco}
\]

The human calibration layer is evaluated under three deployment protocols:

1. **Cold-start / known tasks:** unseen users, but tasks may have historical organizational statistics.
2. **Double cold-start:** unseen users **and** unseen tasks.
3. **Dual-adaptive onboarding:** a small calibration set builds a user-specific profile, while training-only task priors capture scenario difficulty before predicting later tasks.

These protocols are intended to strengthen external validity and novelty. **No result is forced to be superior.**
The manuscript must report the values actually obtained by the notebook and distinguish empirical results from simulation.


## Final robustness cycle added in v6

The v6 notebook adds the final robustness checks requested before manuscript rewriting.

### Technical external-validation refinements
1. **Fair domain-only benchmark:** PhishTank and Tranco are compared using only hostname/domain features that are available in the same way for both sources. Path, query, fragment, and scheme/HTTPS features are excluded.
2. **Zero-shot vs calibrated deployment:** the external domains are split into a calibration partition and a strictly untouched test partition. Source-trained models are first evaluated zero-shot. A one-dimensional Platt calibrator is then fitted only on the external calibration partition and evaluated on the untouched external test partition.
3. **No test-label threshold tuning:** the test partition is never used to fit the model, calibrator, or operating threshold.

### Human-layer robustness refinement
The Dual-Adaptive CRB onboarding experiment is repeated across **30 independent warm-up selections**. Each selection uses five phishing and five genuine warm-up tasks. The notebook reports mean ± SD across selections, bootstrap confidence intervals over warm-up selections, and paired tests for Dual-Adaptive CRB versus its direct baselines.

### Competitor-comparison policy
The notebook separates:
- **Direct head-to-head comparisons** performed on the same data and evaluation protocol; and
- **Contextual literature comparisons** that report published values from recent papers but are explicitly marked as not directly comparable when datasets/splits/tasks differ.

This prevents invalid claims of superiority while still positioning the contribution against recent high-reputation work.


## v6.1 correction

The v6 external-domain splitter was corrected after identifying an important edge case:
the same registrable domain can appear with different labels across the PhishTank and
Tranco sources. v6.1 removes all such conflicting domains before creating calibration
and test domain groups, then verifies strict domain disjointness. This makes the external
benchmark more conservative and more defensible for publication.


In [ ]:

# CELL 1 — Environment, Google Drive, packages, restart-safe helpers
# Safe to rerun. It creates folders and helper functions but never deletes prior results.

import os, sys, json, math, time, re, shutil, hashlib, warnings, subprocess, pkgutil
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ---------- Mount Google Drive when running in Colab ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path("/content/drive/MyDrive/Cyber_Confidence_Q1_Extension")
else:
    ROOT = Path.cwd() / "Cyber_Confidence_Q1_Extension"

DIRS = {
    "raw": ROOT / "raw",
    "processed": ROOT / "processed",
    "models": ROOT / "models",
    "figures": ROOT / "figures",
    "tables": ROOT / "tables",
    "reports": ROOT / "reports",
    "checkpoints": ROOT / "checkpoints",
    "logs": ROOT / "logs",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# ---------- Install only missing packages ----------
REQ = {
    "ucimlrepo": "ucimlrepo",
    "pyreadstat": "pyreadstat",
    "xgboost": "xgboost",
    "openpyxl": "openpyxl",
    "statsmodels": "statsmodels",
    "pyarrow": "pyarrow",
    "joblib": "joblib",
    "tldextract": "tldextract",
}
missing = [pip for mod, pip in REQ.items() if pkgutil.find_loader(mod) is None]
if missing:
    print("[INSTALL]", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import pandas as pd
import joblib
import requests
import matplotlib.pyplot as plt
import tldextract
from urllib.parse import urlsplit, parse_qsl

from scipy import stats
from scipy.special import expit, logit
from statsmodels.stats.multitest import multipletests

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, matthews_corrcoef,
    balanced_accuracy_score, brier_score_loss, log_loss, confusion_matrix,
    precision_score, recall_score, roc_curve, precision_recall_curve
)
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

SEED = 20260927
RNG = np.random.default_rng(SEED)
FORCE = False  # Set True only when you intentionally want to recompute everything.

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, block=1024*1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(block)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def artifact_ok(path, min_bytes=64):
    path = Path(path)
    return path.exists() and path.is_file() and path.stat().st_size >= min_bytes

def save_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    tmp.replace(path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def save_df(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        tmp = path.with_suffix(".tmp.parquet")
        df.to_parquet(tmp, index=index)
        tmp.replace(path)
    elif path.suffix.lower() == ".csv":
        tmp = path.with_suffix(".tmp.csv")
        df.to_csv(tmp, index=index)
        tmp.replace(path)
    elif path.suffix.lower() in [".xlsx", ".xls"]:
        tmp = path.with_name(path.stem + "_tmp" + path.suffix)
        df.to_excel(tmp, index=index)
        tmp.replace(path)
    else:
        raise ValueError(path)

def checkpoint(name, payload):
    payload = {"name": name, "time_utc": now(), **payload}
    save_json(payload, DIRS["checkpoints"] / f"{name}.json")

def load_or_run_df(path, fn, label):
    path = Path(path)
    if artifact_ok(path) and not FORCE:
        print(f"[LOAD] {label}: {path}")
        return pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    print(f"[RUN ] {label}")
    df = fn()
    save_df(df, path)
    checkpoint(label, {"artifact": str(path), "rows": len(df), "cols": len(df.columns)})
    return df

CONFIG = {
    "seed": SEED,
    "created_utc": now(),
    "root": str(ROOT),
    "force": FORCE,
    "primary_human_figshare_article": 7205903,
    "phiusiil_uci_id": 967,
    "secondary_zenodo_records": [19410549, 18701713],
    "cv_splits": 5,
    "cv_repeats": 5,
    "cluster_bootstrap_B": 1000,
}
save_json(CONFIG, ROOT / "config.json")

print("PROJECT ROOT:", ROOT)
print("XGBoost available:", HAVE_XGB)
print("Restart-safe mode:", "ON (load existing outputs)" if not FORCE else "FORCE RECOMPUTE")


Mounted at /content/drive
[INSTALL] ['ucimlrepo', 'pyreadstat', 'tldextract']
PROJECT ROOT: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension
XGBoost available: True
Restart-safe mode: ON (load existing outputs)


In [ ]:
# CELL 2 — Literature/competitor registry + dataset manifest
# v6 fix: canonical entries are MERGED into any existing restart-safe CSVs.
# This prevents an older saved manifest from hiding newly added 2026 competitors.

lit_path = DIRS["tables"] / "literature_competitors.csv"
data_path = DIRS["tables"] / "dataset_manifest.csv"

canonical_competitors = pd.DataFrame([
    {
        "year": 2021, "study": "Chen et al.",
        "venue": "Information & Management",
        "doi": "10.1016/j.im.2020.103394",
        "role": "Foundational trust-calibration comparator in phishing/security tools",
        "what_to_compare": "Calibration concept, reliance/performance consequences, controlled experiment",
        "reported_performance": "",
        "comparison_status": "Conceptual/contextual; different experiment"
    },
    {
        "year": 2024, "study": "Gallo et al. / Spamley",
        "venue": "Computers & Security",
        "doi": "10.1016/j.cose.2023.103671",
        "role": "Real-user phishing behavior benchmark",
        "what_to_compare": "Behavioral data collection, user profiling, phishing susceptibility",
        "reported_performance": "",
        "comparison_status": "Contextual; dataset available on request"
    },
    {
        "year": 2024, "study": "Baltuttis & Teubner",
        "venue": "Computers & Security",
        "doi": "10.1016/j.cose.2024.103940",
        "role": "Human-computer phishing risk indicator experiment",
        "what_to_compare": "Risk cues, trustworthiness ratings, response behavior, calibration trade-off",
        "reported_performance": "",
        "comparison_status": "Contextual; different human experiment"
    },
    {
        "year": 2024, "study": "Prasad & Chandra / PhiUSIIL",
        "venue": "Computers & Security",
        "doi": "10.1016/j.cose.2023.103545",
        "role": "Technical phishing-risk benchmark and source dataset",
        "what_to_compare": "PhiUSIIL phishing classification",
        "reported_performance": "Accuracy 99.24% fully incremental; 99.79% pre-training (authors' reported results)",
        "comparison_status": "Same source dataset but different training/split protocol; contextual numeric comparison"
    },
    {
        "year": 2025, "study": "Security awareness, decision style, knowledge, and phishing detection",
        "venue": "Computers & Security",
        "doi": "10.1016/j.cose.2024.104129",
        "role": "Recent human phishing-behavior comparator",
        "what_to_compare": "Awareness, knowledge, decision style, behavioral intentions/detection",
        "reported_performance": "",
        "comparison_status": "Contextual; different predictors and protocol"
    },
    {
        "year": 2025, "study": "Bera & Kim",
        "venue": "Information & Management",
        "doi": "10.1016/j.im.2025.104110",
        "role": "Recent high-reputation phishing-identification comparator",
        "what_to_compare": "Information processing and phishing identification",
        "reported_performance": "",
        "comparison_status": "Contextual; different human experiment"
    },
    {
        "year": 2026, "study": "Lu, Jiang, Head & Yang",
        "venue": "European Journal of Information Systems",
        "doi": "10.1080/0960085X.2025.2548543",
        "role": "Recent close behavioral comparator",
        "what_to_compare": "Phishing accuracy + confidence under multitasking/workload",
        "reported_performance": "",
        "comparison_status": "Contextual; different human experiment"
    },
    {
        "year": 2025, "study": "Malatji",
        "venue": "Human-Centric Intelligent Systems",
        "doi": "10.1007/s44230-025-00103-8",
        "role": "Same-target-journal human-AI cybersecurity comparator",
        "what_to_compare": "Human-AI cybersecurity architecture and feedback",
        "reported_performance": "",
        "comparison_status": "Architecture/contextual comparison"
    },
    {
        "year": 2026, "study": "Jamshidi / HINN",
        "venue": "Human-Centric Intelligent Systems",
        "doi": "10.1007/s44230-026-00173-2",
        "role": "Same-target-journal trust-adaptive human supervision comparator",
        "what_to_compare": "Dynamic trust weighting, human-informed learning, graceful fallback",
        "reported_performance": "Authors report empirical + simulation validation under noisy/intermittent supervision",
        "comparison_status": "Architecture/contextual; different task/data"
    },
    {
        "year": 2026, "study": "X-PHIDE cross-dataset validation",
        "venue": "IEEE Access",
        "doi": "10.1109/ACCESS.2026.3672690",
        "role": "Recent cross-dataset phishing detector / generalization benchmark",
        "what_to_compare": "Cross-dataset accuracy, feature stability, training efficiency",
        "reported_performance": "90.7% accuracy and 91.2% F1 on custom data; 77.8% average accuracy across three independent test sets",
        "comparison_status": "Contextual numeric comparison; different datasets/features/splits"
    },
    {
        "year": 2026, "study": "Chaudhuri & Mohankumar",
        "venue": "Scientific Reports",
        "doi": "10.1038/s41598-026-60046-3",
        "role": "Recent adversarial-resilient lightweight phishing detector",
        "what_to_compare": "Lexical/metadata robustness under URL evasion",
        "reported_performance": "",
        "comparison_status": "Methodological/contextual; exact metric table should be copied only from full paper"
    },
    {
        "year": 2026, "study": "Vu / PhishVN",
        "venue": "Data in Brief",
        "doi": "10.1016/j.dib.2026.113195",
        "role": "Recent time-stamped phishing benchmark aligned for cross-dataset study",
        "what_to_compare": "Temporal/domain-aware splits, label-confidence tiers, shortcut-aware evaluation",
        "reported_performance": "",
        "comparison_status": "Dataset/protocol comparator"
    },
])

canonical_datasets = pd.DataFrame([
    {
        "dataset": "PhiUSIIL Phishing URL",
        "source": "UCI", "id": "967", "status": "auto-download",
        "use": "Technical Actual Cyber Risk engine",
        "notes": "235,795 URLs, 54 features; label 0=phishing, 1=legitimate"
    },
    {
        "dataset": "DeceiverAndReceiver",
        "source": "Figshare", "id": "7205903", "status": "auto-download",
        "use": "Primary human calibration/overconfidence validation",
        "notes": "Task-level decisions + perceived maliciousness + confidence; Q1-20 phishing, Q21-40 genuine"
    },
    {
        "dataset": "Threat Terminal / Human Detection of AI-Generated Phishing",
        "source": "Zenodo", "id": "19410549",
        "status": "attempt-auto; optional if tabular file is available",
        "use": "Recent external calibration robustness",
        "notes": "2026 ongoing study; current record may provide only a PDF"
    },
    {
        "dataset": "Phishing simulation study",
        "source": "Zenodo", "id": "18701713", "status": "attempt-auto",
        "use": "Secondary behavioral robustness if compatible fields exist",
        "notes": "Anonymized study_data_merged.xlsx plus analysis code"
    },
    {
        "dataset": "Spamley",
        "source": "Authors / request", "id": "Gallo et al. 2024",
        "status": "manual optional",
        "use": "External human phishing validation",
        "notes": "Computers & Security article says data are available on request"
    },
    {
        "dataset": "Current PhishTank + Tranco external test",
        "source": "PhishTank / Tranco", "id": "live snapshot",
        "status": "auto-download in Cell 4B",
        "use": "Current external validation for source-trained URL/domain risk engines",
        "notes": "PhishTank verified-online phishing + Tranco benign domains; exact/domain overlap filtered where possible"
    },
    {
        "dataset": "PhishVN",
        "source": "Mendeley Data / Data in Brief",
        "id": "10.17632/b97hxbxtpd.4",
        "status": "optional external benchmark",
        "use": "2026 time-stamped cross-dataset robustness",
        "notes": "Useful future independent benchmark; authors explicitly discuss collection-artifact concerns"
    },
])

# Merge canonical rows into old restart-safe files. Canonical rows win on matching DOI/dataset,
# while any manually added unique rows are preserved.
if artifact_ok(lit_path) and not FORCE:
    try:
        old_comp = pd.read_csv(lit_path)
    except Exception:
        old_comp = pd.DataFrame()
else:
    old_comp = pd.DataFrame()

if len(old_comp):
    competitors = pd.concat([old_comp, canonical_competitors], ignore_index=True, sort=False)
    if "doi" in competitors.columns:
        competitors["_key"] = competitors["doi"].fillna("").astype(str).str.strip().str.lower()
        no_doi = competitors["_key"].eq("")
        competitors.loc[no_doi, "_key"] = (
            competitors.loc[no_doi, "study"].fillna("").astype(str).str.strip().str.lower()
        )
        competitors = competitors.drop_duplicates("_key", keep="last").drop(columns="_key")
else:
    competitors = canonical_competitors.copy()

if artifact_ok(data_path) and not FORCE:
    try:
        old_data = pd.read_csv(data_path)
    except Exception:
        old_data = pd.DataFrame()
else:
    old_data = pd.DataFrame()

if len(old_data):
    dataset_manifest = pd.concat([old_data, canonical_datasets], ignore_index=True, sort=False)
    dataset_manifest["_key"] = dataset_manifest["dataset"].fillna("").astype(str).str.strip().str.lower()
    dataset_manifest = dataset_manifest.drop_duplicates("_key", keep="last").drop(columns="_key")
else:
    dataset_manifest = canonical_datasets.copy()

save_df(competitors, lit_path)
save_df(dataset_manifest, data_path)
checkpoint("manifests_v6", {
    "literature_rows": len(competitors),
    "dataset_rows": len(dataset_manifest),
    "note": "Canonical v6 registry merged with any prior restart-safe rows."
})

print("[OK] Literature/dataset manifests merged and refreshed for v6.")
display(competitors.sort_values(["year","study"]))
display(dataset_manifest)

[OK] Literature/dataset manifests merged and refreshed for v6.


,year,study,venue,doi,role,what_to_compare,reported_performance,comparison_status
12,2021,Chen et al.,Information & Management,10.1016/j.im.2020.103394,Foundational trust-calibration comparator in p...,"Calibration concept, reliance/performance cons...",,Conceptual/contextual; different experiment
14,2024,Baltuttis & Teubner,Computers & Security,10.1016/j.cose.2024.103940,Human-computer phishing risk indicator experiment,"Risk cues, trustworthiness ratings, response b...",,Contextual; different human experiment
13,2024,Gallo et al. / Spamley,Computers & Security,10.1016/j.cose.2023.103671,Real-user phishing behavior benchmark,"Behavioral data collection, user profiling, ph...",,Contextual; dataset available on request
15,2024,Prasad & Chandra / PhiUSIIL,Computers & Security,10.1016/j.cose.2023.103545,Technical phishing-risk benchmark and source d...,PhiUSIIL phishing classification,Accuracy 99.24% fully incremental; 99.79% pre-...,Same source dataset but different training/spl...
17,2025,Bera & Kim,Information & Management,10.1016/j.im.2025.104110,Recent high-reputation phishing-identification...,Information processing and phishing identifica...,,Contextual; different human experiment
19,2025,Malatji,Human-Centric Intelligent Systems,10.1007/s44230-025-00103-8,Same-target-journal human-AI cybersecurity com...,Human-AI cybersecurity architecture and feedback,,Architecture/contextual comparison
16,2025,"Security awareness, decision style, knowledge,...",Computers & Security,10.1016/j.cose.2024.104129,Recent human phishing-behavior comparator,"Awareness, knowledge, decision style, behavior...",,Contextual; different predictors and protocol
22,2026,Chaudhuri & Mohankumar,Scientific Reports,10.1038/s41598-026-60046-3,Recent adversarial-resilient lightweight phish...,Lexical/metadata robustness under URL evasion,,Methodological/contextual; exact metric table ...
20,2026,Jamshidi / HINN,Human-Centric Intelligent Systems,10.1007/s44230-026-00173-2,Same-target-journal trust-adaptive human super...,"Dynamic trust weighting, human-informed learni...",Authors report empirical + simulation validati...,Architecture/contextual; different task/data
18,2026,"Lu, Jiang, Head & Yang",European Journal of Information Systems,10.1080/0960085X.2025.2548543,Recent close behavioral comparator,Phishing accuracy + confidence under multitask...,,Contextual; different human experiment


,dataset,source,id,status,use,notes
7,PhiUSIIL Phishing URL,UCI,967,auto-download,Technical Actual Cyber Risk engine,"235,795 URLs, 54 features; label 0=phishing, 1..."
8,DeceiverAndReceiver,Figshare,7205903,auto-download,Primary human calibration/overconfidence valid...,Task-level decisions + perceived maliciousness...
9,Threat Terminal / Human Detection of AI-Genera...,Zenodo,19410549,attempt-auto; optional if tabular file is avai...,Recent external calibration robustness,2026 ongoing study; current record may provide...
10,Phishing simulation study,Zenodo,18701713,attempt-auto,Secondary behavioral robustness if compatible ...,Anonymized study_data_merged.xlsx plus analysi...
11,Spamley,Authors / request,Gallo et al. 2024,manual optional,External human phishing validation,Computers & Security article says data are ava...
12,Current PhishTank + Tranco external test,PhishTank / Tranco,live snapshot,auto-download in Cell 4B,Current external validation for source-trained...,PhishTank verified-online phishing + Tranco be...
13,PhishVN,Mendeley Data / Data in Brief,10.17632/b97hxbxtpd.4,optional external benchmark,2026 time-stamped cross-dataset robustness,Useful future independent benchmark; authors e...


In [ ]:

# CELL 3 — Acquire real datasets (restart-safe) + schema audit
# Uses official APIs/repositories. Existing downloaded files are never re-downloaded unless FORCE=True.

import zipfile
import pyreadstat

def robust_get(url, timeout=120, stream=False):
    headers = {"User-Agent": "Mozilla/5.0 CRB-Q1-Research/1.0"}
    r = requests.get(url, headers=headers, timeout=timeout, stream=stream)
    r.raise_for_status()
    return r

def download_to(url, dest):
    dest = Path(dest)
    if artifact_ok(dest) and not FORCE:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    with robust_get(url, stream=True) as r:
        with tmp.open("wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
    tmp.replace(dest)
    return dest

def fetch_figshare_article(article_id, outdir):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    meta_path = outdir / "figshare_metadata.json"
    if artifact_ok(meta_path) and not FORCE:
        meta = load_json(meta_path)
    else:
        meta = robust_get(f"https://api.figshare.com/v2/articles/{article_id}").json()
        save_json(meta, meta_path)
    got = []
    for f in meta.get("files", []):
        name = f.get("name") or f"file_{f.get('id')}"
        url = f.get("download_url")
        if url:
            try:
                got.append(download_to(url, outdir / name))
            except Exception as e:
                print("[WARN] Figshare file failed:", name, e)
    return got

def fetch_zenodo_record(record_id, outdir):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    meta_path = outdir / "zenodo_metadata.json"
    try:
        if artifact_ok(meta_path) and not FORCE:
            meta = load_json(meta_path)
        else:
            meta = robust_get(f"https://zenodo.org/api/records/{record_id}").json()
            save_json(meta, meta_path)
        got = []
        for f in meta.get("files", []):
            name = f.get("key") or f.get("filename") or "zenodo_file"
            links = f.get("links", {})
            url = links.get("content") or links.get("self")
            if url:
                try:
                    got.append(download_to(url, outdir / name))
                except Exception as e:
                    print("[WARN] Zenodo file failed:", name, e)
        return got
    except Exception as e:
        print(f"[WARN] Zenodo {record_id} unavailable/blocked at run time:", e)
        return []

def extract_archives(root):
    root = Path(root)
    for z in list(root.rglob("*.zip")):
        out = z.parent / (z.stem + "_extracted")
        marker = out / ".done"
        if marker.exists() and not FORCE:
            continue
        out.mkdir(parents=True, exist_ok=True)
        try:
            with zipfile.ZipFile(z, "r") as zz:
                zz.extractall(out)
            marker.write_text(now(), encoding="utf-8")
        except Exception as e:
            print("[WARN] zip extraction failed:", z, e)

# 3A. Human dataset: Figshare
fig_dir = DIRS["raw"] / "figshare_7205903"
if not any(fig_dir.iterdir()) if fig_dir.exists() else True:
    pass
try:
    fig_files = fetch_figshare_article(7205903, fig_dir)
    print("Figshare files:", [p.name for p in fig_files])
except Exception as e:
    print("[ERROR] Figshare acquisition:", e)
    fig_files = list(fig_dir.glob("*"))

# 3B. Secondary Zenodo datasets
zen_files = {}
for zid in CONFIG["secondary_zenodo_records"]:
    zdir = DIRS["raw"] / f"zenodo_{zid}"
    zen_files[zid] = fetch_zenodo_record(zid, zdir)
    print(f"Zenodo {zid} files:", [p.name for p in zen_files[zid]])

# 3C. PhiUSIIL from UCI
phi_path = DIRS["raw"] / "phiusiil" / "phiusiil_full.parquet"
if artifact_ok(phi_path) and not FORCE:
    print("[LOAD] PhiUSIIL already saved:", phi_path)
else:
    print("[RUN ] Download PhiUSIIL from UCI id=967")
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=967)
    X = ds.data.features.copy()
    y = ds.data.targets.copy()
    phi = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
    phi_path.parent.mkdir(parents=True, exist_ok=True)
    save_df(phi, phi_path)
    save_json({"metadata": str(ds.metadata), "variables": str(ds.variables)}, phi_path.parent / "uci_metadata.json")
    checkpoint("phiusiil_download", {"rows": len(phi), "cols": len(phi.columns), "artifact": str(phi_path)})
    del X, y, phi

# Optional Spamley folder: user can put any CSV/XLSX/SAV file here later.
spamley_dir = DIRS["raw"] / "spamley"
spamley_dir.mkdir(parents=True, exist_ok=True)

extract_archives(DIRS["raw"])

def load_tabular_file(path):
    path = Path(path)
    s = path.suffix.lower()
    if s == ".csv":
        for enc in ["utf-8", "utf-8-sig", "latin1"]:
            try:
                return pd.read_csv(path, encoding=enc)
            except Exception:
                pass
    if s in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if s == ".parquet":
        return pd.read_parquet(path)
    if s == ".sav":
        return pyreadstat.read_sav(path)[0]
    if s == ".dta":
        return pd.read_stata(path)
    if s == ".json":
        try:
            return pd.read_json(path)
        except Exception:
            return None
    return None

def audit_raw_tables():
    rows = []
    exts = {".csv",".xlsx",".xls",".parquet",".sav",".dta",".json"}
    for p in DIRS["raw"].rglob("*"):
        if not p.is_file() or p.suffix.lower() not in exts:
            continue
        # skip metadata JSON
        if "metadata" in p.name.lower():
            continue
        try:
            df = load_tabular_file(p)
            if df is None:
                continue
            rows.append({
                "file": str(p),
                "rows": len(df), "cols": len(df.columns),
                "columns_preview": " | ".join(map(str, list(df.columns)[:30])),
                "size_mb": round(p.stat().st_size/1024**2, 3),
            })
        except Exception as e:
            rows.append({"file": str(p), "rows": -1, "cols": -1, "columns_preview": f"ERROR: {e}", "size_mb": 0})
    return pd.DataFrame(rows).sort_values(["rows","cols"], ascending=False) if rows else pd.DataFrame()

audit_path = DIRS["reports"] / "raw_schema_audit.csv"
if artifact_ok(audit_path) and not FORCE:
    raw_audit = pd.read_csv(audit_path)
    print("[LOAD] raw schema audit")
else:
    raw_audit = audit_raw_tables()
    save_df(raw_audit, audit_path)
    checkpoint("raw_schema_audit", {"tables": len(raw_audit)})

display(raw_audit.head(20))
print("Put a requested Spamley file here if obtained later:", spamley_dir)


Figshare files: ['ReceiverAndDeceiver_DataSet.zip']
Zenodo 19410549 files: ['altiparmak_2026_preliminary_findings_threat_terminal.pdf']
Zenodo 18701713 files: ['study_data_merged.xlsx', 'Repeat_clicker_analysis_5.Rmd', 'Repeat_clicker_analysis_5.html']
[LOAD] PhiUSIIL already saved: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/raw/phiusiil/phiusiil_full.parquet
[LOAD] raw schema audit


,file,rows,cols,columns_preview,size_mb
0,/content/drive/MyDrive/Cyber_Confidence_Q1_Ext...,235795,55,URL | URLLength | Domain | DomainLength | IsDo...,21.197
1,/content/drive/MyDrive/Cyber_Confidence_Q1_Ext...,1625,115,...1 | StartDate | EndDate | Duration (in seco...,0.889
2,/content/drive/MyDrive/Cyber_Confidence_Q1_Ext...,429,3,key | value | Type,0.042
3,/content/drive/MyDrive/Cyber_Confidence_Q1_Ext...,150,524,id | Age | LvlofEdu | YearsinAus | CountryofOr...,0.216
4,/content/drive/MyDrive/Cyber_Confidence_Q1_Ext...,69,3,Columns | Variable | Definition,0.006


Put a requested Spamley file here if obtained later: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/raw/spamley


In [ ]:

# CELL 4 — Technical Actual-Risk Engine on PhiUSIIL (restart-safe)
# Domain-disjoint holdout prevents the easiest form of website/domain memorization.
# Output: calibrated phishing probabilities that can serve as R_t in deployment.

risk_pred_path = DIRS["processed"] / "phiusiil_risk_holdout_predictions.parquet"
risk_metrics_path = DIRS["tables"] / "phiusiil_risk_metrics.csv"
risk_model_path = DIRS["models"] / "phiusiil_best_risk_model.joblib"

if all(artifact_ok(p) for p in [risk_pred_path, risk_metrics_path, risk_model_path]) and not FORCE:
    print("[LOAD] Risk engine outputs")
    risk_holdout = pd.read_parquet(risk_pred_path)
    risk_metrics = pd.read_csv(risk_metrics_path)
    risk_model = joblib.load(risk_model_path)
else:
    print("[RUN ] PhiUSIIL risk engine")
    phi = pd.read_parquet(phi_path)

    # Find target safely.
    target_candidates = [c for c in phi.columns if str(c).lower() in {"label","class","target","phishing"}]
    if not target_candidates:
        target_candidates = [c for c in phi.columns if "label" in str(c).lower()]
    assert target_candidates, "Could not identify PhiUSIIL target label."
    target_col = target_candidates[-1]

    # UCI documentation: Label 1=legitimate, 0=phishing. Convert to 1=phishing.
    y_raw = pd.to_numeric(phi[target_col], errors="coerce")
    y = (y_raw == 0).astype(int)

    # Domain grouping if available; otherwise hostname proxy from URL.
    if "Domain" in phi.columns:
        groups = phi["Domain"].astype(str).fillna("missing")
    elif "URL" in phi.columns:
        groups = phi["URL"].astype(str).str.extract(r"^(?:https?://)?([^/]+)", expand=False).fillna("missing")
    else:
        groups = pd.Series(np.arange(len(phi)).astype(str), index=phi.index)

    # Numeric features only for transparent/reproducible benchmark.
    drop_cols = {target_col, "FILENAME", "URL", "Domain", "TLD", "Title"}
    num_cols = [c for c in phi.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(phi[c])]
    X = phi[num_cols].replace([np.inf,-np.inf], np.nan)

    # Domain-disjoint holdout.
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    tr_idx, te_idx = next(gss.split(X, y, groups))
    Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
    ytr, yte = y.iloc[tr_idx], y.iloc[te_idx]
    gte = groups.iloc[te_idx]

    models = {
        "LogisticRegression": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1500, class_weight="balanced", n_jobs=None, random_state=SEED))
        ]),
        "RandomForest": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=350, min_samples_leaf=2, class_weight="balanced_subsample",
                n_jobs=-1, random_state=SEED
            ))
        ]),
        "HistGradientBoosting": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, random_state=SEED))
        ]),
    }
    if HAVE_XGB:
        models["XGBoost"] = Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=350, max_depth=6, learning_rate=0.06,
                subsample=0.85, colsample_bytree=0.85,
                eval_metric="logloss", random_state=SEED, n_jobs=-1
            ))
        ])

    def binary_metrics(ytrue, prob, threshold=0.5):
        pred = (np.asarray(prob) >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(ytrue, pred, labels=[0,1]).ravel()
        return {
            "AUROC": roc_auc_score(ytrue, prob),
            "AUPRC": average_precision_score(ytrue, prob),
            "F1": f1_score(ytrue, pred, zero_division=0),
            "MCC": matthews_corrcoef(ytrue, pred),
            "BalancedAccuracy": balanced_accuracy_score(ytrue, pred),
            "Sensitivity": tp/(tp+fn) if tp+fn else np.nan,
            "Specificity": tn/(tn+fp) if tn+fp else np.nan,
            "Brier": brier_score_loss(ytrue, prob),
            "LogLoss": log_loss(ytrue, np.clip(prob,1e-8,1-1e-8)),
        }

    result_rows, pred_frames, fitted = [], [], {}
    for name, model in models.items():
        print(" fitting", name)
        model.fit(Xtr, ytr)
        prob = model.predict_proba(Xte)[:,1]
        m = binary_metrics(yte, prob)
        m["Model"] = name
        result_rows.append(m)
        pred_frames.append(pd.DataFrame({
            "row_id": phi.index[te_idx].to_numpy(),
            "domain_group": gte.astype(str).to_numpy(),
            "y_true": yte.to_numpy(),
            "model": name,
            "prob_phishing": prob
        }))
        fitted[name] = model

    risk_metrics = pd.DataFrame(result_rows).sort_values(["AUPRC","Brier"], ascending=[False,True])
    risk_holdout = pd.concat(pred_frames, ignore_index=True)
    best_name = risk_metrics.iloc[0]["Model"]

    # Calibrate best model using CV on training data; preserve independent holdout.
    # Ensemble=False keeps a compact final estimator.
    best_base = clone(models[best_name])
    calibrated = CalibratedClassifierCV(best_base, method="sigmoid", cv=3, ensemble=False)
    calibrated.fit(Xtr, ytr)
    cal_prob = calibrated.predict_proba(Xte)[:,1]
    cal_metrics = binary_metrics(yte, cal_prob)
    cal_metrics["Model"] = best_name + "_Calibrated"
    risk_metrics = pd.concat([risk_metrics, pd.DataFrame([cal_metrics])], ignore_index=True)
    risk_holdout = pd.concat([
        risk_holdout,
        pd.DataFrame({
            "row_id": phi.index[te_idx].to_numpy(),
            "domain_group": gte.astype(str).to_numpy(),
            "y_true": yte.to_numpy(),
            "model": best_name + "_Calibrated",
            "prob_phishing": cal_prob
        })
    ], ignore_index=True)

    save_df(risk_holdout, risk_pred_path)
    save_df(risk_metrics, risk_metrics_path)
    joblib.dump({"model": calibrated, "features": num_cols, "target_col": target_col}, risk_model_path)
    risk_model = {"model": calibrated, "features": num_cols, "target_col": target_col}
    checkpoint("phiusiil_risk_engine", {
        "train_rows": len(tr_idx), "test_rows": len(te_idx),
        "unique_test_domains": int(pd.Series(gte).nunique()),
        "features": len(num_cols), "best_base": best_name
    })
    del phi, X, Xtr, Xte, y, ytr, yte

display(risk_metrics.sort_values("AUPRC", ascending=False))

# Plot ROC + calibration for the calibrated model
fig_path = DIRS["figures"] / "phiusiil_risk_engine_roc_calibration.png"
if not artifact_ok(fig_path) or FORCE:
    best_cal = risk_metrics.loc[risk_metrics["Model"].str.endswith("_Calibrated"), "Model"].iloc[0]
    d = risk_holdout[risk_holdout.model == best_cal]
    fpr, tpr, _ = roc_curve(d.y_true, d.prob_phishing)
    frac_pos, mean_pred = calibration_curve(d.y_true, d.prob_phishing, n_bins=10, strategy="quantile")

    plt.figure(figsize=(6.5,5))
    plt.plot(fpr, tpr, label=f"{best_cal} AUROC={roc_auc_score(d.y_true,d.prob_phishing):.3f}")
    plt.plot([0,1],[0,1],"--", linewidth=1)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("PhiUSIIL Risk Engine — Domain-Disjoint Holdout")
    plt.legend(); plt.tight_layout()
    plt.savefig(DIRS["figures"] / "phiusiil_risk_engine_roc.pdf", bbox_inches="tight")
    plt.savefig(DIRS["figures"] / "phiusiil_risk_engine_roc.png", dpi=220, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(6.5,5))
    plt.plot(mean_pred, frac_pos, marker="o")
    plt.plot([0,1],[0,1],"--", linewidth=1)
    plt.xlabel("Predicted phishing probability"); plt.ylabel("Observed phishing frequency")
    plt.title("Risk Probability Calibration")
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "phiusiil_risk_engine_calibration.pdf", bbox_inches="tight")
    plt.savefig(fig_path, dpi=220, bbox_inches="tight")
    plt.close()

print("Saved risk-engine figures in:", DIRS["figures"])


[LOAD] Risk engine outputs


,AUROC,AUPRC,F1,MCC,BalancedAccuracy,Sensitivity,Specificity,Brier,LogLoss,Model
0,1.0,1.0,0.999977,0.999958,0.999977,0.999954,1.0,6.091256e-05,0.000983,RandomForest
1,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.0,1.352218e-09,0.000031,HistGradientBoosting
2,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.0,1.165820e-06,0.000017,XGBoost
4,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.0,1.591341e-07,0.000016,RandomForest_Calibrated
3,1.0,1.0,0.999908,0.999833,0.999908,0.999816,1.0,6.584402e-05,0.000389,LogisticRegression


Saved risk-engine figures in: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/figures


In [ ]:
# CELL 4A — PhiUSIIL shortcut-risk audit and ablation
# Goal: determine whether near-perfect in-dataset performance is driven by one/few
# dataset-specific features. High single-feature predictivity is flagged as "shortcut risk",
# not automatically called leakage.
#
# Restart-safe outputs:
#   tables/phiusiil_univariate_shortcut_audit.csv
#   tables/phiusiil_shortcut_ablation_metrics.csv
#   figures/phiusiil_shortcut_audit_top20.png

audit_csv = DIRS["tables"] / "phiusiil_univariate_shortcut_audit.csv"
ablation_csv = DIRS["tables"] / "phiusiil_shortcut_ablation_metrics.csv"
audit_fig = DIRS["figures"] / "phiusiil_shortcut_audit_top20.png"

if artifact_ok(audit_csv) and artifact_ok(ablation_csv) and artifact_ok(audit_fig) and not FORCE:
    print("[LOAD] PhiUSIIL shortcut-risk audit")
    shortcut_audit = pd.read_csv(audit_csv)
    shortcut_ablation = pd.read_csv(ablation_csv)
else:
    print("[RUN ] PhiUSIIL shortcut-risk audit")
    phi = pd.read_parquet(phi_path)

    target_candidates = [c for c in phi.columns if str(c).lower() in {"label","class","target","phishing"}]
    if not target_candidates:
        target_candidates = [c for c in phi.columns if "label" in str(c).lower()]
    target_col = target_candidates[-1]
    y = (pd.to_numeric(phi[target_col], errors="coerce") == 0).astype(int)  # 1=phishing

    if "Domain" in phi.columns:
        groups = phi["Domain"].astype(str).fillna("missing")
    elif "URL" in phi.columns:
        groups = phi["URL"].astype(str).str.extract(r"^(?:https?://)?([^/]+)", expand=False).fillna("missing")
    else:
        groups = pd.Series(np.arange(len(phi)).astype(str), index=phi.index)

    drop_cols = {target_col, "FILENAME", "URL", "Domain", "TLD", "Title"}
    num_cols = [c for c in phi.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(phi[c])]
    X = phi[num_cols].replace([np.inf,-np.inf], np.nan)

    # Same domain-disjoint holdout protocol used in Cell 4.
    gss = GroupShuffleSplit(n_splits=1, test_size=.20, random_state=SEED)
    tr_idx, te_idx = next(gss.split(X, y, groups))
    Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
    ytr, yte = y.iloc[tr_idx], y.iloc[te_idx]

    rows = []
    for c in num_cols:
        s = pd.to_numeric(Xtr[c], errors="coerce")
        med = s.median()
        ss = s.fillna(med)
        if ss.nunique() < 2:
            auc = .5
        else:
            try:
                auc0 = roc_auc_score(ytr, ss)
                auc = max(auc0, 1-auc0)
            except Exception:
                auc = np.nan
        m0 = ss[ytr.to_numpy()==0].mean()
        m1 = ss[ytr.to_numpy()==1].mean()
        rows.append({
            "feature": c,
            "univariate_abs_AUROC": auc,
            "legitimate_mean": m0,
            "phishing_mean": m1,
            "missing_rate": float(s.isna().mean()),
            "unique_values": int(s.nunique(dropna=True)),
        })

    shortcut_audit = pd.DataFrame(rows).sort_values("univariate_abs_AUROC", ascending=False)
    shortcut_audit["shortcut_risk_flag"] = shortcut_audit["univariate_abs_AUROC"] >= .98
    save_df(shortcut_audit, audit_csv)

    flagged = shortcut_audit.loc[shortcut_audit.shortcut_risk_flag, "feature"].tolist()
    https_like = [c for c in num_cols if "https" in str(c).lower() or "ssl" in str(c).lower()]

    feature_sets = {
        "FullNumeric": num_cols,
        "NoHTTPS_SSL": [c for c in num_cols if c not in https_like],
        "NoShortcutRiskFeatures": [c for c in num_cols if c not in flagged],
    }

    def fit_eval_xgb_or_hgb(cols):
        if not cols:
            return None
        if HAVE_XGB:
            model = Pipeline([
                ("imp",SimpleImputer(strategy="median")),
                ("clf",XGBClassifier(
                    n_estimators=300,max_depth=5,learning_rate=.06,
                    subsample=.85,colsample_bytree=.85,eval_metric="logloss",
                    random_state=SEED,n_jobs=-1
                ))
            ])
        else:
            model = Pipeline([
                ("imp",SimpleImputer(strategy="median")),
                ("clf",HistGradientBoostingClassifier(max_iter=250,learning_rate=.06,random_state=SEED))
            ])
        model.fit(Xtr[cols], ytr)
        p = model.predict_proba(Xte[cols])[:,1]
        pred = (p>=.5).astype(int)
        tn,fp,fn,tp = confusion_matrix(yte,pred,labels=[0,1]).ravel()
        return {
            "AUROC":roc_auc_score(yte,p),
            "AUPRC":average_precision_score(yte,p),
            "F1":f1_score(yte,pred),
            "MCC":matthews_corrcoef(yte,pred),
            "BalancedAccuracy":balanced_accuracy_score(yte,pred),
            "Brier":brier_score_loss(yte,p),
            "Sensitivity":tp/(tp+fn) if tp+fn else np.nan,
            "Specificity":tn/(tn+fp) if tn+fp else np.nan,
        }

    ar = []
    for name, cols in feature_sets.items():
        mm = fit_eval_xgb_or_hgb(cols)
        if mm:
            ar.append({
                "FeatureSet":name,
                "n_features":len(cols),
                "removed_features":len(num_cols)-len(cols),
                **mm
            })

    shortcut_ablation = pd.DataFrame(ar)
    save_df(shortcut_ablation, ablation_csv)

    top = shortcut_audit.head(20).sort_values("univariate_abs_AUROC")
    plt.figure(figsize=(8,6))
    plt.barh(top.feature.astype(str), top.univariate_abs_AUROC)
    plt.axvline(.98, linestyle="--", linewidth=1)
    plt.xlabel("Single-feature absolute AUROC")
    plt.title("PhiUSIIL Shortcut-Risk Audit — Top 20 Features")
    plt.tight_layout()
    plt.savefig(audit_fig,dpi=220,bbox_inches="tight")
    plt.savefig(DIRS["figures"]/"phiusiil_shortcut_audit_top20.pdf",bbox_inches="tight")
    plt.close()

    save_json({
        "flag_threshold":.98,
        "flagged_features":flagged,
        "https_ssl_features":https_like,
        "important_note":"A high univariate AUROC is a shortcut-risk indicator, not proof of leakage."
    }, DIRS["reports"]/"phiusiil_shortcut_audit.json")

    checkpoint("phiusiil_shortcut_audit", {
        "numeric_features":len(num_cols),
        "shortcut_risk_features":len(flagged),
        "https_ssl_features":len(https_like)
    })

display(shortcut_audit.head(20))
display(shortcut_ablation)

[LOAD] PhiUSIIL shortcut-risk audit


,feature,univariate_abs_AUROC,legitimate_mean,phishing_mean,missing_rate,unique_values,shortcut_risk_flag
0,URLSimilarityIndex,0.996093,100.000000,50.143761,0.0,30651,True
1,LineOfCode,0.990742,1949.551475,64.402126,0.0,9908,True
2,NoOfExternalRef,0.987818,85.184899,1.140172,0.0,1107,True
3,NoOfImage,0.979587,44.754329,0.862920,0.0,906,False
4,NoOfSelfRef,0.971824,113.323231,0.539395,0.0,1285,False
5,NoOfJS,0.970604,17.760950,0.908740,0.0,243,False
6,NoOfCSS,0.957624,10.769413,0.438996,0.0,194,False
7,HasSocialNet,0.894671,0.794516,0.005175,0.0,2,False
8,HasCopyrightInfo,0.876875,0.808267,0.054516,0.0,2,False
9,HasDescription,0.846618,0.737365,0.044128,0.0,2,False


,FeatureSet,n_features,removed_features,AUROC,AUPRC,F1,MCC,BalancedAccuracy,Brier,Sensitivity,Specificity
0,FullNumeric,50,0,1.000000,1.000000,1.000000,1.000000,1.000000,0.000001,1.000000,1.000000
1,NoHTTPS_SSL,49,1,0.999999,0.999999,0.999954,0.999917,0.999954,0.000044,0.999908,1.000000
2,NoShortcutRiskFeatures,47,3,1.000000,1.000000,0.999724,0.999500,0.999750,0.000176,0.999724,0.999777


In [ ]:
# CELL 4B — Robust URL-only risk engine + CURRENT external validation
#
# Training source: PhiUSIIL raw URLs.
# External test: current verified PhishTank URLs + current Tranco benign domains.
#
# This avoids dataset-specific metadata and uses only features computable from a URL string.
# External URLs are de-duplicated against PhiUSIIL and domain overlap is removed where feasible.
#
# Restart-safe outputs:
#   processed/phiusiil_lexical_features.parquet
#   processed/external_phishtank_tranco_urls.parquet
#   processed/external_lexical_features.parquet
#   tables/lexical_risk_internal_metrics.csv
#   tables/lexical_risk_external_metrics.csv
#   tables/lexical_risk_deployment10_metrics.csv

lex_phi_path = DIRS["processed"] / "phiusiil_lexical_features.parquet"
ext_url_path = DIRS["processed"] / "external_phishtank_tranco_urls.parquet"
ext_feat_path = DIRS["processed"] / "external_lexical_features.parquet"
lex_int_metrics_path = DIRS["tables"] / "lexical_risk_internal_metrics.csv"
lex_ext_metrics_path = DIRS["tables"] / "lexical_risk_external_metrics.csv"
lex_dep_metrics_path = DIRS["tables"] / "lexical_risk_deployment10_metrics.csv"
lex_models_path = DIRS["models"] / "lexical_external_models.joblib"

_extract = tldextract.TLDExtract(suffix_list_urls=())

def normalize_url(u):
    u = "" if pd.isna(u) else str(u).strip()
    if not re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", u):
        u = "http://" + u
    return u

def registrable_domain(u):
    try:
        h = (urlsplit(normalize_url(u)).hostname or "").lower()
        ex = _extract(h)
        return ".".join(x for x in [ex.domain,ex.suffix] if x) or h
    except Exception:
        return ""

def shannon_entropy(s):
    from collections import Counter
    s = str(s)
    if not s:
        return 0.0
    n = len(s)
    cnt = Counter(s)
    return float(-sum((v/n)*math.log2(v/n) for v in cnt.values()))

SUSPICIOUS_TOKENS = [
    "login","verify","secure","account","update","signin","bank","wallet",
    "confirm","password","support","recover","payment","invoice","bonus","gift"
]
SHORTENERS = {
    "bit.ly","tinyurl.com","t.co","goo.gl","ow.ly","is.gd","buff.ly",
    "cutt.ly","tiny.cc","rebrand.ly"
}

def lexical_row(u):
    raw = "" if pd.isna(u) else str(u).strip()
    norm = normalize_url(raw)
    try:
        p = urlsplit(norm)
    except Exception:
        p = urlsplit("http://invalid/")
    host = (p.hostname or "").lower()
    path = p.path or ""
    query = p.query or ""
    full = raw.lower()
    ex = _extract(host)
    regdom = ".".join(x for x in [ex.domain,ex.suffix] if x)
    sub = ex.subdomain or ""
    digits = sum(ch.isdigit() for ch in raw)
    letters = sum(ch.isalpha() for ch in raw)
    alnum = sum(ch.isalnum() for ch in raw)
    special = max(len(raw)-alnum,0)
    dots = raw.count(".")
    qparams = len(parse_qsl(query, keep_blank_values=True))
    ipv4 = bool(re.fullmatch(r"\d{1,3}(?:\.\d{1,3}){3}", host))
    ipv6 = ":" in host and all(c in "0123456789abcdef:" for c in host)
    try:
        has_port = int(p.port is not None) if host else 0
    except Exception:
        has_port = 0
    return {
        "registrable_domain": regdom,
        "url_length":len(raw),
        "host_length":len(host),
        "path_length":len(path),
        "query_length":len(query),
        "fragment_length":len(p.fragment or ""),
        "num_dots":dots,
        "num_hyphens":raw.count("-"),
        "num_underscores":raw.count("_"),
        "num_slashes":raw.count("/"),
        "num_question":raw.count("?"),
        "num_equal":raw.count("="),
        "num_ampersand":raw.count("&"),
        "num_at":raw.count("@"),
        "num_percent":raw.count("%"),
        "num_digits":digits,
        "digit_ratio":digits/max(len(raw),1),
        "letter_ratio":letters/max(len(raw),1),
        "special_ratio":special/max(len(raw),1),
        "entropy":shannon_entropy(raw),
        "subdomain_count":0 if not sub else sub.count(".")+1,
        "path_depth":len([x for x in path.split("/") if x]),
        "query_param_count":qparams,
        "has_ip_host":int(ipv4 or ipv6),
        "has_port":has_port,
        "has_punycode":int("xn--" in host),
        "uses_https":int((p.scheme or "").lower()=="https"),
        "suspicious_token_count":sum(tok in full for tok in SUSPICIOUS_TOKENS),
        "is_shortener":int(regdom in SHORTENERS),
        "tld_length":len(ex.suffix or ""),
        "domain_length":len(ex.domain or ""),
    }

def featurize_urls(df, url_col="url", label_col="label"):
    feats = pd.DataFrame([lexical_row(u) for u in df[url_col].astype(str)])
    feats.insert(0,"url",df[url_col].astype(str).to_numpy())
    if label_col in df.columns:
        feats.insert(1,"label",pd.to_numeric(df[label_col],errors="coerce").to_numpy())
    return feats

# ---------- PhiUSIIL lexical features ----------
if artifact_ok(lex_phi_path) and not FORCE:
    phi_lex = pd.read_parquet(lex_phi_path)
    print("[LOAD] PhiUSIIL lexical features")
else:
    print("[RUN ] Featurize PhiUSIIL URLs")
    phi = pd.read_parquet(phi_path)
    target_candidates = [c for c in phi.columns if str(c).lower() in {"label","class","target","phishing"}]
    if not target_candidates:
        target_candidates = [c for c in phi.columns if "label" in str(c).lower()]
    target_col = target_candidates[-1]
    if "URL" not in phi.columns:
        raise ValueError("PhiUSIIL raw URL column is required for robust lexical experiment.")
    tmp = pd.DataFrame({
        "url":phi["URL"].astype(str),
        "label":(pd.to_numeric(phi[target_col],errors="coerce")==0).astype(int)  # 1=phishing
    })
    phi_lex = featurize_urls(tmp)
    save_df(phi_lex,lex_phi_path)
    del phi,tmp

# ---------- Download current PhishTank + Tranco ----------
def get_bytes(url, timeout=180):
    headers={"User-Agent":"Mozilla/5.0 CRB-Q1-Research/1.0"}
    r=requests.get(url,headers=headers,timeout=timeout)
    r.raise_for_status()
    return r.content

def build_external_urls():
    import io, gzip, zipfile

    # Current verified online phishing.
    ph_urls = None
    ph_err = None
    for url in [
        "https://data.phishtank.com/data/online-valid.csv.gz",
        "http://data.phishtank.com/data/online-valid.csv.gz",
    ]:
        try:
            b=get_bytes(url)
            with gzip.GzipFile(fileobj=io.BytesIO(b)) as gz:
                phdf=pd.read_csv(gz)
            url_candidates=[c for c in phdf.columns if str(c).lower()=="url" or "url" in str(c).lower()]
            if not url_candidates:
                raise ValueError("No URL field in PhishTank dump")
            ph_urls=phdf[url_candidates[0]].astype(str).dropna().drop_duplicates()
            break
        except Exception as e:
            ph_err=str(e)

    if ph_urls is None:
        raise RuntimeError(f"PhishTank download failed: {ph_err}")

    # Current Tranco top list.
    tr_err=None
    trdf=None
    for url in [
        "https://tranco-list.eu/top-1m.csv.zip",
        "https://tranco-list.eu/top-1m.csv",
    ]:
        try:
            b=get_bytes(url)
            if url.endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(b)) as z:
                    name=z.namelist()[0]
                    with z.open(name) as f:
                        trdf=pd.read_csv(f,header=None,names=["rank","domain"])
            else:
                trdf=pd.read_csv(io.BytesIO(b),header=None,names=["rank","domain"])
            break
        except Exception as e:
            tr_err=str(e)

    if trdf is None:
        # stable fallback used only if the live Tranco endpoint is unavailable
        fallback="https://raw.githubusercontent.com/1xyz/tranco/main/data/tranco_top_50K.txt"
        b=get_bytes(fallback)
        domains=pd.Series(b.decode("utf-8",errors="ignore").splitlines(),name="domain")
        trdf=pd.DataFrame({"rank":np.arange(1,len(domains)+1),"domain":domains})

    # Keep acquisition lightweight while still large enough for external testing.
    ph_urls=ph_urls.head(30000)
    ben_urls=("https://"+trdf["domain"].astype(str).str.strip()+"/").drop_duplicates().head(50000)

    ext=pd.concat([
        pd.DataFrame({"url":ph_urls,"label":1,"source":"PhishTank_current"}),
        pd.DataFrame({"url":ben_urls,"label":0,"source":"Tranco_current"}),
    ],ignore_index=True)
    ext["url_norm"]=ext["url"].astype(str).str.strip().str.lower()
    ext=ext.drop_duplicates("url_norm").drop(columns="url_norm")
    return ext

if artifact_ok(ext_url_path) and not FORCE:
    ext_urls=pd.read_parquet(ext_url_path)
    print("[LOAD] External PhishTank/Tranco snapshot")
else:
    print("[RUN ] Download current PhishTank + Tranco")
    ext_urls=build_external_urls()

    # Exclude exact URL overlap and, where possible, registrable-domain overlap with PhiUSIIL.
    phi_norm=set(phi_lex["url"].astype(str).str.strip().str.lower())
    ext_urls=ext_urls[~ext_urls["url"].astype(str).str.strip().str.lower().isin(phi_norm)].copy()
    ext_urls["registrable_domain"]=[registrable_domain(u) for u in ext_urls.url]
    phi_domains=set(phi_lex["registrable_domain"].dropna().astype(str))
    domain_clean=ext_urls[~ext_urls["registrable_domain"].isin(phi_domains)].copy()

    # Keep domain-clean set if there are enough examples of both classes.
    counts=domain_clean.label.value_counts()
    if counts.get(1,0)>=3000 and counts.get(0,0)>=10000:
        ext_urls=domain_clean
        overlap_mode="registrable-domain-disjoint"
    else:
        overlap_mode="exact-URL-disjoint (domain-disjoint left too few samples)"

    save_df(ext_urls,ext_url_path)
    save_json({
        "snapshot_time_utc":now(),
        "overlap_filter":overlap_mode,
        "rows":len(ext_urls),
        "class_counts":ext_urls.label.value_counts().to_dict()
    },DIRS["reports"]/"external_snapshot_manifest.json")

# ---------- External lexical features ----------
if artifact_ok(ext_feat_path) and not FORCE:
    ext_lex=pd.read_parquet(ext_feat_path)
    print("[LOAD] External lexical features")
else:
    print("[RUN ] Featurize external URLs")
    ext_lex=featurize_urls(ext_urls[["url","label"]])
    save_df(ext_lex,ext_feat_path)

feature_cols=[c for c in phi_lex.columns if c not in {"url","label","registrable_domain"}]

# ---------- Internal domain-disjoint lexical benchmark ----------
if artifact_ok(lex_int_metrics_path) and artifact_ok(lex_ext_metrics_path) and artifact_ok(lex_models_path) and not FORCE:
    print("[LOAD] Lexical risk benchmark outputs")
    lexical_internal=pd.read_csv(lex_int_metrics_path)
    lexical_external=pd.read_csv(lex_ext_metrics_path)
    lexical_deployment=pd.read_csv(lex_dep_metrics_path) if artifact_ok(lex_dep_metrics_path) else pd.DataFrame()
    lexical_models=joblib.load(lex_models_path)
else:
    print("[RUN ] Train/evaluate URL-only lexical models")
    y=phi_lex.label.astype(int)
    groups=phi_lex.registrable_domain.astype(str)
    X=phi_lex[feature_cols].replace([np.inf,-np.inf],np.nan)

    gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)
    tr_idx,te_idx=next(gss.split(X,y,groups))
    Xtr,Xte=X.iloc[tr_idx],X.iloc[te_idx]
    ytr,yte=y.iloc[tr_idx],y.iloc[te_idx]

    models={
        "Lexical_Logistic":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("sc",StandardScaler()),
            ("clf",LogisticRegression(max_iter=1500,class_weight="balanced",random_state=SEED))
        ]),
        "Lexical_RF":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",RandomForestClassifier(
                n_estimators=400,min_samples_leaf=2,class_weight="balanced_subsample",
                n_jobs=-1,random_state=SEED
            ))
        ]),
        "Lexical_HGB":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",HistGradientBoostingClassifier(max_iter=300,learning_rate=.05,random_state=SEED))
        ]),
    }
    if HAVE_XGB:
        models["Lexical_XGB"]=Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",XGBClassifier(
                n_estimators=400,max_depth=5,learning_rate=.05,
                subsample=.85,colsample_bytree=.85,eval_metric="logloss",
                random_state=SEED,n_jobs=-1
            ))
        ])

    def op_metrics(ytrue,p):
        pred=(p>=.5).astype(int)
        tn,fp,fn,tp=confusion_matrix(ytrue,pred,labels=[0,1]).ravel()
        fpr,tpr,thr=roc_curve(ytrue,p)
        def tpr_at(max_fpr):
            ok=np.where(fpr<=max_fpr)[0]
            return float(tpr[ok[-1]]) if len(ok) else 0.0
        return {
            "AUROC":roc_auc_score(ytrue,p),
            "AUPRC":average_precision_score(ytrue,p),
            "F1":f1_score(ytrue,pred),
            "MCC":matthews_corrcoef(ytrue,pred),
            "BalancedAccuracy":balanced_accuracy_score(ytrue,pred),
            "Sensitivity":tp/(tp+fn) if tp+fn else np.nan,
            "Specificity":tn/(tn+fp) if tn+fp else np.nan,
            "Brier":brier_score_loss(ytrue,p),
            "TPR_at_FPR_1pct":tpr_at(.01),
            "TPR_at_FPR_0_1pct":tpr_at(.001),
        }

    int_rows=[]
    fitted={}
    for name,m in models.items():
        print(" fitting",name)
        m.fit(Xtr,ytr)
        p=m.predict_proba(Xte)[:,1]
        int_rows.append({"Model":name,"Protocol":"PhiUSIIL_domain_disjoint",**op_metrics(yte,p)})
        fitted[name]=m

    lexical_internal=pd.DataFrame(int_rows)
    save_df(lexical_internal,lex_int_metrics_path)

    # Balanced current external test.
    rng=np.random.default_rng(SEED)
    ph=ext_lex[ext_lex.label.eq(1)]
    be=ext_lex[ext_lex.label.eq(0)]
    nbal=int(min(len(ph),len(be),10000))
    balanced=pd.concat([
        ph.sample(nbal,random_state=SEED),
        be.sample(nbal,random_state=SEED)
    ],ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)

    ext_rows=[]
    for name,m in fitted.items():
        p=m.predict_proba(balanced[feature_cols])[:,1]
        ext_rows.append({
            "Model":name,"Protocol":"Current_external_balanced",
            "n":len(balanced),**op_metrics(balanced.label.astype(int),p)
        })
    lexical_external=pd.DataFrame(ext_rows)
    save_df(lexical_external,lex_ext_metrics_path)

    # Deployment-like prevalence: 10% phishing.
    nph=int(min(len(ph),5000))
    nbe=int(min(len(be),nph*9))
    nph=min(nph,max(1,nbe//9))
    deploy=pd.concat([
        ph.sample(nph,random_state=SEED+1),
        be.sample(nph*9,random_state=SEED+2)
    ],ignore_index=True).sample(frac=1,random_state=SEED+3).reset_index(drop=True)

    dep_rows=[]
    for name,m in fitted.items():
        p=m.predict_proba(deploy[feature_cols])[:,1]
        dep_rows.append({
            "Model":name,"Protocol":"Current_external_10pct_phishing",
            "n":len(deploy),**op_metrics(deploy.label.astype(int),p)
        })
    lexical_deployment=pd.DataFrame(dep_rows)
    save_df(lexical_deployment,lex_dep_metrics_path)

    joblib.dump({"models":fitted,"features":feature_cols},lex_models_path)
    lexical_models={"models":fitted,"features":feature_cols}

    checkpoint("lexical_external_validation",{
        "phiusiil_rows":len(phi_lex),
        "external_rows":len(ext_lex),
        "feature_count":len(feature_cols),
        "balanced_test_n":len(balanced),
        "deployment_test_n":len(deploy),
    })

display(lexical_internal.sort_values("AUPRC",ascending=False))
display(lexical_external.sort_values("AUPRC",ascending=False))
display(lexical_deployment.sort_values("AUPRC",ascending=False))

# Visual comparison: internal vs external AUROC
cmp=lexical_internal[["Model","AUROC"]].rename(columns={"AUROC":"Internal_AUROC"}).merge(
    lexical_external[["Model","AUROC"]].rename(columns={"AUROC":"External_AUROC"}),
    on="Model",how="inner"
)
x=np.arange(len(cmp))
w=.36
plt.figure(figsize=(8,5))
plt.bar(x-w/2,cmp.Internal_AUROC,width=w,label="PhiUSIIL domain-disjoint")
plt.bar(x+w/2,cmp.External_AUROC,width=w,label="Current external")
plt.xticks(x,cmp.Model,rotation=25,ha="right")
plt.ylim(0,1.02)
plt.ylabel("AUROC")
plt.title("URL-only Risk Engine: Internal vs Current External Generalization")
plt.legend()
plt.tight_layout()
plt.savefig(DIRS["figures"]/"lexical_internal_vs_external.png",dpi=220,bbox_inches="tight")
plt.savefig(DIRS["figures"]/"lexical_internal_vs_external.pdf",bbox_inches="tight")
plt.close()

[LOAD] PhiUSIIL lexical features
[LOAD] External PhishTank/Tranco snapshot
[LOAD] External lexical features
[LOAD] Lexical risk benchmark outputs


,Model,Protocol,AUROC,AUPRC,F1,MCC,BalancedAccuracy,Sensitivity,Specificity,Brier,TPR_at_FPR_1pct,TPR_at_FPR_0_1pct
3,Lexical_XGB,PhiUSIIL_domain_disjoint,0.998948,0.999310,0.997958,0.995941,0.997962,0.996256,0.999668,0.001959,0.997072,0.996442
1,Lexical_RF,PhiUSIIL_domain_disjoint,0.998847,0.999279,0.997567,0.995167,0.997573,0.995552,0.999595,0.002558,0.997220,0.996108
2,Lexical_HGB,PhiUSIIL_domain_disjoint,0.998855,0.999261,0.997846,0.995719,0.997851,0.996108,0.999595,0.001953,0.997146,0.996590
0,Lexical_Logistic,PhiUSIIL_domain_disjoint,0.998195,0.998930,0.995651,0.991369,0.995668,0.992846,0.998490,0.003701,0.995774,0.992438


,Model,Protocol,n,AUROC,AUPRC,F1,MCC,BalancedAccuracy,Sensitivity,Specificity,Brier,TPR_at_FPR_1pct,TPR_at_FPR_0_1pct
0,Lexical_Logistic,Current_external_balanced,20000,0.870400,0.923571,0.651746,-0.129924,0.48340,0.9668,0.0,0.512542,0.7250,0.4173
3,Lexical_XGB,Current_external_balanced,20000,0.774606,0.856689,0.657853,-0.099740,0.49015,0.9803,0.0,0.508387,0.6056,0.5728
2,Lexical_HGB,Current_external_balanced,20000,0.553616,0.573973,0.658438,-0.096361,0.49080,0.9816,0.0,0.507972,0.1819,0.0017
1,Lexical_RF,Current_external_balanced,20000,0.442310,0.473921,0.657448,-0.102016,0.48970,0.9794,0.0,0.507615,0.0000,0.0000


,Model,Protocol,n,AUROC,AUPRC,F1,MCC,BalancedAccuracy,Sensitivity,Specificity,Brier,TPR_at_FPR_1pct,TPR_at_FPR_0_1pct
0,Lexical_Logistic,Current_external_10pct_phishing,47550,0.862700,0.809517,0.176169,-0.175406,0.482965,0.965931,0.0,0.902592,0.721136,0.438275
3,Lexical_XGB,Current_external_10pct_phishing,47550,0.773443,0.691617,0.178580,-0.132804,0.490221,0.980442,0.0,0.901586,0.602313,0.567403
2,Lexical_HGB,Current_external_10pct_phishing,47550,0.549850,0.243196,0.178824,-0.127699,0.490957,0.981914,0.0,0.901586,0.178759,0.002944
1,Lexical_RF,Current_external_10pct_phishing,47550,0.438621,0.090361,0.178475,-0.134934,0.489905,0.979811,0.0,0.900020,0.000000,0.000000


In [ ]:
# CELL 4C — FAIR DOMAIN-ONLY external validation (v6.1 FIX):
# ZERO-SHOT transfer vs CALIBRATED deployment
#
# v6.1 FIX:
# Some registrable domains can appear in BOTH PhishTank and Tranco. If class-specific
# domain lists are split independently, the same domain can accidentally enter CALIBRATION
# under one class and TEST under the other. This is a group-leakage risk.
#
# Therefore v6.1:
#   1) identifies registrable domains carrying conflicting labels;
#   2) removes those ambiguous domains entirely from this external benchmark;
#   3) splits the remaining UNIQUE domains class-wise;
#   4) verifies zero domain overlap between calibration and test;
#   5) saves a complete domain-conflict/split audit.
#
# Fairness:
# Only hostname/domain features are used. Path, query, fragment, and HTTPS are excluded.
#
# External protocol:
# 1) Train ONLY on PhiUSIIL.
# 2) Split cleaned external registrable domains into CALIBRATION and untouched TEST domains.
# 3) Evaluate source model ZERO-SHOT on untouched TEST.
# 4) Fit one-dimensional Platt calibration ONLY on external CALIBRATION domains.
# 5) Evaluate recalibrated probabilities on untouched TEST.
#
# Test labels are never used for source training, Platt fitting, or split selection.

dom_phi_path = DIRS["processed"] / "phiusiil_domainonly_features_v6.parquet"
dom_ext_path = DIRS["processed"] / "external_domainonly_features_v6.parquet"

# v6.1 result names prevent accidental loading of a partially created v6 protocol.
dom_internal_path = DIRS["tables"] / "domainonly_internal_metrics_v6_1.csv"
dom_zero_path = DIRS["tables"] / "domainonly_external_zero_shot_v6_1.csv"
dom_cal_path = DIRS["tables"] / "domainonly_external_recalibrated_v6_1.csv"
dom_manifest_path = DIRS["reports"] / "domainonly_external_protocol_v6_1.json"
dom_conflict_path = DIRS["tables"] / "domainonly_external_conflicting_domains_v6_1.csv"
dom_split_path = DIRS["tables"] / "domainonly_external_domain_split_v6_1.csv"
dom_model_path = DIRS["models"] / "domainonly_external_models_v6_1.joblib"

HOST_SUSPICIOUS = [
    "login","verify","secure","account","update","signin","bank","wallet",
    "confirm","password","support","recover","payment","invoice","bonus","gift"
]

def _host_entropy(s):
    from collections import Counter
    s = str(s)
    if not s:
        return 0.0
    n = len(s)
    cc = Counter(s)
    return float(-sum((v/n)*math.log2(v/n) for v in cc.values()))

def domain_only_row(u):
    raw = "" if pd.isna(u) else str(u).strip()
    norm = normalize_url(raw)
    try:
        p = urlsplit(norm)
        host = (p.hostname or "").lower().strip(".")
    except Exception:
        host = ""

    ex = _extract(host)
    regdom = ".".join(x for x in [ex.domain, ex.suffix] if x)
    sub = ex.subdomain or ""
    labels = [x for x in host.split(".") if x]
    digits = sum(ch.isdigit() for ch in host)
    letters = sum(ch.isalpha() for ch in host)
    hyphens = host.count("-")
    ipv4 = bool(re.fullmatch(r"\d{1,3}(?:\.\d{1,3}){3}", host))
    ipv6 = ":" in host and all(c in "0123456789abcdef:" for c in host)

    return {
        "registrable_domain": regdom or host,
        "host_length": len(host),
        "label_count": len(labels),
        "subdomain_count": 0 if not sub else len([x for x in sub.split(".") if x]),
        "dot_count": host.count("."),
        "hyphen_count": hyphens,
        "digit_count": digits,
        "digit_ratio_host": digits/max(len(host),1),
        "letter_ratio_host": letters/max(len(host),1),
        "hostname_entropy": _host_entropy(host),
        "has_ip_host": int(ipv4 or ipv6),
        "has_punycode": int("xn--" in host),
        "suspicious_host_token_count": sum(tok in host for tok in HOST_SUSPICIOUS),
        "is_shortener_host": int((regdom or host) in SHORTENERS),
        "tld_length": len(ex.suffix or ""),
        "registered_label_length": len(ex.domain or ""),
        "max_label_length": max([len(x) for x in labels], default=0),
        "hyphen_ratio_host": hyphens/max(len(host),1),
    }

def domain_featurize(df):
    out = pd.DataFrame([domain_only_row(u) for u in df["url"].astype(str)])
    out.insert(0, "url", df["url"].astype(str).to_numpy())
    out.insert(1, "label", pd.to_numeric(df["label"],errors="coerce").astype(int).to_numpy())
    return out

def ece_local(ytrue, prob, bins=10):
    ytrue = np.asarray(ytrue)
    prob = np.asarray(prob)
    edges = np.linspace(0,1,bins+1)
    out = 0.0
    for a,b in zip(edges[:-1],edges[1:]):
        mask = (prob>=a) & ((prob<b) if b<1 else (prob<=b))
        if mask.sum():
            out += mask.mean() * abs(ytrue[mask].mean() - prob[mask].mean())
    return float(out)

def eval_prob(ytrue, p, threshold=.5):
    ytrue = np.asarray(ytrue).astype(int)
    p = np.asarray(p,dtype=float)
    pred = (p>=threshold).astype(int)
    tn,fp,fn,tp = confusion_matrix(ytrue,pred,labels=[0,1]).ravel()
    fpr,tpr,_ = roc_curve(ytrue,p)

    def _tpr_at(max_fpr):
        ok=np.where(fpr<=max_fpr)[0]
        return float(tpr[ok[-1]]) if len(ok) else 0.0

    return {
        "AUROC":roc_auc_score(ytrue,p),
        "AUPRC":average_precision_score(ytrue,p),
        "F1":f1_score(ytrue,pred,zero_division=0),
        "MCC":matthews_corrcoef(ytrue,pred),
        "BalancedAccuracy":balanced_accuracy_score(ytrue,pred),
        "Sensitivity":tp/(tp+fn) if tp+fn else np.nan,
        "Specificity":tn/(tn+fp) if tn+fp else np.nan,
        "Brier":brier_score_loss(ytrue,p),
        "ECE":ece_local(ytrue,p,10),
        "TPR_at_FPR_1pct":_tpr_at(.01),
        "TPR_at_FPR_0_1pct":_tpr_at(.001),
        "threshold":threshold,
    }

# ---------- feature extraction ----------
if artifact_ok(dom_phi_path) and not FORCE:
    phi_dom = pd.read_parquet(dom_phi_path)
    print("[LOAD] PhiUSIIL domain-only features")
else:
    print("[RUN ] Build PhiUSIIL domain-only features")
    phi = pd.read_parquet(phi_path)
    tc = [c for c in phi.columns if str(c).lower() in {"label","class","target","phishing"}]
    if not tc:
        tc = [c for c in phi.columns if "label" in str(c).lower()]
    target_col = tc[-1]
    src = pd.DataFrame({
        "url":phi["URL"].astype(str),
        "label":(pd.to_numeric(phi[target_col],errors="coerce")==0).astype(int)
    })
    phi_dom = domain_featurize(src)
    save_df(phi_dom,dom_phi_path)
    del phi,src

if artifact_ok(dom_ext_path) and not FORCE:
    ext_dom = pd.read_parquet(dom_ext_path)
    print("[LOAD] External domain-only features")
else:
    print("[RUN ] Build external domain-only features")
    # ext_urls is created by Cell 4B and already removes source overlap with PhiUSIIL where feasible.
    ext_dom = domain_featurize(ext_urls[["url","label"]].copy())
    save_df(ext_dom,dom_ext_path)

dom_features=[c for c in phi_dom.columns if c not in {"url","label","registrable_domain"}]

# ---------- source internal domain-disjoint split ----------
phi_dom = phi_dom.dropna(subset=["registrable_domain","label"]).copy()
phi_dom["registrable_domain"]=phi_dom["registrable_domain"].astype(str).str.lower().str.strip()

y_src=phi_dom.label.astype(int)
g_src=phi_dom.registrable_domain.astype(str)
X_src=phi_dom[dom_features].replace([np.inf,-np.inf],np.nan)

gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED+606)
tr_idx,te_idx=next(gss.split(X_src,y_src,g_src))
Xtr,Xte=X_src.iloc[tr_idx],X_src.iloc[te_idx]
ytr,yte=y_src.iloc[tr_idx],y_src.iloc[te_idx]

def domain_model_bank():
    bank={
        "Domain_Logistic":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("sc",StandardScaler()),
            ("clf",LogisticRegression(max_iter=1500,class_weight="balanced",random_state=SEED))
        ]),
        "Domain_RF":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",RandomForestClassifier(
                n_estimators=450,min_samples_leaf=2,class_weight="balanced_subsample",
                n_jobs=-1,random_state=SEED
            ))
        ]),
        "Domain_HGB":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",HistGradientBoostingClassifier(max_iter=300,learning_rate=.05,random_state=SEED))
        ])
    }
    if HAVE_XGB:
        bank["Domain_XGB"]=Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",XGBClassifier(
                n_estimators=450,max_depth=5,learning_rate=.05,
                subsample=.85,colsample_bytree=.85,eval_metric="logloss",
                random_state=SEED,n_jobs=-1
            ))
        ])
    return bank

# ------------------------------------------------------------------
# EXTERNAL CLEANING + STRICT DOMAIN-GROUP SPLIT
# ------------------------------------------------------------------
ext_dom = ext_dom.dropna(subset=["registrable_domain","label"]).copy()
ext_dom["registrable_domain"] = (
    ext_dom["registrable_domain"].astype(str).str.lower().str.strip()
)
ext_dom = ext_dom[ext_dom.registrable_domain.ne("")].copy()
ext_dom["label"] = pd.to_numeric(ext_dom["label"],errors="coerce").astype(int)

# Detect registrable domains carrying BOTH labels.
label_counts = (
    ext_dom.groupby("registrable_domain")["label"]
    .agg(["nunique","count","min","max"])
    .reset_index()
)
conflicting_domains = set(
    label_counts.loc[label_counts["nunique"]>1,"registrable_domain"].astype(str)
)

conflict_rows = ext_dom[ext_dom.registrable_domain.isin(conflicting_domains)].copy()
if len(conflict_rows):
    conflict_summary = (
        conflict_rows.groupby(["registrable_domain","label"])
        .size().rename("rows").reset_index()
    )
else:
    conflict_summary = pd.DataFrame(
        columns=["registrable_domain","label","rows"]
    )
save_df(conflict_summary,dom_conflict_path)

# Remove ambiguous domains entirely.
ext_clean = ext_dom[~ext_dom.registrable_domain.isin(conflicting_domains)].copy()

# Verify every remaining domain has exactly one class.
remaining_nunique = ext_clean.groupby("registrable_domain")["label"].nunique()
assert int(remaining_nunique.max()) == 1

# Domain-level label table.
domain_table = (
    ext_clean.groupby("registrable_domain",as_index=False)
    .agg(label=("label","first"),rows=("label","size"))
)

print("External rows before conflict removal:",len(ext_dom))
print("Conflicting registrable domains removed:",len(conflicting_domains))
print("Rows removed due to domain-label conflict:",len(conflict_rows))
print("External rows after conflict removal:",len(ext_clean))
print("Unique clean external domains:",domain_table.registrable_domain.nunique())
print("Clean domain labels:",domain_table.label.value_counts().to_dict())

# Deterministic class-stratified split of UNIQUE CLEAN DOMAINS.
rng=np.random.default_rng(SEED+607)
split_records=[]
cal_domains=set()
test_domains=set()

for cls in [0,1]:
    cls_domains=np.array(
        sorted(domain_table.loc[domain_table.label.eq(cls),"registrable_domain"].unique())
    )
    if len(cls_domains) < 10:
        raise ValueError(
            f"Too few clean unique external domains for class {cls}: {len(cls_domains)}"
        )
    rng.shuffle(cls_domains)
    ncal=max(2,int(round(.20*len(cls_domains))))
    ncal=min(ncal,len(cls_domains)-2)

    class_cal=cls_domains[:ncal]
    class_test=cls_domains[ncal:]

    cal_domains.update(class_cal.tolist())
    test_domains.update(class_test.tolist())

    split_records.extend([
        {"registrable_domain":d,"label":cls,"split":"calibration"} for d in class_cal
    ])
    split_records.extend([
        {"registrable_domain":d,"label":cls,"split":"test"} for d in class_test
    ])

split_df=pd.DataFrame(split_records)
save_df(split_df,dom_split_path)

# The key v6.1 invariant:
overlap = set(cal_domains) & set(test_domains)
assert len(overlap)==0, f"Unexpected calibration/test domain overlap: {list(overlap)[:10]}"

cal_pool=ext_clean[ext_clean.registrable_domain.isin(cal_domains)].copy()
test_pool=ext_clean[ext_clean.registrable_domain.isin(test_domains)].copy()

assert set(cal_pool.registrable_domain).isdisjoint(set(test_pool.registrable_domain))
assert cal_pool.label.nunique()==2 and test_pool.label.nunique()==2
assert not cal_pool.registrable_domain.isin(test_pool.registrable_domain).any()

# deployment-like calibration prevalence: 10% phishing, calibration domains only
phc=cal_pool[cal_pool.label.eq(1)]
bec=cal_pool[cal_pool.label.eq(0)]

nphc=int(min(len(phc),1500))
nbec=int(min(len(bec),nphc*9))
nphc=min(nphc,max(1,nbec//9))

if nphc < 50:
    raise ValueError(f"Too few phishing calibration rows after strict domain split: {nphc}")

cal_deploy=pd.concat([
    phc.sample(nphc,random_state=SEED+608),
    bec.sample(nphc*9,random_state=SEED+609)
],ignore_index=True).sample(frac=1,random_state=SEED+610).reset_index(drop=True)

# untouched balanced external test
pht=test_pool[test_pool.label.eq(1)]
bet=test_pool[test_pool.label.eq(0)]
nbal=int(min(len(pht),len(bet),10000))

if nbal < 500:
    raise ValueError(f"Too few balanced untouched test rows after strict split: {nbal}")

test_bal=pd.concat([
    pht.sample(nbal,random_state=SEED+611),
    bet.sample(nbal,random_state=SEED+612)
],ignore_index=True).sample(frac=1,random_state=SEED+613).reset_index(drop=True)

# untouched deployment-like 10% phishing test
npht=int(min(len(pht),5000))
nbet=int(min(len(bet),npht*9))
npht=min(npht,max(1,nbet//9))

if npht < 100:
    raise ValueError(f"Too few deployment-like phishing test rows: {npht}")

test_dep=pd.concat([
    pht.sample(npht,random_state=SEED+614),
    bet.sample(npht*9,random_state=SEED+615)
],ignore_index=True).sample(frac=1,random_state=SEED+616).reset_index(drop=True)

# ---------- fit source models + zero-shot + Platt recalibration ----------
if all(artifact_ok(p) for p in [dom_internal_path,dom_zero_path,dom_cal_path,dom_model_path]) and not FORCE:
    print("[LOAD] Domain-only v6.1 zero-shot/recalibrated benchmark")
    dom_internal=pd.read_csv(dom_internal_path)
    dom_zero=pd.read_csv(dom_zero_path)
    dom_calibrated=pd.read_csv(dom_cal_path)
    dom_bundle=joblib.load(dom_model_path)
else:
    print("[RUN ] Domain-only v6.1 zero-shot/recalibrated benchmark")
    internal_rows=[]
    zero_rows=[]
    cal_rows=[]
    fitted={}
    calibrators={}

    for name,m in domain_model_bank().items():
        print(" fitting",name)
        m.fit(Xtr,ytr)
        fitted[name]=m

        p_int=m.predict_proba(Xte)[:,1]
        internal_rows.append({
            "Model":name,
            "Protocol":"PhiUSIIL_domain_only_domain_disjoint",
            "n":len(yte),
            **eval_prob(yte,p_int,.5)
        })

        # External raw predictions
        p_calraw=np.clip(
            m.predict_proba(cal_deploy[dom_features])[:,1],1e-6,1-1e-6
        )
        p_bal=np.clip(
            m.predict_proba(test_bal[dom_features])[:,1],1e-6,1-1e-6
        )
        p_dep=np.clip(
            m.predict_proba(test_dep[dom_features])[:,1],1e-6,1-1e-6
        )

        # ZERO-SHOT: untouched external test
        zero_rows.append({
            "Model":name,
            "Protocol":"ZeroShot_balanced_external_test",
            "n":len(test_bal),
            **eval_prob(test_bal.label,p_bal,.5)
        })
        zero_rows.append({
            "Model":name,
            "Protocol":"ZeroShot_10pct_external_test",
            "n":len(test_dep),
            **eval_prob(test_dep.label,p_dep,.5)
        })

        # Platt calibration using calibration-domain rows only.
        zcal=logit(p_calraw).reshape(-1,1)
        platt=LogisticRegression(
            max_iter=1000,
            class_weight=None,  # preserve 10% deployment-like calibration prior
            random_state=SEED
        )
        platt.fit(zcal,cal_deploy.label.astype(int))
        calibrators[name]=platt

        p_bal_c=platt.predict_proba(logit(p_bal).reshape(-1,1))[:,1]
        p_dep_c=platt.predict_proba(logit(p_dep).reshape(-1,1))[:,1]

        cal_rows.append({
            "Model":name,
            "Protocol":"PlattRecalibrated_balanced_external_test",
            "n":len(test_bal),
            **eval_prob(test_bal.label,p_bal_c,.5)
        })
        cal_rows.append({
            "Model":name,
            "Protocol":"PlattRecalibrated_10pct_external_test",
            "n":len(test_dep),
            **eval_prob(test_dep.label,p_dep_c,.5)
        })

    dom_internal=pd.DataFrame(internal_rows)
    dom_zero=pd.DataFrame(zero_rows)
    dom_calibrated=pd.DataFrame(cal_rows)

    save_df(dom_internal,dom_internal_path)
    save_df(dom_zero,dom_zero_path)
    save_df(dom_calibrated,dom_cal_path)

    joblib.dump({
        "source_models":fitted,
        "platt_calibrators":calibrators,
        "features":dom_features,
    },dom_model_path)

    save_json({
        "protocol_version":"v6.1-domain-only-conflict-clean-platt",
        "domain_features":dom_features,
        "external_rows_before_conflict_removal":len(ext_dom),
        "conflicting_domains_removed":len(conflicting_domains),
        "rows_removed_due_to_conflict":len(conflict_rows),
        "external_rows_after_conflict_removal":len(ext_clean),
        "clean_unique_domains":int(domain_table.registrable_domain.nunique()),
        "external_calibration_pool_rows":len(cal_pool),
        "external_test_pool_rows":len(test_pool),
        "external_calibration_rows_used_for_platt":len(cal_deploy),
        "external_balanced_test_rows":len(test_bal),
        "external_10pct_test_rows":len(test_dep),
        "calibration_unique_domains":int(cal_pool.registrable_domain.nunique()),
        "test_unique_domains":int(test_pool.registrable_domain.nunique()),
        "domain_overlap_cal_vs_test":0,
        "important_note":"Domains with conflicting labels across PhishTank/Tranco were removed before splitting. External test domains remain untouched until final evaluation."
    },dom_manifest_path)

    checkpoint("domainonly_external_v6_1",{
        "features":len(dom_features),
        "conflicting_domains_removed":len(conflicting_domains),
        "calibration_unique_domains":int(cal_pool.registrable_domain.nunique()),
        "test_unique_domains":int(test_pool.registrable_domain.nunique()),
        "calibration_rows":len(cal_deploy),
        "test_balanced_rows":len(test_bal),
        "test_10pct_rows":len(test_dep),
    })

print("\n--- INTERNAL DOMAIN-ONLY ---")
display(dom_internal.sort_values("AUPRC",ascending=False))

print("\n--- EXTERNAL ZERO-SHOT ---")
display(dom_zero.sort_values(["Protocol","AUPRC"],ascending=[True,False]))

print("\n--- EXTERNAL RECALIBRATED ---")
display(dom_calibrated.sort_values(["Protocol","Brier"],ascending=[True,True]))

# Paired visualization: zero-shot vs recalibrated Brier on deployment-like 10% test.
z=dom_zero[
    dom_zero.Protocol.eq("ZeroShot_10pct_external_test")
][["Model","Brier","ECE","BalancedAccuracy"]].copy()
z=z.rename(columns={
    "Brier":"ZeroShot_Brier",
    "ECE":"ZeroShot_ECE",
    "BalancedAccuracy":"ZeroShot_BalAcc"
})

c=dom_calibrated[
    dom_calibrated.Protocol.eq("PlattRecalibrated_10pct_external_test")
][["Model","Brier","ECE","BalancedAccuracy"]].copy()
c=c.rename(columns={
    "Brier":"Calibrated_Brier",
    "ECE":"Calibrated_ECE",
    "BalancedAccuracy":"Calibrated_BalAcc"
})

cmp=z.merge(c,on="Model",how="inner")

x=np.arange(len(cmp)); w=.36
plt.figure(figsize=(8,5))
plt.bar(x-w/2,cmp.ZeroShot_Brier,width=w,label="Zero-shot")
plt.bar(x+w/2,cmp.Calibrated_Brier,width=w,label="Recalibrated")
plt.xticks(x,cmp.Model,rotation=25,ha="right")
plt.ylabel("Brier score (lower is better)")
plt.title("Domain-only External Transfer: Zero-shot vs Recalibrated")
plt.legend()
plt.tight_layout()
plt.savefig(
    DIRS["figures"]/"domainonly_zero_vs_recalibrated_brier_v6_1.png",
    dpi=230,bbox_inches="tight"
)
plt.savefig(
    DIRS["figures"]/"domainonly_zero_vs_recalibrated_brier_v6_1.pdf",
    bbox_inches="tight"
)
plt.close()

print("\n[OK] Cell 4C v6.1 completed with strict domain-group isolation.")
print("Conflict audit:",dom_conflict_path)
print("Domain split audit:",dom_split_path)

[LOAD] PhiUSIIL domain-only features
[LOAD] External domain-only features
External rows before conflict removal: 63973
Conflicting registrable domains removed: 66
Rows removed due to domain-label conflict: 4359
External rows after conflict removal: 59614
Unique clean external domains: 53971
Clean domain labels: {0: 42734, 1: 11237}
[LOAD] Domain-only v6.1 zero-shot/recalibrated benchmark

--- INTERNAL DOMAIN-ONLY ---


,Model,Protocol,n,AUROC,AUPRC,F1,MCC,BalancedAccuracy,Sensitivity,Specificity,Brier,ECE,TPR_at_FPR_1pct,TPR_at_FPR_0_1pct,threshold
3,Domain_XGB,PhiUSIIL_domain_only_domain_disjoint,43601,0.876686,0.876914,0.763364,0.678907,0.808453,0.645986,0.970919,0.117254,0.021137,0.566840,0.496037,0.5
2,Domain_HGB,PhiUSIIL_domain_only_domain_disjoint,43601,0.874061,0.872934,0.763200,0.680242,0.808353,0.643960,0.972746,0.118546,0.019886,0.562727,0.495977,0.5
1,Domain_RF,PhiUSIIL_domain_only_domain_disjoint,43601,0.869769,0.872047,0.773852,0.681939,0.815741,0.671673,0.959809,0.117188,0.028851,0.571786,0.458073,0.5
0,Domain_Logistic,PhiUSIIL_domain_only_domain_disjoint,43601,0.826369,0.833169,0.743976,0.653795,0.795253,0.624769,0.965737,0.141224,0.086457,0.414327,0.282913,0.5



--- EXTERNAL ZERO-SHOT ---


,Model,Protocol,n,AUROC,AUPRC,F1,MCC,BalancedAccuracy,Sensitivity,Specificity,Brier,ECE,TPR_at_FPR_1pct,TPR_at_FPR_0_1pct,threshold
1,Domain_Logistic,ZeroShot_10pct_external_test,37980,0.733984,0.593847,0.170572,-0.061711,0.472149,0.870195,0.074103,0.472622,0.627912,0.478146,0.000000,0.5
3,Domain_RF,ZeroShot_10pct_external_test,37980,0.243632,0.063183,0.169972,-0.227192,0.464543,0.927330,0.001755,0.895600,0.896387,0.000000,0.000000,0.5
5,Domain_HGB,ZeroShot_10pct_external_test,37980,0.199717,0.059048,0.168992,-0.220767,0.461749,0.920221,0.003277,0.900564,0.902128,0.003949,0.000527,0.5
7,Domain_XGB,ZeroShot_10pct_external_test,37980,0.187819,0.058325,0.167659,-0.232400,0.457814,0.912059,0.003569,0.900667,0.900313,0.001843,0.000263,0.5
0,Domain_Logistic,ZeroShot_balanced_external_test,20000,0.728419,0.821010,0.622340,-0.088664,0.472900,0.868600,0.077200,0.300890,0.328928,0.467600,0.000000,0.5
2,Domain_RF,ZeroShot_balanced_external_test,20000,0.247036,0.372380,0.633008,-0.190042,0.463550,0.925300,0.001800,0.518575,0.521028,0.000000,0.000000,0.5
4,Domain_HGB,ZeroShot_balanced_external_test,20000,0.203577,0.351015,0.630091,-0.193037,0.460950,0.918200,0.003700,0.523976,0.534167,0.004100,0.000700,0.5
6,Domain_XGB,ZeroShot_balanced_external_test,20000,0.191516,0.345396,0.626192,-0.202900,0.456950,0.909700,0.004200,0.524564,0.535472,0.001200,0.000100,0.5



--- EXTERNAL RECALIBRATED ---


,Model,Protocol,n,AUROC,AUPRC,F1,MCC,BalancedAccuracy,Sensitivity,Specificity,Brier,ECE,TPR_at_FPR_1pct,TPR_at_FPR_0_1pct,threshold
1,Domain_Logistic,PlattRecalibrated_10pct_external_test,37980,0.733984,0.593847,0.565217,0.598195,0.699315,0.400474,0.998157,0.055561,0.011845,0.478146,0.000000,0.5
7,Domain_XGB,PlattRecalibrated_10pct_external_test,37980,0.812181,0.404593,0.303214,0.325427,0.592900,0.197472,0.988327,0.073560,0.026460,0.180358,0.038968,0.5
3,Domain_RF,PlattRecalibrated_10pct_external_test,37980,0.756368,0.363760,0.207866,0.250385,0.558174,0.124539,0.991809,0.074638,0.009068,0.140600,0.044497,0.5
5,Domain_HGB,PlattRecalibrated_10pct_external_test,37980,0.800283,0.386187,0.293406,0.331830,0.588292,0.185097,0.991487,0.076289,0.038766,0.191943,0.001580,0.5
0,Domain_Logistic,PlattRecalibrated_balanced_external_test,20000,0.728419,0.821010,0.568224,0.494593,0.697800,0.397700,0.997900,0.256994,0.257878,0.467600,0.000000,0.5
2,Domain_RF,PlattRecalibrated_balanced_external_test,20000,0.752964,0.743577,0.222496,0.233531,0.558650,0.126300,0.991000,0.309710,0.334118,0.139300,0.045100,0.5
6,Domain_XGB,PlattRecalibrated_balanced_external_test,20000,0.808484,0.807659,0.320928,0.296925,0.590350,0.193600,0.987100,0.310129,0.325856,0.173000,0.034500,0.5
4,Domain_HGB,PlattRecalibrated_balanced_external_test,20000,0.796423,0.798344,0.307679,0.295115,0.587100,0.183500,0.990700,0.323101,0.335578,0.190900,0.000000,0.5



[OK] Cell 4C v6.1 completed with strict domain-group isolation.
Conflict audit: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/tables/domainonly_external_conflicting_domains_v6_1.csv
Domain split audit: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/tables/domainonly_external_domain_split_v6_1.csv


In [ ]:
# CELL 5 — FINAL ROBUST PARSER FOR ReceiverAndDeceiver_PublicData.csv
#
# This version does NOT depend on descriptive column names.
# It identifies the three 40-item PDT blocks from their documented measurement properties:
#
#   1) Perceived maliciousness: 0–100, one rating per email.
#   2) On-task confidence: six-point 0–100 scale {0,20,40,60,80,100}.
#   3) Behavioural action: three choices (KEEP / TRASH / SEEK MORE INFORMATION).
#
# Published study statistics are used only to identify/validate the correct raw columns.
# They are NOT inserted as experimental outcomes.

human_canon_path = DIRS["processed"] / "human_calibration_canonical.parquet"
canon_report_path = DIRS["reports"] / "human_canonicalization_report.json"

from collections import defaultdict

def _clean(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).strip().lower()).strip("_")

def _read_public_table(fig_dir):
    candidates = []
    for p in Path(fig_dir).rglob("*"):
        if not p.is_file() or p.suffix.lower() not in {".csv",".xlsx",".xls",".sav",".dta",".parquet"}:
            continue
        try:
            d = load_tabular_file(p)
        except Exception:
            continue
        if d is None:
            continue
        if 100 <= len(d) <= 300 and d.shape[1] >= 400:
            candidates.append((d.shape[1], p, d.copy()))
    if not candidates:
        raise FileNotFoundError("ReceiverAndDeceiver participant-level public table was not found.")
    candidates.sort(key=lambda z: z[0], reverse=True)
    _, p, d = candidates[0]
    return p, d

def _read_codebook(fig_dir):
    for p in Path(fig_dir).rglob("*"):
        if not p.is_file() or p.suffix.lower() not in {".csv",".xlsx",".xls"}:
            continue
        try:
            d = load_tabular_file(p)
        except Exception:
            continue
        if d is None:
            continue
        cc = {_clean(c):c for c in d.columns}
        if {"columns","variable","definition"}.issubset(set(cc)):
            return p, d
    return None, None

def _find_40_families(columns):
    groups = defaultdict(dict)
    for col in map(str, columns):
        for m in re.finditer(r"\d+", col):
            n = int(m.group())
            if 1 <= n <= 40:
                sig = col[:m.start()] + "{ITEM}" + col[m.end():]
                groups[sig].setdefault(n, col)
    families, seen = [], set()
    for sig, d in groups.items():
        if set(d.keys()) == set(range(1,41)):
            cols = tuple(d[i] for i in range(1,41))
            if cols not in seen:
                seen.add(cols)
                families.append({"signature":sig, "columns":list(cols)})
    return families

def _norm01(x):
    x = pd.to_numeric(x, errors="coerce").astype(float)
    if x.notna().sum() == 0:
        return x
    lo, hi = float(x.min()), float(x.max())
    if lo >= 0 and hi <= 1:
        return x
    if lo >= 0 and hi <= 6:
        return (x-lo) / max(hi-lo, 1e-12)
    if lo >= 0 and hi <= 100:
        return x/100.0
    return (x-lo) / max(hi-lo, 1e-12)

def _median_nunique(wide, cols):
    return float(np.median([wide[c].nunique(dropna=True) for c in cols]))

def _cronbach_alpha_matrix(X):
    X = pd.DataFrame(X).apply(pd.to_numeric, errors="coerce")
    X = X.apply(lambda s: s.fillna(s.median()), axis=0)
    if X.shape[1] < 2:
        return np.nan
    k = X.shape[1]
    total_var = X.sum(axis=1).var(ddof=1)
    if total_var <= 0:
        return np.nan
    return float((k/(k-1))*(1-X.var(axis=0,ddof=1).sum()/total_var))

def _family_stats(wide, family):
    cols = family["columns"]
    pooled = pd.concat([pd.to_numeric(wide[c], errors="coerce") for c in cols], ignore_index=True)
    numeric_fraction = pooled.notna().mean()
    med_u = _median_nunique(wide, cols)

    ph = _norm01(pd.concat([wide[c] for c in cols[:20]], ignore_index=True))
    ge = _norm01(pd.concat([wide[c] for c in cols[20:]], ignore_index=True))
    ph_mean = float(ph.mean()) if ph.notna().any() else np.nan
    ge_mean = float(ge.mean()) if ge.notna().any() else np.nan

    vals = np.sort(pooled.dropna().unique())
    refs = [
        np.array([0,20,40,60,80,100], dtype=float),
        np.array([1,2,3,4,5,6], dtype=float),
        np.array([0,1,2,3,4,5], dtype=float),
    ]
    def grid_match(ref):
        if len(vals)==0:
            return 0.0
        return float(np.mean([np.min(np.abs(ref-v)) <= 1e-6 for v in vals]))
    conf_grid = max(grid_match(r) for r in refs)

    a_ph = _cronbach_alpha_matrix(wide[cols[:20]]) if numeric_fraction > .90 else np.nan
    a_ge = _cronbach_alpha_matrix(wide[cols[20:]]) if numeric_fraction > .90 else np.nan

    return {
        "signature":family["signature"],
        "first_col":cols[0], "last_col":cols[-1],
        "median_nunique":med_u,
        "numeric_fraction":float(numeric_fraction),
        "ph_mean_norm":ph_mean, "ge_mean_norm":ge_mean,
        "confidence_grid_match":conf_grid,
        "alpha_ph":a_ph, "alpha_ge":a_ge,
        "columns":cols,
    }

def _exact_family(wide, templates):
    lower = {str(c).lower():str(c) for c in wide.columns}
    for tmpl in templates:
        cols, ok = [], True
        for i in range(1,41):
            key = tmpl.format(i=i).lower()
            if key not in lower:
                ok = False
                break
            cols.append(lower[key])
        if ok:
            return cols
    return None

def _decode_action_family(wide, cols, risk_long):
    raw = pd.concat([wide[c] for c in cols], ignore_index=True)
    s = raw.astype(str).str.lower()
    action = pd.Series(np.nan, index=raw.index, dtype=object)

    action[s.str.contains(r"seek|more info|investigate|verify|check", regex=True, na=False)] = "seek"
    action[s.str.contains(r"trash|delete|discard|remove", regex=True, na=False)] = "trash"
    action[s.str.contains(r"keep|retain|leave", regex=True, na=False)] = "keep"

    mapping = {}
    if action.notna().mean() < .80:
        num = pd.to_numeric(raw, errors="coerce")
        codes = sorted(num.dropna().unique().tolist())
        if len(codes) != 3:
            return None, {}, np.inf, {}

        z = pd.DataFrame({"code":num, "risk":risk_long}).dropna()
        means = z.groupby("code")["risk"].mean().sort_values()
        if len(means) != 3:
            return None, {}, np.inf, {}
        mapping = {means.index[0]:"keep", means.index[1]:"seek", means.index[2]:"trash"}
        action = num.map(mapping)

    if action.notna().mean() < .80:
        return None, mapping, np.inf, {}

    n = len(wide)
    ph = action.iloc[:20*n]
    ge = action.iloc[20*n:40*n]

    obs = {
        "ph_trash":float((ph=="trash").mean()),
        "ph_keep":float((ph=="keep").mean()),
        "ph_seek":float((ph=="seek").mean()),
        "ge_trash":float((ge=="trash").mean()),
        "ge_keep":float((ge=="keep").mean()),
        "ge_seek":float((ge=="seek").mean()),
    }
    target = {
        "ph_trash":.66, "ph_keep":.26, "ph_seek":.08,
        "ge_trash":.22, "ge_keep":.71, "ge_seek":.07,
    }
    dist = float(np.mean([abs(obs[k]-target[k]) for k in target]))
    return action, mapping, dist, obs

primary_file, wide = _read_public_table(fig_dir)
codebook_file, codebook_df = _read_codebook(fig_dir)

print("Primary human file:", primary_file)
print("Shape:", wide.shape)
print("Codebook:", codebook_file)

families = _find_40_families(wide.columns)
profiles = [_family_stats(wide,f) for f in families]
profile_df = pd.DataFrame([{k:v for k,v in p.items() if k!="columns"} for p in profiles])
save_df(profile_df, DIRS["reports"]/"PDT_40item_family_profiles.csv")
print("Detected repeated 40-item families:", len(families))

# 1) PERCEIVED MALICIOUSNESS
risk_cols = _exact_family(wide, [
    "PhishingQ{i}L",
    "PhishingQ{i}Mal",
    "PhishingQ{i}Malicious",
    "PhQ{i}L",
])
if risk_cols is None:
    candidates = []
    for p in profiles:
        if p["numeric_fraction"] < .95 or p["median_nunique"] < 8:
            continue
        if np.isnan(p["ph_mean_norm"]) or np.isnan(p["ge_mean_norm"]):
            continue
        score = abs(p["ph_mean_norm"]-.664) + abs(p["ge_mean_norm"]-.278)
        candidates.append((score,p))
    if not candidates:
        raise ValueError("No defensible perceived-maliciousness block found.")
    candidates.sort(key=lambda z:z[0])
    risk_cols = candidates[0][1]["columns"]

risk_long = _norm01(pd.concat([wide[c] for c in risk_cols], ignore_index=True))

# 2) DECISION CONFIDENCE — six-point scale
conf_cols = _exact_family(wide, [
    "PhishingQ{i}C",
    "PhishingQ{i}Conf",
    "PhishingQ{i}Confidence",
    "PhQ{i}C",
    "PhQ{i}Conf",
])
if conf_cols is None:
    conf_candidates = []
    for p in profiles:
        if tuple(p["columns"]) == tuple(risk_cols):
            continue
        if p["numeric_fraction"] < .95:
            continue
        if not (4 <= p["median_nunique"] <= 7):
            continue
        sig = _clean(p["signature"])
        if any(k in sig for k in ["right","wrong","correct","accuracy","acc_"]):
            continue
        alpha_ok = (not np.isnan(p["alpha_ph"]) and not np.isnan(p["alpha_ge"]) and
                    p["alpha_ph"] >= .75 and p["alpha_ge"] >= .75)
        grid_ok = p["confidence_grid_match"] >= .80
        if not (grid_ok or alpha_ok):
            continue
        alpha_dist = abs(p["alpha_ph"]-.93) + abs(p["alpha_ge"]-.94) if alpha_ok else 10
        score = alpha_dist + (1-p["confidence_grid_match"])
        conf_candidates.append((score,p))

    if not conf_candidates:
        cand = profile_df[
            (profile_df.numeric_fraction >= .90) &
            (profile_df.median_nunique.between(3,8))
        ].sort_values(["confidence_grid_match","alpha_ph"], ascending=False)
        save_df(cand, DIRS["reports"]/"PDT_confidence_candidates.csv")
        raise ValueError("Confidence block unresolved; see reports/PDT_confidence_candidates.csv.")

    conf_candidates.sort(key=lambda z:z[0])
    conf_cols = conf_candidates[0][1]["columns"]

# 3) RAW THREE-CHOICE ACTION
action_cols = _exact_family(wide, [
    "PhishingQ{i}A",
    "PhishingQ{i}Action",
    "PhQ{i}A",
])
action_long = None
action_mapping, action_obs, action_dist = {}, {}, np.inf

if action_cols is not None:
    action_long, action_mapping, action_dist, action_obs = _decode_action_family(
        wide, action_cols, risk_long
    )
    if action_long is None:
        action_cols = None

if action_cols is None:
    candidates = []
    for p in profiles:
        if tuple(p["columns"]) in {tuple(risk_cols), tuple(conf_cols)}:
            continue
        sig = _clean(p["signature"])
        if any(k in sig for k in ["rightseek","rightkeep","righttrash","wrong","correct","accuracy"]):
            continue
        if not (2.5 <= p["median_nunique"] <= 4.0):
            continue
        decoded, mapping, dist, obs = _decode_action_family(wide, p["columns"], risk_long)
        if decoded is not None:
            candidates.append((dist,p,decoded,mapping,obs))

    if not candidates:
        raise ValueError("No raw three-choice action block found; see PDT_40item_family_profiles.csv.")

    candidates.sort(key=lambda z:z[0])
    action_dist, best, action_long, action_mapping, action_obs = candidates[0]
    action_cols = best["columns"]

# Build canonical long format
id_candidates = [c for c in wide.columns if _clean(c) in {
    "id","participant","participant_id","subject","subject_id","respondent","respondent_id"
}]
pid_col = id_candidates[0] if id_candidates else None

records = []
for ridx,row in wide.iterrows():
    pid = str(row[pid_col]) if pid_col else str(ridx)
    for item in range(1,41):
        records.append({
            "participant_id":pid,
            "item_id":item,
            "actual_risk":1.0 if item <= 20 else 0.0,
            "ground_truth":"phishing" if item <= 20 else "genuine",
            "perceived_risk_raw":row[risk_cols[item-1]],
            "decision_confidence_raw":row[conf_cols[item-1]],
            "action_raw":row[action_cols[item-1]],
            "source_dataset":"DeceiverAndReceiver_2018",
            "perceived_risk_proxy":False,
        })

human = pd.DataFrame(records)
human["perceived_risk"] = _norm01(human["perceived_risk_raw"])
human["decision_confidence"] = _norm01(human["decision_confidence_raw"])

# item-major decoded action -> participant-major
n = len(wide)
action_matrix = pd.DataFrame({
    item: action_long.iloc[(item-1)*n:item*n].to_numpy()
    for item in range(1,41)
})
action_matrix.index = wide.index
acts = []
for ridx in wide.index:
    for item in range(1,41):
        acts.append(action_matrix.loc[ridx,item])
human["action"] = acts

human["correct"] = np.where(
    human["ground_truth"].eq("phishing"),
    human["action"].isin(["trash","seek"]).astype(float),
    human["action"].isin(["keep","seek"]).astype(float),
)

# Corrected CRB constructs
human["perceived_safety"] = 1.0 - human["perceived_risk"]
human["calibration_error"] = (human["actual_risk"] - human["perceived_risk"]).abs()
human["overconfidence"] = (human["actual_risk"] - human["perceived_risk"]).clip(lower=0)
human["underconfidence"] = (human["perceived_risk"] - human["actual_risk"]).clip(lower=0)
human["hazardous_overconfidence"] = human["overconfidence"] * human["decision_confidence"]
human["incorrect"] = 1.0 - human["correct"]
human["security_failure"] = ((human["actual_risk"]>=.5) & (human["correct"]==0)).astype(float)
human["draft_symmetric_gap"] = (human["perceived_safety"] - human["actual_risk"]).abs()
human["draft_overconfidence_term"] = (human["perceived_safety"] - human["actual_risk"]).clip(lower=0)

# Validation
conf_alpha_ph = _cronbach_alpha_matrix(wide[conf_cols[:20]])
conf_alpha_ge = _cronbach_alpha_matrix(wide[conf_cols[20:]])
ph = human[human.ground_truth.eq("phishing")]
ge = human[human.ground_truth.eq("genuine")]

validation = {
    "source_file":str(primary_file),
    "codebook_file":str(codebook_file) if codebook_file else None,
    "rows":int(len(human)),
    "participants":int(human.participant_id.nunique()),
    "items":int(human.item_id.nunique()),
    "risk_first_col":str(risk_cols[0]),
    "risk_last_col":str(risk_cols[-1]),
    "confidence_first_col":str(conf_cols[0]),
    "confidence_last_col":str(conf_cols[-1]),
    "action_first_col":str(action_cols[0]),
    "action_last_col":str(action_cols[-1]),
    "phishing_maliciousness_mean":float(ph.perceived_risk.mean()*100),
    "genuine_maliciousness_mean":float(ge.perceived_risk.mean()*100),
    "phishing_accuracy":float(ph.correct.mean()),
    "genuine_accuracy":float(ge.correct.mean()),
    "phishing_conf_alpha":float(conf_alpha_ph),
    "genuine_conf_alpha":float(conf_alpha_ge),
    "action_mapping_if_numeric":{str(k):v for k,v in action_mapping.items()},
    "parsed_action_distribution":action_obs,
    "action_distribution_mean_abs_error_vs_published":float(action_dist),
}

assert len(human) == 150*40
assert human.perceived_risk.notna().mean() > .95
assert human.decision_confidence.notna().mean() > .95
assert human.action.notna().mean() > .95

if abs(validation["phishing_maliciousness_mean"]-66.4) > 12:
    raise ValueError("Maliciousness block failed phishing-mean validation.")
if abs(validation["genuine_maliciousness_mean"]-27.8) > 12:
    raise ValueError("Maliciousness block failed genuine-mean validation.")
if abs(validation["phishing_accuracy"]-.739) > .12:
    raise ValueError("Action block failed phishing-accuracy validation.")
if abs(validation["genuine_accuracy"]-.781) > .12:
    raise ValueError("Action block failed genuine-accuracy validation.")
if action_dist > .12:
    raise ValueError("Action response proportions are too far from published PDT rates.")
if conf_alpha_ph < .75 or conf_alpha_ge < .75:
    raise ValueError("Confidence block failed reliability validation.")

# Save traceable mapping and data only after validation passes
mapping_rows = []
for role,cols in [("perceived_risk",risk_cols),("decision_confidence",conf_cols),("action",action_cols)]:
    for item,c in enumerate(cols,1):
        mapping_rows.append({"role":role,"item_id":item,"column":str(c)})
mapping_df = pd.DataFrame(mapping_rows)

save_df(mapping_df, DIRS["reports"]/"PDT_40item_mapping.csv")
save_df(human, human_canon_path)
save_json(validation, canon_report_path)
checkpoint("human_canonical_V4", {
    "rows":len(human),
    "participants":int(human.participant_id.nunique()),
    "items":40,
    "parser":"measurement-fingerprint-v4"
})

print("\n[OK] CELL 5 PASSED ALL DATA-MAPPING VALIDATION GATES.")
print("Selected raw blocks:")
print("  perceived maliciousness :", risk_cols[0], "...", risk_cols[-1])
print("  decision confidence     :", conf_cols[0], "...", conf_cols[-1])
print("  behavioural action      :", action_cols[0], "...", action_cols[-1])

display(pd.DataFrame([validation]).T.rename(columns={0:"value"}))
display(
    human.groupby("ground_truth").agg(
        n=("participant_id","size"),
        participants=("participant_id","nunique"),
        maliciousness_mean=("perceived_risk","mean"),
        confidence_mean=("decision_confidence","mean"),
        accuracy=("correct","mean"),
        calibration_error=("calibration_error","mean"),
        overconfidence=("overconfidence","mean"),
        underconfidence=("underconfidence","mean"),
    )
)

Primary human file: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/raw/figshare_7205903/ReceiverAndDeceiver_DataSet_extracted/ReceiverAndDeceiver_PublicData.csv
Shape: (150, 524)
Codebook: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/raw/figshare_7205903/ReceiverAndDeceiver_DataSet_extracted/ReceiverAndDeceiver_CodeBook.csv
Detected repeated 40-item families: 9

[OK] CELL 5 PASSED ALL DATA-MAPPING VALIDATION GATES.
Selected raw blocks:
  perceived maliciousness : PhishingQ1L ... PhishingQ40L
  decision confidence     : PhishingQ1C ... PhishingQ40C
  behavioural action      : PhishingQ1A ... PhishingQ40A


,value
source_file,/content/drive/MyDrive/Cyber_Confidence_Q1_Ext...
codebook_file,/content/drive/MyDrive/Cyber_Confidence_Q1_Ext...
rows,6000
participants,150
items,40
risk_first_col,PhishingQ1L
risk_last_col,PhishingQ40L
confidence_first_col,PhishingQ1C
confidence_last_col,PhishingQ40C
action_first_col,PhishingQ1A


,n,participants,maliciousness_mean,confidence_mean,accuracy,calibration_error,overconfidence,underconfidence
ground_truth,,,,,,,,
genuine,3000,150,0.278430,0.740470,0.777667,0.278430,0.000000,0.27843
phishing,3000,150,0.663795,0.746107,0.728667,0.336205,0.336205,0.00000


In [ ]:

# CELL 6 — Human-data validation, descriptive analyses, reliability, signal-detection metrics
# Restart-safe output tables.

desc_path = DIRS["tables"] / "human_descriptives.csv"
sdt_path = DIRS["tables"] / "signal_detection_by_participant.csv"
rel_path = DIRS["tables"] / "human_reliability.csv"

if all(artifact_ok(p) for p in [desc_path, sdt_path, rel_path]) and not FORCE:
    print("[LOAD] Human descriptives / SDT / reliability")
    human_desc = pd.read_csv(desc_path)
    sdt = pd.read_csv(sdt_path)
    reliability = pd.read_csv(rel_path)
else:
    print("[RUN ] Human descriptive/statistical preparation")

    # Use direct perceived-risk ratings as PRIMARY. Proxies remain secondary.
    primary = human[
        human["source_dataset"].eq("DeceiverAndReceiver_2018") &
        human["perceived_risk"].notna() &
        human["decision_confidence"].notna() &
        human["correct"].notna()
    ].copy()

    assert len(primary) > 100, "Primary calibration dataset too small after parsing; inspect canonicalization report."

    human_desc = (
        primary.groupby(["actual_risk"])
        .agg(
            n=("participant_id","size"),
            participants=("participant_id","nunique"),
            perceived_risk_mean=("perceived_risk","mean"),
            perceived_risk_sd=("perceived_risk","std"),
            confidence_mean=("decision_confidence","mean"),
            confidence_sd=("decision_confidence","std"),
            accuracy=("correct","mean"),
            calibration_error=("calibration_error","mean"),
            overconfidence=("overconfidence","mean"),
            underconfidence=("underconfidence","mean"),
        ).reset_index()
    )

    # Signal detection theory by participant.
    def sdt_one(g):
        ph = g[g.actual_risk >= .5]
        ge = g[g.actual_risk < .5]
        hits = ph["correct"].sum()
        misses = len(ph) - hits
        # False alarm = genuine classified as phishing/trash => incorrect genuine.
        fas = (1-ge["correct"]).sum()
        crs = ge["correct"].sum()

        # log-linear correction avoids +/-inf
        hit_rate = (hits + 0.5) / (hits + misses + 1.0) if len(ph) else np.nan
        fa_rate = (fas + 0.5) / (fas + crs + 1.0) if len(ge) else np.nan
        if np.isnan(hit_rate) or np.isnan(fa_rate):
            return pd.Series({"d_prime":np.nan,"criterion":np.nan,"hit_rate":hit_rate,"false_alarm_rate":fa_rate})
        zH, zF = stats.norm.ppf(hit_rate), stats.norm.ppf(fa_rate)
        return pd.Series({
            "d_prime": zH-zF,
            "criterion": -0.5*(zH+zF),
            "hit_rate": hit_rate,
            "false_alarm_rate": fa_rate
        })

    sdt = primary.groupby("participant_id", group_keys=False).apply(sdt_one).reset_index()

    # Cronbach alpha across items for direct perceived risk/confidence/correctness.
    def cronbach_alpha(wide):
        X = wide.dropna(axis=1, how="all").dropna(axis=0, how="all")
        if X.shape[1] < 2:
            return np.nan
        # pairwise impute item medians for reliability diagnostic only
        X = X.apply(lambda c: c.fillna(c.median()))
        item_vars = X.var(axis=0, ddof=1)
        total_var = X.sum(axis=1).var(ddof=1)
        k = X.shape[1]
        return (k/(k-1))*(1-item_vars.sum()/total_var) if total_var > 0 else np.nan

    rel_rows = []
    for metric in ["perceived_risk","decision_confidence","correct"]:
        for riskv, label in [(1.0,"phishing"),(0.0,"genuine")]:
            w = primary[primary.actual_risk.eq(riskv)].pivot_table(
                index="participant_id", columns="item_id", values=metric, aggfunc="first"
            )
            rel_rows.append({
                "metric":metric, "email_type":label,
                "n_participants":len(w), "n_items":w.shape[1],
                "cronbach_alpha":cronbach_alpha(w)
            })
    reliability = pd.DataFrame(rel_rows)

    save_df(human_desc, desc_path)
    save_df(sdt, sdt_path)
    save_df(reliability, rel_path)
    checkpoint("human_descriptives", {"primary_rows":len(primary),"participants":primary.participant_id.nunique()})

display(human_desc)
display(reliability)
display(sdt.describe(include="all").T.head(12))


[LOAD] Human descriptives / SDT / reliability


,actual_risk,n,participants,perceived_risk_mean,perceived_risk_sd,confidence_mean,confidence_sd,accuracy,calibration_error,overconfidence,underconfidence
0,0.0,2980,149,0.278430,0.284517,0.740470,0.232244,0.782886,0.278430,0.000000,0.27843
1,1.0,2980,149,0.663795,0.304405,0.746107,0.238340,0.733557,0.336205,0.336205,0.00000


,metric,email_type,n_participants,n_items,cronbach_alpha
0,perceived_risk,phishing,149,20,0.861875
1,perceived_risk,genuine,149,20,0.891827
2,decision_confidence,phishing,149,20,0.931518
3,decision_confidence,genuine,149,20,0.941075
4,correct,phishing,149,20,0.821790
5,correct,genuine,149,20,0.807736


,count,mean,std,min,25%,50%,75%,max
participant_id,149.0,75.503356,43.591876,1.000000,38.000000,76.000000,113.000000,150.000000
d_prime,149.0,1.601802,0.828949,0.000000,1.060113,1.465234,2.100401,3.961505
criterion,149.0,0.101437,0.539206,-1.386196,-0.242877,0.000000,0.423887,1.980752
hit_rate,149.0,0.722435,0.189867,0.023810,0.595238,0.785714,0.880952,0.976190
false_alarm_rate,149.0,0.230585,0.178155,0.023810,0.071429,0.214286,0.357143,0.833333


In [ ]:

# CELL 7 — Predictive benchmarking with repeated participant-grouped CV
# Primary endpoint: security_failure = missed phishing / unsafe response to phishing.
# Participants never appear in both train and test within a fold.

bench_pred_path = DIRS["processed"] / "human_benchmark_oof_predictions.parquet"
bench_metrics_path = DIRS["tables"] / "human_benchmark_fold_metrics.csv"
bench_summary_path = DIRS["tables"] / "human_benchmark_summary.csv"

def ece_score(y, p, n_bins=10):
    y = np.asarray(y); p = np.asarray(p)
    bins = np.linspace(0,1,n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        mask = (p >= bins[i]) & (p < bins[i+1] if i < n_bins-1 else p <= bins[i+1])
        if mask.sum():
            ece += mask.mean() * abs(y[mask].mean() - p[mask].mean())
    return float(ece)

def metric_row(y, p):
    pr = (np.asarray(p) >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
    return {
        "AUROC": roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan,
        "AUPRC": average_precision_score(y,p) if len(np.unique(y))>1 else np.nan,
        "F1": f1_score(y,pr,zero_division=0),
        "MCC": matthews_corrcoef(y,pr),
        "BalancedAccuracy": balanced_accuracy_score(y,pr),
        "Sensitivity": tp/(tp+fn) if (tp+fn) else np.nan,
        "Specificity": tn/(tn+fp) if (tn+fp) else np.nan,
        "Brier": brier_score_loss(y,p),
        "LogLoss": log_loss(y,np.clip(p,1e-8,1-1e-8)),
        "ECE": ece_score(y,p,10),
    }

if all(artifact_ok(p) for p in [bench_pred_path, bench_metrics_path, bench_summary_path]) and not FORCE:
    print("[LOAD] Human predictive benchmark")
    oof = pd.read_parquet(bench_pred_path)
    fold_metrics = pd.read_csv(bench_metrics_path)
    bench_summary = pd.read_csv(bench_summary_path)
else:
    print("[RUN ] Human predictive benchmark")
    d = human[
        human["source_dataset"].eq("DeceiverAndReceiver_2018") &
        human["perceived_risk"].notna() &
        human["decision_confidence"].notna() &
        human["correct"].notna()
    ].copy()

    # Main endpoint is meaningful only for phishing tasks:
    # a "security failure" is a missed phishing message.
    d = d[d.actual_risk >= 0.5].copy()
    d["y"] = (1-d["correct"]).astype(int)
    d["oc_x_conf"] = d["overconfidence"] * d["decision_confidence"]

    # Keep the proposed model interpretable.
    feature_sets = {
        "ConfidenceOnly_LR": ["decision_confidence"],
        "PerceivedRisk_LR": ["perceived_risk"],
        "SymmetricCalibration_LR": ["calibration_error","decision_confidence"],
        "DraftTerms_LR": ["draft_symmetric_gap","draft_overconfidence_term","decision_confidence"],
        "ProposedCRB_LR": ["overconfidence","decision_confidence","oc_x_conf"],
        "ProposedCRB_RF": ["overconfidence","decision_confidence","oc_x_conf"],
        "ProposedCRB_HGB": ["overconfidence","decision_confidence","oc_x_conf"],
    }
    if HAVE_XGB:
        feature_sets["ProposedCRB_XGB"] = ["overconfidence","decision_confidence","oc_x_conf"]

    def make_model(name):
        if name.endswith("_LR"):
            return Pipeline([
                ("imp",SimpleImputer(strategy="median")),
                ("sc",StandardScaler()),
                ("clf",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEED))
            ])
        if name.endswith("_RF"):
            return Pipeline([
                ("imp",SimpleImputer(strategy="median")),
                ("clf",RandomForestClassifier(
                    n_estimators=400,min_samples_leaf=5,class_weight="balanced_subsample",
                    n_jobs=-1,random_state=SEED
                ))
            ])
        if name.endswith("_HGB"):
            return Pipeline([
                ("imp",SimpleImputer(strategy="median")),
                ("clf",HistGradientBoostingClassifier(max_iter=200,learning_rate=.05,random_state=SEED))
            ])
        if name.endswith("_XGB"):
            return Pipeline([
                ("imp",SimpleImputer(strategy="median")),
                ("clf",XGBClassifier(
                    n_estimators=250,max_depth=4,learning_rate=.05,
                    subsample=.85,colsample_bytree=.9,eval_metric="logloss",
                    random_state=SEED,n_jobs=-1
                ))
            ])
        raise ValueError(name)

    pred_rows = []
    metric_rows = []
    for rep in range(CONFIG["cv_repeats"]):
        cv = StratifiedGroupKFold(
            n_splits=CONFIG["cv_splits"], shuffle=True, random_state=SEED+rep
        )
        for fold, (tri, tei) in enumerate(cv.split(d, d["y"], groups=d["participant_id"])):
            tr, te = d.iloc[tri], d.iloc[tei]
            for name, feats in feature_sets.items():
                model = make_model(name)
                model.fit(tr[feats], tr["y"])
                p = model.predict_proba(te[feats])[:,1]
                mm = metric_row(te["y"].to_numpy(), p)
                metric_rows.append({
                    "repeat":rep,"fold":fold,"model":name,
                    "n_test":len(te),**mm
                })
                pred_rows.append(pd.DataFrame({
                    "repeat":rep,"fold":fold,"model":name,
                    "row_index":te.index.to_numpy(),
                    "participant_id":te["participant_id"].astype(str).to_numpy(),
                    "item_id":te["item_id"].astype(str).to_numpy(),
                    "y_true":te["y"].to_numpy(),
                    "prob":p,
                }))

    oof = pd.concat(pred_rows, ignore_index=True)
    fold_metrics = pd.DataFrame(metric_rows)
    bench_summary = (
        fold_metrics.groupby("model")
        .agg(
            AUROC_mean=("AUROC","mean"), AUROC_sd=("AUROC","std"),
            AUPRC_mean=("AUPRC","mean"), AUPRC_sd=("AUPRC","std"),
            F1_mean=("F1","mean"), MCC_mean=("MCC","mean"),
            BalAcc_mean=("BalancedAccuracy","mean"),
            Brier_mean=("Brier","mean"), ECE_mean=("ECE","mean")
        ).reset_index()
        .sort_values(["AUPRC_mean","Brier_mean"], ascending=[False,True])
    )

    save_df(oof, bench_pred_path)
    save_df(fold_metrics, bench_metrics_path)
    save_df(bench_summary, bench_summary_path)
    checkpoint("human_benchmark", {"rows":len(d),"participants":d.participant_id.nunique(),
                                   "models":list(feature_sets),"repeats":CONFIG["cv_repeats"]})

display(bench_summary)


[LOAD] Human predictive benchmark


,model,AUROC_mean,AUROC_sd,AUPRC_mean,AUPRC_sd,F1_mean,MCC_mean,BalAcc_mean,Brier_mean,ECE_mean
0,ProposedCRB_LR,0.888158,0.022389,0.712365,0.051248,0.691671,0.572462,0.815288,0.138912,0.127890
1,PerceivedRisk_LR,0.886417,0.022173,0.707114,0.050236,0.694234,0.575289,0.813474,0.137257,0.129955
2,DraftTerms_LR,0.886965,0.023710,0.703908,0.050142,0.694008,0.574949,0.814076,0.137819,0.127429
3,SymmetricCalibration_LR,0.886965,0.023710,0.703908,0.050142,0.694008,0.574949,0.814076,0.137819,0.127429
4,ProposedCRB_XGB,0.873021,0.026229,0.695929,0.046287,0.649215,0.532317,0.761280,0.125786,0.054818
5,ProposedCRB_RF,0.867342,0.026993,0.693118,0.045864,0.674482,0.546754,0.795821,0.143060,0.108509
6,ProposedCRB_HGB,0.860225,0.030503,0.683296,0.043466,0.639480,0.517968,0.754778,0.130525,0.063155
7,ConfidenceOnly_LR,0.654259,0.041850,0.362204,0.043047,0.451699,0.213964,0.617520,0.234353,0.218945


In [ ]:
# CELL 7B — Task-aware CRB + DOUBLE-COLD-START evaluation
#
# Protocol A: USER-COLD / TASK-SHARED
#   Test participants are unseen.
#   For each task, a smoothed historical task-failure prior is estimated ONLY from
#   training participants. This mimics an organization that already knows which scenarios
#   are difficult but is onboarding a new user.
#
# Protocol B: DOUBLE-COLD-START
#   Both test participants and test tasks are absent from training.
#   This is a much harder generalization test.
#
# No test outcome is used to compute task priors.

taskaware_pred_path=DIRS["processed"]/"human_taskaware_oof_predictions.parquet"
taskaware_metrics_path=DIRS["tables"]/"human_taskaware_fold_metrics.csv"
taskaware_summary_path=DIRS["tables"]/"human_taskaware_summary.csv"
doublecold_path=DIRS["tables"]/"human_double_cold_start_summary.csv"

if all(artifact_ok(p) for p in [taskaware_pred_path,taskaware_metrics_path,taskaware_summary_path,doublecold_path]) and not FORCE:
    print("[LOAD] Task-aware / double-cold-start CRB")
    task_oof=pd.read_parquet(taskaware_pred_path)
    task_fold=pd.read_csv(taskaware_metrics_path)
    task_summary=pd.read_csv(taskaware_summary_path)
    doublecold=pd.read_csv(doublecold_path)
else:
    print("[RUN ] Task-aware / double-cold-start CRB")
    d=human[
        human.source_dataset.eq("DeceiverAndReceiver_2018") &
        human.perceived_risk.notna() &
        human.decision_confidence.notna() &
        human.correct.notna() &
        human.actual_risk.ge(.5)
    ].copy()
    d["y"]=(1-d.correct).astype(int)
    d["oc_x_conf"]=d.overconfidence*d.decision_confidence

    def lr_model():
        return Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("sc",StandardScaler()),
            ("clf",LogisticRegression(max_iter=1200,class_weight="balanced",random_state=SEED))
        ])

    def _metrics(y,p):
        pr=(np.asarray(p)>=.5).astype(int)
        tn,fp,fn,tp=confusion_matrix(y,pr,labels=[0,1]).ravel()
        return {
            "AUROC":roc_auc_score(y,p),
            "AUPRC":average_precision_score(y,p),
            "F1":f1_score(y,pr),
            "MCC":matthews_corrcoef(y,pr),
            "BalancedAccuracy":balanced_accuracy_score(y,pr),
            "Brier":brier_score_loss(y,p),
            "ECE":ece_score(y,p,10),
            "Sensitivity":tp/(tp+fn) if tp+fn else np.nan,
            "Specificity":tn/(tn+fp) if tn+fp else np.nan,
        }

    # ---------- Protocol A: unseen users, known task catalogue ----------
    feature_sets={
        "PerceivedRisk_LR":["perceived_risk"],
        "CRB_LR":["overconfidence","decision_confidence","oc_x_conf"],
        "TaskPrior_LR":["task_failure_prior"],
        "CRB_TaskAware_LR":[
            "overconfidence","decision_confidence","oc_x_conf",
            "task_failure_prior","oc_x_taskprior"
        ],
    }

    pred_rows=[]
    met_rows=[]
    alpha=10.0  # empirical-Bayes shrinkage strength

    for rep in range(CONFIG["cv_repeats"]):
        cv=StratifiedGroupKFold(n_splits=CONFIG["cv_splits"],shuffle=True,random_state=SEED+100+rep)
        for fold,(tri,tei) in enumerate(cv.split(d,d.y,groups=d.participant_id)):
            tr=d.iloc[tri].copy()
            te=d.iloc[tei].copy()

            global_prior=float(tr.y.mean())
            stat=tr.groupby("item_id").y.agg(["sum","count"])
            prior=((stat["sum"]+alpha*global_prior)/(stat["count"]+alpha)).to_dict()

            tr["task_failure_prior"]=tr.item_id.map(prior).fillna(global_prior)
            te["task_failure_prior"]=te.item_id.map(prior).fillna(global_prior)
            tr["oc_x_taskprior"]=tr.overconfidence*tr.task_failure_prior
            te["oc_x_taskprior"]=te.overconfidence*te.task_failure_prior

            for name,feats in feature_sets.items():
                m=lr_model()
                m.fit(tr[feats],tr.y)
                p=m.predict_proba(te[feats])[:,1]
                met_rows.append({
                    "repeat":rep,"fold":fold,"model":name,"n_test":len(te),**_metrics(te.y,p)
                })
                pred_rows.append(pd.DataFrame({
                    "repeat":rep,"fold":fold,"model":name,
                    "row_index":te.index.to_numpy(),
                    "participant_id":te.participant_id.astype(str).to_numpy(),
                    "item_id":te.item_id.astype(str).to_numpy(),
                    "y_true":te.y.to_numpy(),"prob":p,
                }))

    task_oof=pd.concat(pred_rows,ignore_index=True)
    task_fold=pd.DataFrame(met_rows)
    task_summary=task_fold.groupby("model").agg(
        AUROC_mean=("AUROC","mean"),AUROC_sd=("AUROC","std"),
        AUPRC_mean=("AUPRC","mean"),AUPRC_sd=("AUPRC","std"),
        F1_mean=("F1","mean"),MCC_mean=("MCC","mean"),
        BalAcc_mean=("BalancedAccuracy","mean"),
        Brier_mean=("Brier","mean"),ECE_mean=("ECE","mean")
    ).reset_index().sort_values(["AUPRC_mean","Brier_mean"],ascending=[False,True])

    save_df(task_oof,taskaware_pred_path)
    save_df(task_fold,taskaware_metrics_path)
    save_df(task_summary,taskaware_summary_path)

    # ---------- Protocol B: both users AND tasks unseen ----------
    # 5 user folds x 5 item folds = 25 disjoint test cells covering the full user-item matrix.
    users=np.array(sorted(d.participant_id.astype(str).unique()))
    items=np.array(sorted(d.item_id.unique()))
    rng=np.random.default_rng(SEED)
    rng.shuffle(users); rng.shuffle(items)
    user_folds=np.array_split(users,5)
    item_folds=np.array_split(items,5)

    dc_rows=[]
    dc_preds=[]
    base_feats=["perceived_risk"]
    crb_feats=["overconfidence","decision_confidence","oc_x_conf"]

    for uf_i,uf in enumerate(user_folds):
        for it_i,itf in enumerate(item_folds):
            test_mask=d.participant_id.astype(str).isin(uf) & d.item_id.isin(itf)
            train_mask=(~d.participant_id.astype(str).isin(uf)) & (~d.item_id.isin(itf))
            tr=d[train_mask].copy()
            te=d[test_mask].copy()
            if len(te)<20 or te.y.nunique()<2:
                continue
            for name,feats in [("PerceivedRisk_LR",base_feats),("CRB_LR",crb_feats)]:
                m=lr_model()
                m.fit(tr[feats],tr.y)
                p=m.predict_proba(te[feats])[:,1]
                dc_rows.append({
                    "user_fold":uf_i,"item_fold":it_i,"model":name,"n_test":len(te),**_metrics(te.y,p)
                })
                dc_preds.append(pd.DataFrame({
                    "user_fold":uf_i,"item_fold":it_i,"model":name,
                    "participant_id":te.participant_id.astype(str).to_numpy(),
                    "item_id":te.item_id.astype(str).to_numpy(),
                    "y_true":te.y.to_numpy(),"prob":p
                }))

    dc_fold=pd.DataFrame(dc_rows)
    doublecold=dc_fold.groupby("model").agg(
        AUROC_mean=("AUROC","mean"),AUROC_sd=("AUROC","std"),
        AUPRC_mean=("AUPRC","mean"),AUPRC_sd=("AUPRC","std"),
        F1_mean=("F1","mean"),MCC_mean=("MCC","mean"),
        Brier_mean=("Brier","mean")
    ).reset_index()
    save_df(doublecold,doublecold_path)
    if dc_preds:
        save_df(pd.concat(dc_preds,ignore_index=True),DIRS["processed"]/"human_double_cold_start_predictions.parquet")

    checkpoint("human_taskaware_doublecold",{
        "taskaware_rows":len(task_oof),
        "doublecold_cells":len(dc_fold),
        "task_prior_shrinkage_alpha":alpha,
    })

display(task_summary)
display(doublecold)

[LOAD] Task-aware / double-cold-start CRB


,model,AUROC_mean,AUROC_sd,AUPRC_mean,AUPRC_sd,F1_mean,MCC_mean,BalAcc_mean,Brier_mean,ECE_mean
0,CRB_TaskAware_LR,0.890781,0.021276,0.730880,0.041659,0.693732,0.575419,0.816491,0.136391,0.123597
1,CRB_LR,0.888082,0.021717,0.714094,0.046118,0.689304,0.569458,0.814537,0.138710,0.127084
2,PerceivedRisk_LR,0.886139,0.021213,0.708873,0.046335,0.694364,0.575654,0.815038,0.137023,0.129383
3,TaskPrior_LR,0.713634,0.019182,0.430189,0.048462,0.508675,0.294194,0.664109,0.217375,0.199349


,model,AUROC_mean,AUROC_sd,AUPRC_mean,AUPRC_sd,F1_mean,MCC_mean,Brier_mean
0,CRB_LR,0.883151,0.035470,0.700642,0.116957,0.670966,0.548922,0.138777
1,PerceivedRisk_LR,0.880699,0.036302,0.693121,0.120520,0.677586,0.555614,0.137037


In [ ]:
# CELL 7C — Personalized CRB onboarding simulation
#
# Retrospective simulation:
# 5 phishing + 5 genuine items form a fixed onboarding/calibration set.
# Participant-specific profile features are computed ONLY from those onboarding items.
# The remaining phishing items are then predicted under participant-grouped CV.
#
# This is NOT a temporal field experiment. It estimates the value of a short onboarding phase.

pers_pred_path=DIRS["processed"]/"human_personalized_oof_predictions.parquet"
pers_summary_path=DIRS["tables"]/"human_personalized_summary.csv"
pers_boot_path=DIRS["tables"]/"human_personalized_cluster_bootstrap.csv"

if all(artifact_ok(p) for p in [pers_pred_path,pers_summary_path,pers_boot_path]) and not FORCE:
    print("[LOAD] Personalized onboarding CRB")
    pers_oof=pd.read_parquet(pers_pred_path)
    pers_summary=pd.read_csv(pers_summary_path)
    pers_boot=pd.read_csv(pers_boot_path)
else:
    print("[RUN ] Personalized onboarding CRB")
    allh=human[
        human.source_dataset.eq("DeceiverAndReceiver_2018") &
        human.perceived_risk.notna() &
        human.decision_confidence.notna() &
        human.correct.notna()
    ].copy()

    rng=np.random.default_rng(SEED)
    ph_items=np.array(sorted(allh.loc[allh.actual_risk.ge(.5),"item_id"].unique()))
    ge_items=np.array(sorted(allh.loc[allh.actual_risk.lt(.5),"item_id"].unique()))
    warm_ph=sorted(rng.choice(ph_items,size=5,replace=False).tolist())
    warm_ge=sorted(rng.choice(ge_items,size=5,replace=False).tolist())
    warm_items=set(warm_ph+warm_ge)

    warm=allh[allh.item_id.isin(warm_items)].copy()
    prof=warm.groupby("participant_id").agg(
        warm_cal_error=("calibration_error","mean"),
        warm_confidence=("decision_confidence","mean"),
        warm_overconfidence=("overconfidence","mean"),
        warm_underconfidence=("underconfidence","mean"),
        warm_accuracy=("correct","mean"),
    ).reset_index()

    warm_ph_df=warm[warm.actual_risk.ge(.5)].groupby("participant_id").agg(
        warm_phishing_miss=("incorrect","mean")
    ).reset_index()
    warm_ge_df=warm[warm.actual_risk.lt(.5)].groupby("participant_id").agg(
        warm_genuine_fp=("incorrect","mean")
    ).reset_index()

    prof=prof.merge(warm_ph_df,on="participant_id",how="left").merge(warm_ge_df,on="participant_id",how="left")

    test=allh[
        allh.actual_risk.ge(.5) &
        (~allh.item_id.isin(warm_items))
    ].copy().merge(prof,on="participant_id",how="left")

    test["y"]=(1-test.correct).astype(int)
    test["oc_x_conf"]=test.overconfidence*test.decision_confidence

    base_feats=["overconfidence","decision_confidence","oc_x_conf"]
    pers_feats=base_feats+[
        "warm_cal_error","warm_confidence","warm_overconfidence",
        "warm_accuracy","warm_phishing_miss","warm_genuine_fp"
    ]
    dual_feats=pers_feats+["task_failure_prior","oc_x_taskprior"]

    def make_lr():
        return Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("sc",StandardScaler()),
            ("clf",LogisticRegression(max_iter=1200,class_weight="balanced",random_state=SEED))
        ])

    rows=[]; preds=[]
    for rep in range(CONFIG["cv_repeats"]):
        cv=StratifiedGroupKFold(n_splits=CONFIG["cv_splits"],shuffle=True,random_state=SEED+500+rep)
        for fold,(tri,tei) in enumerate(cv.split(test,test.y,groups=test.participant_id)):
            tr,te=test.iloc[tri].copy(),test.iloc[tei].copy()

            # Training-only task prior: historical miss rate for the target task.
            global_prior=float(tr.y.mean())
            stat=tr.groupby("item_id").y.agg(["sum","count"])
            alpha=10.0
            prior=((stat["sum"]+alpha*global_prior)/(stat["count"]+alpha)).to_dict()
            tr["task_failure_prior"]=tr.item_id.map(prior).fillna(global_prior)
            te["task_failure_prior"]=te.item_id.map(prior).fillna(global_prior)
            tr["oc_x_taskprior"]=tr.overconfidence*tr.task_failure_prior
            te["oc_x_taskprior"]=te.overconfidence*te.task_failure_prior

            for name,feats in [
                ("CRB_Base_LR",base_feats),
                ("PersonalizedCRB_LR",pers_feats),
                ("DualAdaptiveCRB_LR",dual_feats),
            ]:
                m=make_lr()
                m.fit(tr[feats],tr.y)
                p=m.predict_proba(te[feats])[:,1]
                pred=(p>=.5).astype(int)
                rows.append({
                    "repeat":rep,"fold":fold,"model":name,"n_test":len(te),
                    "AUROC":roc_auc_score(te.y,p),
                    "AUPRC":average_precision_score(te.y,p),
                    "F1":f1_score(te.y,pred),
                    "MCC":matthews_corrcoef(te.y,pred),
                    "BalancedAccuracy":balanced_accuracy_score(te.y,pred),
                    "Brier":brier_score_loss(te.y,p),
                    "ECE":ece_score(te.y,p,10),
                })
                preds.append(pd.DataFrame({
                    "repeat":rep,"fold":fold,"model":name,
                    "row_index":te.index.to_numpy(),
                    "participant_id":te.participant_id.astype(str).to_numpy(),
                    "item_id":te.item_id.astype(str).to_numpy(),
                    "y_true":te.y.to_numpy(),"prob":p
                }))

    pers_fold=pd.DataFrame(rows)
    pers_oof=pd.concat(preds,ignore_index=True)
    pers_summary=pers_fold.groupby("model").agg(
        AUROC_mean=("AUROC","mean"),AUROC_sd=("AUROC","std"),
        AUPRC_mean=("AUPRC","mean"),AUPRC_sd=("AUPRC","std"),
        F1_mean=("F1","mean"),MCC_mean=("MCC","mean"),
        BalAcc_mean=("BalancedAccuracy","mean"),
        Brier_mean=("Brier","mean"),ECE_mean=("ECE","mean")
    ).reset_index()
    save_df(pers_oof,pers_pred_path)
    save_df(pers_summary,pers_summary_path)

    # Participant-cluster bootstrap on repeat 0.
    oo=pers_oof[pers_oof.repeat.eq(0)].pivot_table(
        index=["row_index","participant_id","item_id","y_true"],
        columns="model",values="prob",aggfunc="first"
    ).reset_index()

    participants=oo.participant_id.unique()
    rng=np.random.default_rng(SEED)
    boot_rows=[]
    for anchor,baseline in [
        ("PersonalizedCRB_LR","CRB_Base_LR"),
        ("DualAdaptiveCRB_LR","CRB_Base_LR"),
        ("DualAdaptiveCRB_LR","PersonalizedCRB_LR"),
    ]:
        da=[]; db=[]
        for _ in range(1000):
            samp=rng.choice(participants,size=len(participants),replace=True)
            bs=pd.concat([oo[oo.participant_id.eq(pid)].assign(_copy=j) for j,pid in enumerate(samp)],ignore_index=True)
            da.append(
                average_precision_score(bs.y_true,bs[anchor]) -
                average_precision_score(bs.y_true,bs[baseline])
            )
            db.append(
                brier_score_loss(bs.y_true,bs[baseline]) -
                brier_score_loss(bs.y_true,bs[anchor])
            )
        da=np.asarray(da); db=np.asarray(db)
        boot_rows.append({
            "comparison":f"{anchor} - {baseline}",
            "AUPRC_diff_mean":da.mean(),
            "AUPRC_CI_low":np.quantile(da,.025),
            "AUPRC_CI_high":np.quantile(da,.975),
            "AUPRC_boot_p":min(1.0,2*min((da<=0).mean(),(da>=0).mean())),
            "Brier_improvement_mean":db.mean(),
            "Brier_CI_low":np.quantile(db,.025),
            "Brier_CI_high":np.quantile(db,.975),
            "Brier_boot_p":min(1.0,2*min((db<=0).mean(),(db>=0).mean())),
            "warmup_phishing_items":str(warm_ph),
            "warmup_genuine_items":str(warm_ge),
        })
    pers_boot=pd.DataFrame(boot_rows)
    save_df(pers_boot,pers_boot_path)
    save_json({
        "warmup_phishing_items":warm_ph,
        "warmup_genuine_items":warm_ge,
        "note":"Retrospective onboarding simulation, not temporal field deployment."
    },DIRS["reports"]/"personalized_onboarding_protocol.json")

    checkpoint("human_personalized",{
        "warmup_items":len(warm_items),
        "prediction_rows":len(test),
        "participants":int(test.participant_id.nunique())
    })

display(pers_summary)
display(pers_boot)

[LOAD] Personalized onboarding CRB


,model,AUROC_mean,AUROC_sd,AUPRC_mean,AUPRC_sd,F1_mean,MCC_mean,BalAcc_mean,Brier_mean,ECE_mean
0,CRB_Base_LR,0.887207,0.021168,0.714450,0.048243,0.690697,0.567570,0.811040,0.138679,0.125006
1,DualAdaptiveCRB_LR,0.908872,0.014933,0.774991,0.034948,0.723610,0.614481,0.832084,0.123178,0.108194
2,PersonalizedCRB_LR,0.903293,0.015047,0.753109,0.032683,0.720414,0.610053,0.830660,0.126963,0.111209


,comparison,AUPRC_diff_mean,AUPRC_CI_low,AUPRC_CI_high,AUPRC_boot_p,Brier_improvement_mean,Brier_CI_low,Brier_CI_high,Brier_boot_p,warmup_phishing_items,warmup_genuine_items
0,PersonalizedCRB_LR - CRB_Base_LR,0.050945,0.009672,0.096240,0.012,0.011760,0.003942,0.019531,0.006,"[2, 5, 8, 9, 19]","[21, 29, 31, 35, 37]"
1,DualAdaptiveCRB_LR - CRB_Base_LR,0.070045,0.027290,0.112769,0.004,0.015114,0.007114,0.023105,0.000,"[2, 5, 8, 9, 19]","[21, 29, 31, 35, 37]"
2,DualAdaptiveCRB_LR - PersonalizedCRB_LR,0.018507,0.001076,0.037272,0.040,0.003279,0.000616,0.006232,0.008,"[2, 5, 8, 9, 19]","[21, 29, 31, 35, 37]"


In [ ]:
# CELL 7D — FINAL ROBUSTNESS:
# Dual-Adaptive CRB across 30 independent warm-up selections
#
# Each selection:
#   5 phishing + 5 genuine warm-up items
#   -> participant profile from warm-up only
#   -> remaining phishing items form prediction set
#   -> grouped CV by participant
#   -> training-only task prior inside every fold
#
# Restart-safe: completed selection IDs are saved after each selection and skipped on rerun.

WARMUP_SELECTIONS = 30
WARM_PH_N = 5
WARM_GE_N = 5
ROBUST_CV_REPEATS = 3
ROBUST_CV_SPLITS = 5
PROTOCOL_TAG = "warm30_v6"

warm_fold_path = DIRS["tables"] / f"dualadaptive_{PROTOCOL_TAG}_fold_metrics.csv"
warm_sel_path = DIRS["tables"] / f"dualadaptive_{PROTOCOL_TAG}_selection_summary.csv"
warm_overall_path = DIRS["tables"] / f"dualadaptive_{PROTOCOL_TAG}_overall_summary.csv"
warm_delta_path = DIRS["tables"] / f"dualadaptive_{PROTOCOL_TAG}_paired_deltas.csv"
warm_protocol_path = DIRS["reports"] / f"dualadaptive_{PROTOCOL_TAG}_protocol.json"

allh=human[
    human.source_dataset.eq("DeceiverAndReceiver_2018") &
    human.perceived_risk.notna() &
    human.decision_confidence.notna() &
    human.correct.notna()
].copy()

ph_items=np.array(sorted(allh.loc[allh.actual_risk.ge(.5),"item_id"].unique()))
ge_items=np.array(sorted(allh.loc[allh.actual_risk.lt(.5),"item_id"].unique()))

expected_rows_per_selection = ROBUST_CV_REPEATS * ROBUST_CV_SPLITS * 3

if artifact_ok(warm_fold_path) and not FORCE:
    warm_fold=pd.read_csv(warm_fold_path)
    complete_ids=set(
        warm_fold.groupby("selection_id").size()
        .loc[lambda s:s>=expected_rows_per_selection].index.astype(int).tolist()
    )
    print(f"[LOAD] partial/completed robustness file; completed selections={len(complete_ids)}/{WARMUP_SELECTIONS}")
else:
    warm_fold=pd.DataFrame()
    complete_ids=set()

def _robust_lr():
    return Pipeline([
        ("imp",SimpleImputer(strategy="median")),
        ("sc",StandardScaler()),
        ("clf",LogisticRegression(max_iter=1200,class_weight="balanced",random_state=SEED))
    ])

def _robust_metrics(y,p):
    pred=(np.asarray(p)>=.5).astype(int)
    return {
        "AUROC":roc_auc_score(y,p),
        "AUPRC":average_precision_score(y,p),
        "F1":f1_score(y,pred,zero_division=0),
        "MCC":matthews_corrcoef(y,pred),
        "BalancedAccuracy":balanced_accuracy_score(y,pred),
        "Brier":brier_score_loss(y,p),
        "ECE":ece_score(y,p,10),
    }

new_rows=[]

for sid in range(WARMUP_SELECTIONS):
    if sid in complete_ids:
        continue

    srng=np.random.default_rng(SEED+20000+sid)
    warm_ph=sorted(srng.choice(ph_items,size=WARM_PH_N,replace=False).tolist())
    warm_ge=sorted(srng.choice(ge_items,size=WARM_GE_N,replace=False).tolist())
    warm_items=set(warm_ph+warm_ge)

    warm=allh[allh.item_id.isin(warm_items)].copy()

    prof=warm.groupby("participant_id").agg(
        warm_cal_error=("calibration_error","mean"),
        warm_confidence=("decision_confidence","mean"),
        warm_overconfidence=("overconfidence","mean"),
        warm_underconfidence=("underconfidence","mean"),
        warm_accuracy=("correct","mean"),
    ).reset_index()

    warm_ph_df=warm[warm.actual_risk.ge(.5)].groupby("participant_id").agg(
        warm_phishing_miss=("incorrect","mean")
    ).reset_index()
    warm_ge_df=warm[warm.actual_risk.lt(.5)].groupby("participant_id").agg(
        warm_genuine_fp=("incorrect","mean")
    ).reset_index()

    prof=prof.merge(warm_ph_df,on="participant_id",how="left").merge(
        warm_ge_df,on="participant_id",how="left"
    )

    test=allh[
        allh.actual_risk.ge(.5) &
        (~allh.item_id.isin(warm_items))
    ].copy().merge(prof,on="participant_id",how="left")

    test["y"]=(1-test.correct).astype(int)
    test["oc_x_conf"]=test.overconfidence*test.decision_confidence

    base_feats=["overconfidence","decision_confidence","oc_x_conf"]
    pers_feats=base_feats+[
        "warm_cal_error","warm_confidence","warm_overconfidence",
        "warm_accuracy","warm_phishing_miss","warm_genuine_fp"
    ]
    dual_feats=pers_feats+["task_failure_prior","oc_x_taskprior"]

    sid_rows=[]

    for rep in range(ROBUST_CV_REPEATS):
        cv=StratifiedGroupKFold(
            n_splits=ROBUST_CV_SPLITS,shuffle=True,
            random_state=SEED+30000+sid*100+rep
        )

        for fold,(tri,tei) in enumerate(cv.split(test,test.y,groups=test.participant_id)):
            tr,te=test.iloc[tri].copy(),test.iloc[tei].copy()

            # task difficulty is estimated ONLY from training participants
            global_prior=float(tr.y.mean())
            stat=tr.groupby("item_id").y.agg(["sum","count"])
            alpha=10.0
            prior=((stat["sum"]+alpha*global_prior)/(stat["count"]+alpha)).to_dict()

            tr["task_failure_prior"]=tr.item_id.map(prior).fillna(global_prior)
            te["task_failure_prior"]=te.item_id.map(prior).fillna(global_prior)
            tr["oc_x_taskprior"]=tr.overconfidence*tr.task_failure_prior
            te["oc_x_taskprior"]=te.overconfidence*te.task_failure_prior

            for model_name,feats in [
                ("CRB_Base_LR",base_feats),
                ("PersonalizedCRB_LR",pers_feats),
                ("DualAdaptiveCRB_LR",dual_feats),
            ]:
                m=_robust_lr()
                m.fit(tr[feats],tr.y)
                p=m.predict_proba(te[feats])[:,1]
                sid_rows.append({
                    "selection_id":sid,
                    "warm_phishing_items":str(warm_ph),
                    "warm_genuine_items":str(warm_ge),
                    "repeat":rep,"fold":fold,
                    "model":model_name,
                    "n_test":len(te),
                    **_robust_metrics(te.y,p)
                })

    sid_df=pd.DataFrame(sid_rows)

    # Remove any stale partial rows for this selection, then append complete selection.
    if len(warm_fold):
        warm_fold=warm_fold[warm_fold.selection_id.astype(int).ne(sid)].copy()
    warm_fold=pd.concat([warm_fold,sid_df],ignore_index=True)

    # atomic-ish restart-safe save after every selection
    save_df(warm_fold,warm_fold_path)
    print(f"[SAVE] warm-up selection {sid+1}/{WARMUP_SELECTIONS}")

# ---- summarize each warm-up selection ----
warm_selection=(
    warm_fold.groupby(["selection_id","model"])
    .agg(
        AUROC_mean=("AUROC","mean"),
        AUPRC_mean=("AUPRC","mean"),
        F1_mean=("F1","mean"),
        MCC_mean=("MCC","mean"),
        BalAcc_mean=("BalancedAccuracy","mean"),
        Brier_mean=("Brier","mean"),
        ECE_mean=("ECE","mean"),
    ).reset_index()
)
save_df(warm_selection,warm_sel_path)

# Overall mean ± SD across WARM-UP SELECTIONS (not folds).
warm_overall=(
    warm_selection.groupby("model")
    .agg(
        AUROC_mean=("AUROC_mean","mean"),AUROC_sd_across_warmups=("AUROC_mean","std"),
        AUPRC_mean=("AUPRC_mean","mean"),AUPRC_sd_across_warmups=("AUPRC_mean","std"),
        F1_mean=("F1_mean","mean"),F1_sd_across_warmups=("F1_mean","std"),
        MCC_mean=("MCC_mean","mean"),MCC_sd_across_warmups=("MCC_mean","std"),
        Brier_mean=("Brier_mean","mean"),Brier_sd_across_warmups=("Brier_mean","std"),
        ECE_mean=("ECE_mean","mean"),ECE_sd_across_warmups=("ECE_mean","std"),
        warmup_selections=("selection_id","nunique")
    ).reset_index()
)
save_df(warm_overall,warm_overall_path)

# ---- paired warm-up-selection deltas + bootstrap CI over selections ----
wide_ap=warm_selection.pivot(index="selection_id",columns="model",values="AUPRC_mean")
wide_br=warm_selection.pivot(index="selection_id",columns="model",values="Brier_mean")

comparisons=[
    ("DualAdaptiveCRB_LR","CRB_Base_LR"),
    ("DualAdaptiveCRB_LR","PersonalizedCRB_LR"),
    ("PersonalizedCRB_LR","CRB_Base_LR"),
]

rng=np.random.default_rng(SEED+40000)
delta_rows=[]

for proposed,baseline in comparisons:
    ids=sorted(set(wide_ap.index) & set(wide_br.index))
    dap=(wide_ap.loc[ids,proposed]-wide_ap.loc[ids,baseline]).dropna()
    dbr=(wide_br.loc[ids,baseline]-wide_br.loc[ids,proposed]).dropna()  # positive = proposed lower Brier

    # bootstrap warm-up SELECTIONS
    B=5000
    boot_ap=[]
    boot_br=[]
    vals_ap=dap.to_numpy()
    vals_br=dbr.to_numpy()
    for _ in range(B):
        ii=rng.integers(0,len(vals_ap),size=len(vals_ap))
        boot_ap.append(vals_ap[ii].mean())
        boot_br.append(vals_br[ii].mean())
    boot_ap=np.asarray(boot_ap); boot_br=np.asarray(boot_br)

    # paired non-parametric test across 30 independent warm-up selections
    try:
        w_ap=stats.wilcoxon(vals_ap,alternative="greater",zero_method="wilcox").pvalue
    except Exception:
        w_ap=np.nan
    try:
        w_br=stats.wilcoxon(vals_br,alternative="greater",zero_method="wilcox").pvalue
    except Exception:
        w_br=np.nan

    delta_rows.append({
        "comparison":f"{proposed} vs {baseline}",
        "n_warmup_selections":len(vals_ap),
        "AUPRC_delta_mean":vals_ap.mean(),
        "AUPRC_delta_sd":vals_ap.std(ddof=1),
        "AUPRC_boot_CI_low":np.quantile(boot_ap,.025),
        "AUPRC_boot_CI_high":np.quantile(boot_ap,.975),
        "AUPRC_wilcoxon_one_sided_p":w_ap,
        "AUPRC_positive_selections":int((vals_ap>0).sum()),
        "Brier_improvement_mean":vals_br.mean(),
        "Brier_improvement_sd":vals_br.std(ddof=1),
        "Brier_boot_CI_low":np.quantile(boot_br,.025),
        "Brier_boot_CI_high":np.quantile(boot_br,.975),
        "Brier_wilcoxon_one_sided_p":w_br,
        "Brier_improved_selections":int((vals_br>0).sum()),
    })

warm_deltas=pd.DataFrame(delta_rows)
if len(warm_deltas):
    warm_deltas["AUPRC_p_holm"]=multipletests(
        warm_deltas.AUPRC_wilcoxon_one_sided_p.fillna(1.0),method="holm"
    )[1]
    warm_deltas["Brier_p_holm"]=multipletests(
        warm_deltas.Brier_wilcoxon_one_sided_p.fillna(1.0),method="holm"
    )[1]

save_df(warm_deltas,warm_delta_path)
save_json({
    "protocol_version":PROTOCOL_TAG,
    "warmup_selections":WARMUP_SELECTIONS,
    "warmup_phishing_items_per_selection":WARM_PH_N,
    "warmup_genuine_items_per_selection":WARM_GE_N,
    "cv_repeats":ROBUST_CV_REPEATS,
    "cv_splits":ROBUST_CV_SPLITS,
    "task_prior":"Empirical-Bayes smoothed prior computed on training participants only",
    "primary_robustness_question":"Does DualAdaptiveCRB retain incremental value across many independent onboarding item selections?"
},warm_protocol_path)

checkpoint("dualadaptive_warm30_v6",{
    "selections":int(warm_selection.selection_id.nunique()),
    "fold_metric_rows":len(warm_fold),
    "models":warm_selection.model.nunique()
})

display(warm_overall.sort_values("AUPRC_mean",ascending=False))
display(warm_deltas)

# Figure: AUPRC across 30 warm-up selections
plt.figure(figsize=(9,5.5))
for model_name in ["CRB_Base_LR","PersonalizedCRB_LR","DualAdaptiveCRB_LR"]:
    q=warm_selection[warm_selection.model.eq(model_name)].sort_values("selection_id")
    plt.plot(q.selection_id+1,q.AUPRC_mean,marker="o",markersize=3,label=model_name)
plt.xlabel("Warm-up selection")
plt.ylabel("Mean AUPRC across grouped CV")
plt.title("Dual-Adaptive CRB Robustness Across 30 Onboarding Selections")
plt.legend()
plt.tight_layout()
plt.savefig(DIRS["figures"]/f"dualadaptive_{PROTOCOL_TAG}_auprc.png",dpi=230,bbox_inches="tight")
plt.savefig(DIRS["figures"]/f"dualadaptive_{PROTOCOL_TAG}_auprc.pdf",bbox_inches="tight")
plt.close()

[LOAD] partial/completed robustness file; completed selections=30/30


,model,AUROC_mean,AUROC_sd_across_warmups,AUPRC_mean,AUPRC_sd_across_warmups,F1_mean,F1_sd_across_warmups,MCC_mean,MCC_sd_across_warmups,Brier_mean,Brier_sd_across_warmups,ECE_mean,ECE_sd_across_warmups,warmup_selections
1,DualAdaptiveCRB_LR,0.911174,0.004402,0.777584,0.019984,0.727087,0.017860,0.621068,0.014691,0.121133,0.003313,0.106430,0.008861,30
2,PersonalizedCRB_LR,0.908280,0.004867,0.765574,0.024297,0.726731,0.019372,0.620852,0.016824,0.123126,0.003627,0.109631,0.009469,30
0,CRB_Base_LR,0.887272,0.004923,0.713797,0.019748,0.692032,0.017284,0.570886,0.012171,0.138890,0.003337,0.126146,0.011238,30


,comparison,n_warmup_selections,AUPRC_delta_mean,AUPRC_delta_sd,AUPRC_boot_CI_low,AUPRC_boot_CI_high,AUPRC_wilcoxon_one_sided_p,AUPRC_positive_selections,Brier_improvement_mean,Brier_improvement_sd,Brier_boot_CI_low,Brier_boot_CI_high,Brier_wilcoxon_one_sided_p,Brier_improved_selections,AUPRC_p_holm,Brier_p_holm
0,DualAdaptiveCRB_LR vs CRB_Base_LR,30,0.063787,0.009443,0.060553,0.067019,9.313226e-10,30,0.017757,0.001935,0.017067,0.018431,9.313226e-10,30,2.793968e-09,2.793968e-09
1,DualAdaptiveCRB_LR vs PersonalizedCRB_LR,30,0.012010,0.005726,0.010022,0.014092,9.313226e-10,30,0.001993,0.001011,0.001645,0.002351,9.313226e-10,30,2.793968e-09,2.793968e-09
2,PersonalizedCRB_LR vs CRB_Base_LR,30,0.051777,0.010599,0.048004,0.055539,9.313226e-10,30,0.015764,0.002031,0.015029,0.016472,9.313226e-10,30,2.793968e-09,2.793968e-09


In [ ]:

# CELL 8 — Inferential statistics + participant-cluster bootstrap model comparisons
# GEE accounts for repeated observations within participant.
# Cluster bootstrap resamples participants, not individual email rows.

gee_path = DIRS["tables"] / "gee_hypothesis_tests.csv"
boot_path = DIRS["tables"] / "cluster_bootstrap_model_comparisons.csv"

if all(artifact_ok(p) for p in [gee_path, boot_path]) and not FORCE:
    print("[LOAD] GEE and cluster-bootstrap statistics")
    gee_table = pd.read_csv(gee_path)
    boot_table = pd.read_csv(boot_path)
else:
    print("[RUN ] GEE + cluster bootstrap")
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

    d = human[
        human["source_dataset"].eq("DeceiverAndReceiver_2018") &
        human["perceived_risk"].notna() &
        human["decision_confidence"].notna() &
        human["correct"].notna()
    ].copy()

    # Z-score helper; keeps coefficients comparable.
    def z(x):
        x = pd.to_numeric(x, errors="coerce")
        return (x-x.mean())/(x.std(ddof=0)+1e-12)

    tests = []

    # H1/H2 on phishing: under-perceived risk (our OC when R=1) and confidence interaction predict misses.
    ph = d[d.actual_risk >= .5].copy()
    ph["missed"] = (1-ph.correct).astype(int)
    ph["OCz"] = z(ph["overconfidence"])
    ph["Qz"] = z(ph["decision_confidence"])
    ph["OCxQ"] = ph["OCz"] * ph["Qz"]

    formula_ph = "missed ~ OCz + Qz + OCxQ"
    if ph["item_id"].nunique() <= 60:
        formula_ph += " + C(item_id)"
    gee_ph = smf.gee(
        formula_ph, groups="participant_id", data=ph,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Exchangeable()
    ).fit()

    for term in ["OCz","Qz","OCxQ"]:
        b = gee_ph.params[term]; se = gee_ph.bse[term]
        lo, hi = b-1.96*se, b+1.96*se
        tests.append({
            "analysis":"Phishing miss (GEE)", "term":term,
            "beta":b, "SE":se, "odds_ratio":np.exp(b),
            "OR_CI_low":np.exp(lo),"OR_CI_high":np.exp(hi),
            "p_value":gee_ph.pvalues[term]
        })

    # Symmetric counterpart on genuine emails: over-perceived risk predicts false positives.
    ge = d[d.actual_risk < .5].copy()
    ge["false_positive"] = (1-ge.correct).astype(int)
    ge["UCz"] = z(ge["underconfidence"])
    ge["Qz"] = z(ge["decision_confidence"])
    ge["UCxQ"] = ge["UCz"] * ge["Qz"]

    formula_ge = "false_positive ~ UCz + Qz + UCxQ"
    if ge["item_id"].nunique() <= 60:
        formula_ge += " + C(item_id)"
    gee_ge = smf.gee(
        formula_ge, groups="participant_id", data=ge,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Exchangeable()
    ).fit()

    for term in ["UCz","Qz","UCxQ"]:
        b = gee_ge.params[term]; se = gee_ge.bse[term]
        lo, hi = b-1.96*se, b+1.96*se
        tests.append({
            "analysis":"Genuine false positive (GEE)", "term":term,
            "beta":b, "SE":se, "odds_ratio":np.exp(b),
            "OR_CI_low":np.exp(lo),"OR_CI_high":np.exp(hi),
            "p_value":gee_ge.pvalues[term]
        })

    gee_table = pd.DataFrame(tests)
    gee_table["p_holm"] = multipletests(gee_table["p_value"], method="holm")[1]
    save_df(gee_table, gee_path)

    # -------- Cluster bootstrap: ProposedCRB_LR vs every other model --------
    # Use repeat 0 so each observation has exactly one OOF prediction per model.
    oo = oof[oof.repeat.eq(0)].copy()
    piv = oo.pivot_table(
        index=["row_index","participant_id","item_id","y_true"],
        columns="model", values="prob", aggfunc="first"
    ).reset_index()
    model_cols = [c for c in piv.columns if c not in ["row_index","participant_id","item_id","y_true"]]
    anchor = "ProposedCRB_LR"
    assert anchor in model_cols

    participants = piv["participant_id"].unique()
    rng = np.random.default_rng(SEED)
    comp_rows = []
    B = int(CONFIG["cluster_bootstrap_B"])

    for comp in [m for m in model_cols if m != anchor]:
        diffs_ap, diffs_brier = [], []
        for b in range(B):
            sample_p = rng.choice(participants, size=len(participants), replace=True)
            parts = []
            # Preserve bootstrap multiplicity by adding a bootstrap copy id.
            for j, pid in enumerate(sample_p):
                zdf = piv[piv.participant_id.eq(pid)].copy()
                zdf["_bootcopy"] = j
                parts.append(zdf)
            bs = pd.concat(parts, ignore_index=True)
            if bs.y_true.nunique() < 2:
                continue
            ap_a = average_precision_score(bs.y_true, bs[anchor])
            ap_c = average_precision_score(bs.y_true, bs[comp])
            br_a = brier_score_loss(bs.y_true, bs[anchor])
            br_c = brier_score_loss(bs.y_true, bs[comp])
            diffs_ap.append(ap_a-ap_c)
            # positive means Proposed is better because lower Brier.
            diffs_brier.append(br_c-br_a)

        da = np.array(diffs_ap); db = np.array(diffs_brier)
        # two-sided bootstrap p around zero
        p_ap = 2*min((da<=0).mean(), (da>=0).mean())
        p_br = 2*min((db<=0).mean(), (db>=0).mean())
        comp_rows.append({
            "anchor":anchor,"competitor":comp,
            "AUPRC_diff_mean":da.mean(),
            "AUPRC_diff_CI_low":np.quantile(da,.025),
            "AUPRC_diff_CI_high":np.quantile(da,.975),
            "AUPRC_boot_p":min(p_ap,1.0),
            "Brier_improvement_mean":db.mean(),
            "Brier_improvement_CI_low":np.quantile(db,.025),
            "Brier_improvement_CI_high":np.quantile(db,.975),
            "Brier_boot_p":min(p_br,1.0),
        })

    boot_table = pd.DataFrame(comp_rows)
    if len(boot_table):
        boot_table["AUPRC_p_holm"] = multipletests(boot_table["AUPRC_boot_p"], method="holm")[1]
        boot_table["Brier_p_holm"] = multipletests(boot_table["Brier_boot_p"], method="holm")[1]
    save_df(boot_table, boot_path)

    # Save full model summaries for traceability.
    (DIRS["reports"]/"gee_phishing_summary.txt").write_text(gee_ph.summary().as_text(), encoding="utf-8")
    (DIRS["reports"]/"gee_genuine_summary.txt").write_text(gee_ge.summary().as_text(), encoding="utf-8")
    checkpoint("human_statistics", {"gee_tests":len(gee_table),"bootstrap_comparisons":len(boot_table),"B":B})

display(gee_table)
display(boot_table)


[LOAD] GEE and cluster-bootstrap statistics


,analysis,term,beta,SE,odds_ratio,OR_CI_low,OR_CI_high,p_value,p_holm
0,Phishing miss (GEE),OCz,1.603777,0.107724,4.971774,4.025457,6.140555,3.950672e-50,2.370403e-49
1,Phishing miss (GEE),Qz,-0.334576,0.075361,0.715641,0.617372,0.829552,9.009796e-06,3.603918e-05
2,Phishing miss (GEE),OCxQ,0.184742,0.135655,1.202908,0.922064,1.569293,1.732431e-01,5.197294e-01
3,Genuine false positive (GEE),UCz,1.415714,0.099897,4.119425,3.386903,5.010377,1.369994e-45,6.849972e-45
4,Genuine false positive (GEE),Qz,-0.059469,0.104861,0.942265,0.767208,1.157264,5.706309e-01,1.000000e+00
5,Genuine false positive (GEE),UCxQ,0.028018,0.105555,1.028415,0.836215,1.264791,7.906723e-01,1.000000e+00


,anchor,competitor,AUPRC_diff_mean,AUPRC_diff_CI_low,AUPRC_diff_CI_high,AUPRC_boot_p,Brier_improvement_mean,Brier_improvement_CI_low,Brier_improvement_CI_high,Brier_boot_p,AUPRC_p_holm,Brier_p_holm
0,ProposedCRB_LR,ConfidenceOnly_LR,0.334921,0.272599,0.401764,0.000,0.095177,0.083631,0.105524,0.000,0.000,0.000
1,ProposedCRB_LR,DraftTerms_LR,0.008388,-0.026372,0.043794,0.644,-0.001184,-0.002653,0.000305,0.110,1.000,0.392
2,ProposedCRB_LR,PerceivedRisk_LR,0.003668,-0.010834,0.019584,0.640,-0.001459,-0.004667,0.001490,0.368,1.000,0.396
3,ProposedCRB_LR,ProposedCRB_HGB,0.029793,-0.005466,0.060862,0.084,-0.007255,-0.017991,0.003664,0.198,0.504,0.396
4,ProposedCRB_LR,ProposedCRB_RF,0.014606,-0.020797,0.046949,0.420,0.005055,0.000143,0.009801,0.046,1.000,0.230
5,ProposedCRB_LR,ProposedCRB_XGB,0.015119,-0.018620,0.047082,0.414,-0.012102,-0.022927,-0.001350,0.028,1.000,0.168
6,ProposedCRB_LR,SymmetricCalibration_LR,0.007947,-0.025869,0.042521,0.698,-0.001205,-0.002558,0.000204,0.098,1.000,0.392


In [ ]:
# CELL 8B — Q1-facing statistical comparison table
#
# Consolidates the strongest defensible comparisons. Positive ΔAUPRC and positive
# Brier improvement favor the proposed model. Do not claim superiority unless the
# confidence interval excludes zero / corrected p-value supports it.

q1_stats_path=DIRS["tables"]/"q1_incremental_value_tests.csv"

if artifact_ok(q1_stats_path) and not FORCE:
    q1_tests=pd.read_csv(q1_stats_path)
    print("[LOAD] Q1 incremental-value tests")
else:
    rows=[]

    def cluster_compare(pred_df, proposed, baseline, label, B=1000):
        oo=pred_df[pred_df.repeat.eq(0)].pivot_table(
            index=["row_index","participant_id","item_id","y_true"],
            columns="model",values="prob",aggfunc="first"
        ).reset_index()
        if proposed not in oo.columns or baseline not in oo.columns:
            return None
        parts=oo.participant_id.unique()
        rng=np.random.default_rng(SEED+777)
        da=[]; db=[]
        for _ in range(B):
            samp=rng.choice(parts,size=len(parts),replace=True)
            bs=pd.concat([oo[oo.participant_id.eq(pid)].assign(_copy=j) for j,pid in enumerate(samp)],ignore_index=True)
            if bs.y_true.nunique()<2:
                continue
            da.append(
                average_precision_score(bs.y_true,bs[proposed]) -
                average_precision_score(bs.y_true,bs[baseline])
            )
            db.append(
                brier_score_loss(bs.y_true,bs[baseline]) -
                brier_score_loss(bs.y_true,bs[proposed])
            )
        da=np.asarray(da); db=np.asarray(db)
        return {
            "comparison":label,
            "proposed":proposed,"baseline":baseline,
            "AUPRC_delta":da.mean(),
            "AUPRC_CI_low":np.quantile(da,.025),"AUPRC_CI_high":np.quantile(da,.975),
            "AUPRC_p":min(1.0,2*min((da<=0).mean(),(da>=0).mean())),
            "Brier_improvement":db.mean(),
            "Brier_CI_low":np.quantile(db,.025),"Brier_CI_high":np.quantile(db,.975),
            "Brier_p":min(1.0,2*min((db<=0).mean(),(db>=0).mean())),
        }

    # Original CRB vs simpler baselines.
    for base in ["ConfidenceOnly_LR","PerceivedRisk_LR","DraftTerms_LR","SymmetricCalibration_LR"]:
        rr=cluster_compare(oof,"ProposedCRB_LR",base,f"General CRB vs {base}")
        if rr: rows.append(rr)

    # Task-aware CRB incremental value.
    for base in ["PerceivedRisk_LR","CRB_LR","TaskPrior_LR"]:
        rr=cluster_compare(task_oof,"CRB_TaskAware_LR",base,f"Task-aware CRB vs {base}")
        if rr: rows.append(rr)

    # Personalized and dual-adaptive CRB.
    for anchor,base,label in [
        ("PersonalizedCRB_LR","CRB_Base_LR","Personalized CRB vs base CRB"),
        ("DualAdaptiveCRB_LR","CRB_Base_LR","Dual-adaptive CRB vs base CRB"),
        ("DualAdaptiveCRB_LR","PersonalizedCRB_LR","Dual-adaptive CRB vs personalized CRB"),
    ]:
        rr=cluster_compare(pers_oof,anchor,base,label)
        if rr: rows.append(rr)

    q1_tests=pd.DataFrame(rows)
    if len(q1_tests):
        q1_tests["AUPRC_p_holm"]=multipletests(q1_tests.AUPRC_p,method="holm")[1]
        q1_tests["Brier_p_holm"]=multipletests(q1_tests.Brier_p,method="holm")[1]
        q1_tests["AUPRC_superior_95CI"]=q1_tests.AUPRC_CI_low>0
        q1_tests["Brier_superior_95CI"]=q1_tests.Brier_CI_low>0
    save_df(q1_tests,q1_stats_path)

display(q1_tests)

[LOAD] Q1 incremental-value tests


,comparison,proposed,baseline,AUPRC_delta,AUPRC_CI_low,AUPRC_CI_high,AUPRC_p,Brier_improvement,Brier_CI_low,Brier_CI_high,Brier_p,AUPRC_p_holm,Brier_p_holm,AUPRC_superior_95CI,Brier_superior_95CI
0,General CRB vs ConfidenceOnly_LR,ProposedCRB_LR,ConfidenceOnly_LR,0.337337,0.274703,0.397617,0.000,0.095616,0.084385,0.105467,0.000,0.000,0.000,True,True
1,General CRB vs PerceivedRisk_LR,ProposedCRB_LR,PerceivedRisk_LR,0.003754,-0.009294,0.019384,0.648,-0.001641,-0.005098,0.001301,0.338,1.000,0.676,False,False
2,General CRB vs DraftTerms_LR,ProposedCRB_LR,DraftTerms_LR,0.008238,-0.020121,0.041421,0.640,-0.001248,-0.002630,0.000104,0.074,1.000,0.296,False,False
3,General CRB vs SymmetricCalibration_LR,ProposedCRB_LR,SymmetricCalibration_LR,0.008238,-0.020121,0.041421,0.640,-0.001248,-0.002630,0.000104,0.074,1.000,0.296,False,False
4,Task-aware CRB vs PerceivedRisk_LR,CRB_TaskAware_LR,PerceivedRisk_LR,0.006346,-0.011983,0.024334,0.504,0.000131,-0.003296,0.003394,0.904,1.000,0.904,False,False
5,Task-aware CRB vs CRB_LR,CRB_TaskAware_LR,CRB_LR,0.015126,0.004089,0.026494,0.008,0.002036,0.000217,0.003929,0.032,0.056,0.160,True,True
6,Task-aware CRB vs TaskPrior_LR,CRB_TaskAware_LR,TaskPrior_LR,0.287714,0.231200,0.338909,0.000,0.080278,0.066735,0.092622,0.000,0.000,0.000,True,True
7,Personalized CRB vs base CRB,PersonalizedCRB_LR,CRB_Base_LR,0.050606,0.010330,0.092026,0.008,0.012008,0.004420,0.019735,0.004,0.056,0.028,True,True
8,Dual-adaptive CRB vs base CRB,DualAdaptiveCRB_LR,CRB_Base_LR,0.068816,0.026809,0.112329,0.002,0.015303,0.007401,0.022898,0.000,0.016,0.000,True,True
9,Dual-adaptive CRB vs personalized CRB,DualAdaptiveCRB_LR,PersonalizedCRB_LR,0.018210,0.000097,0.036696,0.050,0.003296,0.000705,0.006035,0.020,0.250,0.120,True,True


In [ ]:
# CELL 8C — Competitor comparison for the manuscript
#
# IMPORTANT:
# A numeric "winner" claim is valid only for DIRECT SAME-PROTOCOL comparisons.
# Literature values from other datasets/splits are preserved separately as contextual benchmarks.

direct_comp_path = DIRS["tables"] / "direct_head_to_head_comparisons_v6.csv"
lit_comp_path = DIRS["tables"] / "literature_contextual_comparison_v6.csv"
comp_note_path = DIRS["reports"] / "competitor_comparison_policy_v6.txt"

direct_rows=[]

# ---- Human direct same-protocol benchmarks ----
if "bench_summary" in globals():
    for _,r in bench_summary.iterrows():
        direct_rows.append({
            "layer":"Human calibration / phishing miss prediction",
            "protocol":"Repeated participant-grouped CV",
            "model":r["model"],
            "AUROC":r.get("AUROC_mean",np.nan),
            "AUPRC":r.get("AUPRC_mean",np.nan),
            "F1":r.get("F1_mean",np.nan),
            "MCC":r.get("MCC_mean",np.nan),
            "Brier":r.get("Brier_mean",np.nan),
            "comparison_type":"DIRECT: same data + same folds/protocol"
        })

if "task_summary" in globals():
    for _,r in task_summary.iterrows():
        direct_rows.append({
            "layer":"Human calibration / task-aware",
            "protocol":"Unseen-user, task-shared; training-only task priors",
            "model":r["model"],
            "AUROC":r.get("AUROC_mean",np.nan),
            "AUPRC":r.get("AUPRC_mean",np.nan),
            "F1":r.get("F1_mean",np.nan),
            "MCC":r.get("MCC_mean",np.nan),
            "Brier":r.get("Brier_mean",np.nan),
            "comparison_type":"DIRECT: same data + same folds/protocol"
        })

if "warm_overall" in globals():
    for _,r in warm_overall.iterrows():
        direct_rows.append({
            "layer":"Human calibration / robust onboarding",
            "protocol":"30 warm-up selections; participant-grouped CV",
            "model":r["model"],
            "AUROC":r.get("AUROC_mean",np.nan),
            "AUPRC":r.get("AUPRC_mean",np.nan),
            "F1":r.get("F1_mean",np.nan),
            "MCC":r.get("MCC_mean",np.nan),
            "Brier":r.get("Brier_mean",np.nan),
            "comparison_type":"DIRECT: same data + repeated warm-up protocol"
        })

# ---- Technical direct same-protocol baselines ----
if "risk_metrics" in globals():
    for _,r in risk_metrics.iterrows():
        direct_rows.append({
            "layer":"Technical risk / PhiUSIIL numeric",
            "protocol":"Domain-disjoint holdout",
            "model":r["Model"],
            "AUROC":r.get("AUROC",np.nan),
            "AUPRC":r.get("AUPRC",np.nan),
            "F1":r.get("F1",np.nan),
            "MCC":r.get("MCC",np.nan),
            "Brier":r.get("Brier",np.nan),
            "comparison_type":"DIRECT: same data + same holdout"
        })

if "dom_zero" in globals():
    for _,r in dom_zero.iterrows():
        direct_rows.append({
            "layer":"Technical risk / fair external domain-only v6.1",
            "protocol":r["Protocol"],
            "model":r["Model"],
            "AUROC":r.get("AUROC",np.nan),
            "AUPRC":r.get("AUPRC",np.nan),
            "F1":r.get("F1",np.nan),
            "MCC":r.get("MCC",np.nan),
            "Brier":r.get("Brier",np.nan),
            "comparison_type":"DIRECT external zero-shot"
        })

if "dom_calibrated" in globals():
    for _,r in dom_calibrated.iterrows():
        direct_rows.append({
            "layer":"Technical risk / fair external domain-only v6.1",
            "protocol":r["Protocol"],
            "model":r["Model"],
            "AUROC":r.get("AUROC",np.nan),
            "AUPRC":r.get("AUPRC",np.nan),
            "F1":r.get("F1",np.nan),
            "MCC":r.get("MCC",np.nan),
            "Brier":r.get("Brier",np.nan),
            "comparison_type":"DIRECT external recalibrated"
        })

direct_comp=pd.DataFrame(direct_rows)
save_df(direct_comp,direct_comp_path)

# ---- Literature contextual comparison ----
# These values are author-reported and are NOT treated as same-protocol head-to-head results.
lit_cols=[
    "year","study","venue","doi","role","what_to_compare",
    "reported_performance","comparison_status"
]
lit_comp=competitors[[c for c in lit_cols if c in competitors.columns]].copy()
save_df(lit_comp,lit_comp_path)

policy = """COMPETITOR COMPARISON POLICY — CRB v6

1. Direct superiority statements may use ONLY rows in direct_head_to_head_comparisons_v6.csv
   where models were evaluated on the same data and same split/protocol.

2. literature_contextual_comparison_v6.csv contains author-reported or conceptual benchmarks.
   Those rows are useful for positioning the contribution but MUST NOT be described as an
   apples-to-apples numerical win when datasets, tasks, features, or split protocols differ.

3. For the human layer, the strongest direct comparators are:
   - ConfidenceOnly_LR
   - PerceivedRisk_LR
   - SymmetricCalibration_LR
   - DraftTerms_LR
   - CRB_Base_LR
   - PersonalizedCRB_LR
   - TaskPrior_LR
   - Task-aware CRB
   The 30-selection DualAdaptiveCRB protocol is the primary robustness comparison.

4. For the technical layer, LR/RF/HGB/XGB are evaluated under identical PhiUSIIL/domain-only
   protocols. Published PhiUSIIL and X-PHIDE results are contextual because their original
   splits/datasets differ.

5. Do not reproduce a literature metric unless it has been verified from the source paper.
"""
comp_note_path.write_text(policy,encoding="utf-8")

print("[OK] Direct and contextual competitor tables generated.")
display(direct_comp)
display(lit_comp)

[OK] Direct and contextual competitor tables generated.


,layer,protocol,model,AUROC,AUPRC,F1,MCC,Brier,comparison_type
0,Human calibration / phishing miss prediction,Repeated participant-grouped CV,ProposedCRB_LR,0.888158,0.712365,0.691671,0.572462,1.389120e-01,DIRECT: same data + same folds/protocol
1,Human calibration / phishing miss prediction,Repeated participant-grouped CV,PerceivedRisk_LR,0.886417,0.707114,0.694234,0.575289,1.372570e-01,DIRECT: same data + same folds/protocol
2,Human calibration / phishing miss prediction,Repeated participant-grouped CV,DraftTerms_LR,0.886965,0.703908,0.694008,0.574949,1.378192e-01,DIRECT: same data + same folds/protocol
3,Human calibration / phishing miss prediction,Repeated participant-grouped CV,SymmetricCalibration_LR,0.886965,0.703908,0.694008,0.574949,1.378192e-01,DIRECT: same data + same folds/protocol
4,Human calibration / phishing miss prediction,Repeated participant-grouped CV,ProposedCRB_XGB,0.873021,0.695929,0.649215,0.532317,1.257863e-01,DIRECT: same data + same folds/protocol
5,Human calibration / phishing miss prediction,Repeated participant-grouped CV,ProposedCRB_RF,0.867342,0.693118,0.674482,0.546754,1.430599e-01,DIRECT: same data + same folds/protocol
6,Human calibration / phishing miss prediction,Repeated participant-grouped CV,ProposedCRB_HGB,0.860225,0.683296,0.639480,0.517968,1.305252e-01,DIRECT: same data + same folds/protocol
7,Human calibration / phishing miss prediction,Repeated participant-grouped CV,ConfidenceOnly_LR,0.654259,0.362204,0.451699,0.213964,2.343530e-01,DIRECT: same data + same folds/protocol
8,Human calibration / task-aware,"Unseen-user, task-shared; training-only task p...",CRB_TaskAware_LR,0.890781,0.730880,0.693732,0.575419,1.363912e-01,DIRECT: same data + same folds/protocol
9,Human calibration / task-aware,"Unseen-user, task-shared; training-only task p...",CRB_LR,0.888082,0.714094,0.689304,0.569458,1.387101e-01,DIRECT: same data + same folds/protocol


,year,study,venue,doi,role,what_to_compare,reported_performance,comparison_status
12,2021,Chen et al.,Information & Management,10.1016/j.im.2020.103394,Foundational trust-calibration comparator in p...,"Calibration concept, reliance/performance cons...",,Conceptual/contextual; different experiment
13,2024,Gallo et al. / Spamley,Computers & Security,10.1016/j.cose.2023.103671,Real-user phishing behavior benchmark,"Behavioral data collection, user profiling, ph...",,Contextual; dataset available on request
14,2024,Baltuttis & Teubner,Computers & Security,10.1016/j.cose.2024.103940,Human-computer phishing risk indicator experiment,"Risk cues, trustworthiness ratings, response b...",,Contextual; different human experiment
15,2024,Prasad & Chandra / PhiUSIIL,Computers & Security,10.1016/j.cose.2023.103545,Technical phishing-risk benchmark and source d...,PhiUSIIL phishing classification,Accuracy 99.24% fully incremental; 99.79% pre-...,Same source dataset but different training/spl...
16,2025,"Security awareness, decision style, knowledge,...",Computers & Security,10.1016/j.cose.2024.104129,Recent human phishing-behavior comparator,"Awareness, knowledge, decision style, behavior...",,Contextual; different predictors and protocol
17,2025,Bera & Kim,Information & Management,10.1016/j.im.2025.104110,Recent high-reputation phishing-identification...,Information processing and phishing identifica...,,Contextual; different human experiment
18,2026,"Lu, Jiang, Head & Yang",European Journal of Information Systems,10.1080/0960085X.2025.2548543,Recent close behavioral comparator,Phishing accuracy + confidence under multitask...,,Contextual; different human experiment
19,2025,Malatji,Human-Centric Intelligent Systems,10.1007/s44230-025-00103-8,Same-target-journal human-AI cybersecurity com...,Human-AI cybersecurity architecture and feedback,,Architecture/contextual comparison
20,2026,Jamshidi / HINN,Human-Centric Intelligent Systems,10.1007/s44230-026-00173-2,Same-target-journal trust-adaptive human super...,"Dynamic trust weighting, human-informed learni...",Authors report empirical + simulation validati...,Architecture/contextual; different task/data
21,2026,X-PHIDE cross-dataset validation,IEEE Access,10.1109/ACCESS.2026.3672690,Recent cross-dataset phishing detector / gener...,"Cross-dataset accuracy, feature stability, tra...",90.7% accuracy and 91.2% F1 on custom data; 77...,Contextual numeric comparison; different datas...


In [ ]:

# CELL 9 — Manuscript-quality figures for the human calibration experiment
# Restart-safe: figures are regenerated only if absent or FORCE=True.

fig_marker = DIRS["figures"] / "human_figures_done.json"
if artifact_ok(fig_marker) and not FORCE:
    print("[LOAD] Human figures already generated")
else:
    print("[RUN ] Human calibration figures")
    d = human[
        human["source_dataset"].eq("DeceiverAndReceiver_2018") &
        human["perceived_risk"].notna() &
        human["decision_confidence"].notna() &
        human["correct"].notna()
    ].copy()

    # 1. Calibration plane / heatmap-style scatter
    plt.figure(figsize=(6.8,5.4))
    sample = d.sample(min(len(d), 5000), random_state=SEED)
    plt.scatter(sample["perceived_risk"], sample["actual_risk"] + RNG.normal(0,0.015,len(sample)),
                s=10, alpha=.18)
    plt.plot([0,1],[0,1],"--", linewidth=1.2)
    plt.xlabel("User Perceived Risk")
    plt.ylabel("Actual Risk")
    plt.title("Cyber-Risk Calibration Plane")
    plt.tight_layout()
    plt.savefig(DIRS["figures"]/"calibration_plane.png",dpi=240,bbox_inches="tight")
    plt.savefig(DIRS["figures"]/"calibration_plane.pdf",bbox_inches="tight")
    plt.close()

    # 2. Overconfidence vs miss rate by bins on phishing items
    ph = d[d.actual_risk>=.5].copy()
    ph["oc_bin"] = pd.qcut(ph["overconfidence"], q=10, duplicates="drop")
    z = ph.groupby("oc_bin", observed=True).agg(
        overconfidence=("overconfidence","mean"),
        miss_rate=("incorrect","mean"),
        n=("incorrect","size")
    ).reset_index()
    plt.figure(figsize=(6.8,5.2))
    plt.plot(z.overconfidence,z.miss_rate,marker="o")
    plt.xlabel("Mean Risk Under-Perception / Overconfidence")
    plt.ylabel("Observed Phishing Miss Rate")
    plt.title("Hazardous Overconfidence and Unsafe Behavior")
    plt.tight_layout()
    plt.savefig(DIRS["figures"]/"overconfidence_vs_miss_rate.png",dpi=240,bbox_inches="tight")
    plt.savefig(DIRS["figures"]/"overconfidence_vs_miss_rate.pdf",bbox_inches="tight")
    plt.close()

    # 3. ROC and PR for benchmark models (repeat 0)
    oo = oof[oof.repeat.eq(0)]
    plt.figure(figsize=(6.8,5.4))
    for name, g in oo.groupby("model"):
        fpr,tpr,_ = roc_curve(g.y_true,g.prob)
        plt.plot(fpr,tpr,label=f"{name} ({roc_auc_score(g.y_true,g.prob):.3f})")
    plt.plot([0,1],[0,1],"--",linewidth=1)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("Participant-Grouped OOF ROC — Phishing Miss Prediction")
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(DIRS["figures"]/"human_models_roc.png",dpi=240,bbox_inches="tight")
    plt.savefig(DIRS["figures"]/"human_models_roc.pdf",bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(6.8,5.4))
    for name, g in oo.groupby("model"):
        pr,rc,_ = precision_recall_curve(g.y_true,g.prob)
        plt.plot(rc,pr,label=f"{name} ({average_precision_score(g.y_true,g.prob):.3f})")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Participant-Grouped OOF Precision–Recall")
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(DIRS["figures"]/"human_models_pr.png",dpi=240,bbox_inches="tight")
    plt.savefig(DIRS["figures"]/"human_models_pr.pdf",bbox_inches="tight")
    plt.close()

    # 4. Reliability diagram for ProposedCRB LR
    prop = oo[oo.model.eq("ProposedCRB_LR")]
    frac, meanp = calibration_curve(prop.y_true, prop.prob, n_bins=10, strategy="quantile")
    plt.figure(figsize=(6.2,5.1))
    plt.plot(meanp,frac,marker="o")
    plt.plot([0,1],[0,1],"--",linewidth=1)
    plt.xlabel("Predicted Security-Failure Probability")
    plt.ylabel("Observed Failure Frequency")
    plt.title("Proposed CRB Probability Calibration")
    plt.tight_layout()
    plt.savefig(DIRS["figures"]/"proposed_crb_reliability.png",dpi=240,bbox_inches="tight")
    plt.savefig(DIRS["figures"]/"proposed_crb_reliability.pdf",bbox_inches="tight")
    plt.close()

    save_json({"generated_utc":now()}, fig_marker)

print("Figures:", DIRS["figures"])


# 5. Q1 extension comparison: general, task-aware, personalized
try:
    comp_frames=[]
    if "bench_summary" in globals():
        q=bench_summary[bench_summary.model.eq("ProposedCRB_LR")][["model","AUPRC_mean","AUROC_mean"]].copy()
        q["Protocol"]="General CRB"
        comp_frames.append(q)
    if "task_summary" in globals():
        q=task_summary[task_summary.model.eq("CRB_TaskAware_LR")][["model","AUPRC_mean","AUROC_mean"]].copy()
        q["Protocol"]="Task-aware CRB"
        comp_frames.append(q)
    if "pers_summary" in globals():
        q=pers_summary[pers_summary.model.eq("DualAdaptiveCRB_LR")][["model","AUPRC_mean","AUROC_mean"]].copy()
        q["Protocol"]="Dual-adaptive CRB"
        comp_frames.append(q)
    if comp_frames:
        qq=pd.concat(comp_frames,ignore_index=True)
        x=np.arange(len(qq))
        plt.figure(figsize=(7,5))
        plt.bar(x-.18,qq.AUROC_mean,width=.36,label="AUROC")
        plt.bar(x+.18,qq.AUPRC_mean,width=.36,label="AUPRC")
        plt.xticks(x,qq.Protocol,rotation=15,ha="right")
        plt.ylim(0,1)
        plt.ylabel("Score")
        plt.title("CRB Performance Across Deployment Protocols")
        plt.legend()
        plt.tight_layout()
        plt.savefig(DIRS["figures"]/"crb_protocol_comparison.png",dpi=220,bbox_inches="tight")
        plt.savefig(DIRS["figures"]/"crb_protocol_comparison.pdf",bbox_inches="tight")
        plt.close()
except Exception as e:
    print("[WARN] Q1 protocol figure skipped:",e)


[RUN ] Human calibration figures
Figures: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/figures


In [ ]:

# CELL 10 — Adaptive intervention policy stress-test (SIMULATION, not empirical intervention evidence)
# This tests the decision logic proposed for the paper.
# It MUST be reported as simulation unless intervention outcomes are collected from real participants.

policy_path = DIRS["tables"] / "adaptive_intervention_simulation.csv"
policy_fig = DIRS["figures"] / "adaptive_intervention_pareto.png"

if artifact_ok(policy_path) and artifact_ok(policy_fig) and not FORCE:
    print("[LOAD] Intervention simulation")
    policy_results = pd.read_csv(policy_path)
else:
    print("[RUN ] Adaptive intervention simulation")
    # Use proposed model OOF probabilities from repeat 0 as baseline estimated unsafe-action risk.
    pp = oof[(oof.repeat==0) & (oof.model=="ProposedCRB_LR")].copy()

    # Actions and plausible ranges are sensitivity assumptions, not measured treatment effects.
    actions = pd.DataFrame([
        # action, residual-risk multiplier, normalized friction cost
        ("none",          1.00, 0.00),
        ("explain",       0.88, 0.06),
        ("warning",       0.72, 0.12),
        ("step_up_MFA",   0.48, 0.28),
        ("peer_confirm",  0.30, 0.46),
        ("block",         0.05, 0.85),
    ], columns=["action","risk_multiplier","friction"])

    SECURITY_LOSS = 1.0
    friction_weights = np.linspace(0.05,0.80,16)

    rows = []
    for eta in friction_weights:
        selected = []
        residual = []
        friction = []
        for p in pp["prob"].to_numpy():
            costs = SECURITY_LOSS*(p*actions.risk_multiplier.to_numpy()) + eta*actions.friction.to_numpy()
            j = int(np.argmin(costs))
            selected.append(actions.iloc[j].action)
            residual.append(p*actions.iloc[j].risk_multiplier)
            friction.append(actions.iloc[j].friction)
        rows.append({
            "friction_weight_eta":eta,
            "mean_baseline_risk":pp["prob"].mean(),
            "mean_residual_risk":np.mean(residual),
            "relative_risk_reduction":1-np.mean(residual)/max(pp["prob"].mean(),1e-12),
            "mean_friction":np.mean(friction),
            "block_rate":np.mean(np.array(selected)=="block"),
            "mfa_rate":np.mean(np.array(selected)=="step_up_MFA"),
            "peer_confirm_rate":np.mean(np.array(selected)=="peer_confirm"),
            "warning_rate":np.mean(np.array(selected)=="warning"),
        })
    policy_results = pd.DataFrame(rows)
    save_df(policy_results, policy_path)

    plt.figure(figsize=(6.6,5.2))
    plt.plot(policy_results.mean_friction,policy_results.relative_risk_reduction,marker="o")
    for _,r in policy_results.iloc[::3].iterrows():
        plt.annotate(f"η={r.friction_weight_eta:.2f}",(r.mean_friction,r.relative_risk_reduction),fontsize=7)
    plt.xlabel("Mean Intervention Friction (simulated)")
    plt.ylabel("Relative Predicted Risk Reduction (simulated)")
    plt.title("Adaptive Intervention Security–Friction Trade-off")
    plt.tight_layout()
    plt.savefig(policy_fig,dpi=240,bbox_inches="tight")
    plt.savefig(DIRS["figures"]/"adaptive_intervention_pareto.pdf",bbox_inches="tight")
    plt.close()

    checkpoint("adaptive_policy_simulation", {
        "important_note":"Simulation only; do not claim empirical intervention effectiveness."
    })

display(policy_results)


[LOAD] Intervention simulation


,friction_weight_eta,mean_baseline_risk,mean_residual_risk,relative_risk_reduction,mean_friction,block_rate,mfa_rate,peer_confirm_rate,warning_rate
0,0.05,0.392113,0.023889,0.939076,0.696292,0.732550,0.136913,0.057718,0.072819
1,0.10,0.392113,0.031231,0.920353,0.591534,0.636577,0.047651,0.069799,0.041611
2,0.15,0.392113,0.038817,0.901007,0.531611,0.544966,0.065436,0.095973,0.049329
3,0.20,0.392113,0.045819,0.883149,0.491903,0.492953,0.061409,0.104362,0.064094
4,0.25,0.392113,0.054183,0.861817,0.454252,0.445973,0.079530,0.094966,0.076846
5,0.30,0.392113,0.063193,0.838841,0.421302,0.397987,0.092953,0.106376,0.067114
6,0.35,0.392113,0.073911,0.811506,0.388352,0.359396,0.077852,0.107718,0.095973
7,0.40,0.392113,0.088492,0.774320,0.349839,0.287248,0.091611,0.150336,0.090604
8,0.45,0.392113,0.102442,0.738743,0.316923,0.231879,0.095973,0.176510,0.097987
9,0.50,0.392113,0.115860,0.704523,0.288638,0.189262,0.088926,0.196309,0.104698


In [ ]:

# CELL 11 — Prospective end-to-end human study template + manuscript-ready exports
# This is what we should collect later to validate the full agentic loop on the SAME tasks.

template_path = DIRS["tables"] / "prospective_human_experiment_template.csv"
latex_marker = DIRS["reports"] / "latex_tables_done.json"

if not artifact_ok(template_path) or FORCE:
    task_types = [
        "phishing_email", "suspicious_url", "external_AI_upload",
        "MFA_prompt", "cloud_permission_grant", "external_file_share",
        "unknown_attachment", "credential_request", "USB_or_file_execution",
        "security_warning_bypass", "sensitive_clipboard_transfer", "untrusted_cloud_sync"
    ]
    rows = []
    for i,t in enumerate(task_types,1):
        rows.append({
            "participant_id":"",
            "task_id":f"T{i:02d}",
            "task_type":t,
            "condition":"control_or_intervention",
            "actual_risk_expert_0_100":"",
            "agent_risk_probability_0_1":"",
            "user_perceived_risk_0_100":"",
            "user_decision_confidence_0_100":"",
            "action_taken":"",
            "secure_action_0_1":"",
            "intervention_selected":"",
            "intervention_accepted_0_1":"",
            "response_time_ms":"",
            "raw_content_stored_0_1":0,
            "notes":""
        })
    save_df(pd.DataFrame(rows),template_path)

if not artifact_ok(latex_marker) or FORCE:
    # Clean table exports
    def to_tex(df,path,caption,label,floatfmt="%.3f"):
        path = Path(path)
        path.write_text(
            df.to_latex(index=False,escape=True,float_format=lambda x: floatfmt % x if isinstance(x,(float,np.floating)) else str(x),
                        caption=caption,label=label),
            encoding="utf-8"
        )

    to_tex(risk_metrics, DIRS["tables"]/"tab_risk_engine.tex",
           "Performance of the technical cyber-risk engine on the domain-disjoint PhiUSIIL holdout.",
           "tab:risk_engine")
    to_tex(bench_summary, DIRS["tables"]/"tab_human_models.tex",
           "Participant-grouped predictive performance for phishing security failures.",
           "tab:human_models")
    to_tex(gee_table, DIRS["tables"]/"tab_gee.tex",
           "GEE estimates for calibration-related predictors of phishing misses and genuine-email false positives.",
           "tab:gee")
    to_tex(reliability, DIRS["tables"]/"tab_reliability.tex",
           "Internal-consistency diagnostics for human calibration measures.",
           "tab:reliability")
    save_json({"generated_utc":now()},latex_marker)

# Write a manuscript-facing method note.
method_note = f"""
CRB Q1 EXTENSION — EXPERIMENTAL METHOD NOTE
Generated: {now()}

MODULE A — TECHNICAL RISK ESTIMATION
Dataset: PhiUSIIL (UCI id 967; associated Computers & Security 2024).
Split: domain-disjoint GroupShuffleSplit; 20% independent holdout.
Models: Logistic Regression, Random Forest, HistGradientBoosting, XGBoost when available.
Probability calibration: sigmoid calibration on training data only.
Metrics: AUROC, AUPRC, F1, MCC, balanced accuracy, sensitivity, specificity, Brier, log-loss.

MODULE B — HUMAN CYBER-CALIBRATION
Primary dataset: DeceiverAndReceiver (Figshare 7205903).
Direct measures: perceived maliciousness -> perceived cyber risk; task-level decision confidence;
behavioral decision/correctness. Q1-Q20 phishing, Q21-Q40 genuine as documented by the paper.
Corrected constructs:
CE=|R-P|; OC=max(0,R-P); UC=max(0,P-R); HOC=OC*decision_confidence.
Primary endpoint: phishing miss / unsafe response to a phishing message.
Validation: repeated StratifiedGroupKFold grouped by participant.
Inference: binomial GEE with participant clusters; email fixed effects where feasible.
Model-comparison uncertainty: participant-cluster bootstrap with Holm correction.

MODULE B2 — STRONGER GENERALIZATION / PERSONALIZATION PROTOCOLS
Task-aware user-cold evaluation uses a training-only smoothed task-failure prior.
Double-cold-start holds out both users and task identities.
Personalized onboarding uses 5 phishing + 5 genuine warm-up tasks to construct a user profile;
this is a retrospective onboarding simulation, not a temporal field experiment.

MODULE A2 — SHORTCUT / EXTERNAL VALIDATION
PhiUSIIL numeric features are audited for extreme single-feature predictivity.
A URL-only lexical model is trained on PhiUSIIL and externally tested on a current
PhishTank + Tranco snapshot, including a 10% phishing prevalence evaluation.

MODULE C — ADAPTIVE INTERVENTION
The intervention Pareto analysis is a policy simulation only. It supports algorithm design and
sensitivity analysis but must not be described as real intervention effectiveness.

END-TO-END LIMITATION
The technical-risk and human-behavior datasets are independent. Therefore this notebook
provides modular validation, not a real end-to-end field deployment. The prospective template
in tables/prospective_human_experiment_template.csv is the next study needed for full validation.
"""
(DIRS["reports"]/"EXPERIMENT_METHOD_NOTE.txt").write_text(method_note,encoding="utf-8")

print("Prospective experiment template:",template_path)
print("LaTeX tables:",DIRS["tables"])
print("Method note:",DIRS["reports"]/"EXPERIMENT_METHOD_NOTE.txt")


Prospective experiment template: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/tables/prospective_human_experiment_template.csv
LaTeX tables: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/tables
Method note: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/reports/EXPERIMENT_METHOD_NOTE.txt


In [ ]:
# CELL 11B — v6 manuscript-ready exports for final robustness cycle

v6_export_marker=DIRS["reports"]/"v6_exports_done.json"

def _tex(df,path,caption,label):
    Path(path).write_text(
        df.to_latex(
            index=False,escape=True,
            float_format=lambda x: f"{x:.4f}" if isinstance(x,(float,np.floating)) else str(x),
            caption=caption,label=label
        ),
        encoding="utf-8"
    )

_tex(
    dom_zero,
    DIRS["tables"]/"tab_domainonly_zero_shot_v6_1.tex",
    "Zero-shot domain-only external validation on untouched PhishTank--Tranco test domains.",
    "tab:domain_zero"
)
_tex(
    dom_calibrated,
    DIRS["tables"]/"tab_domainonly_recalibrated_v6_1.tex",
    "Recalibrated domain-only external validation using a disjoint calibration-domain partition.",
    "tab:domain_recal"
)
_tex(
    warm_overall,
    DIRS["tables"]/"tab_dualadaptive_warm30_overall_v6.tex",
    "Dual-Adaptive CRB robustness summarized across 30 independent onboarding-task selections.",
    "tab:warm30_overall"
)
_tex(
    warm_deltas,
    DIRS["tables"]/"tab_dualadaptive_warm30_deltas_v6.tex",
    "Paired incremental-value tests across 30 independent onboarding-task selections.",
    "tab:warm30_delta"
)
_tex(
    direct_comp,
    DIRS["tables"]/"tab_direct_competitors_v6.tex",
    "Direct head-to-head comparisons evaluated under matched data and protocols.",
    "tab:direct_comp"
)

save_json({
    "generated_utc":now(),
    "version":"v6.1 final robustness",
    "note":"Literature contextual table intentionally kept separate from direct head-to-head comparisons."
},v6_export_marker)

print("[OK] v6 LaTeX robustness/competitor tables exported.")

[OK] v6 LaTeX robustness/competitor tables exported.


In [ ]:

# CELL 12 — Final integrity check, output inventory, and ZIP package
# Safe to rerun. Creates/updates a compact result package in Google Drive.

zip_path = ROOT / "CRB_Q1_Extension_Results_v6_1.zip"

required = [
    DIRS["tables"]/"phiusiil_risk_metrics.csv",
    DIRS["tables"]/"phiusiil_shortcut_ablation_metrics.csv",
    DIRS["tables"]/"lexical_risk_external_metrics.csv",
    DIRS["tables"]/"domainonly_external_zero_shot_v6_1.csv",
    DIRS["tables"]/"domainonly_external_recalibrated_v6_1.csv",
    DIRS["tables"]/"human_benchmark_summary.csv",
    DIRS["tables"]/"human_taskaware_summary.csv",
    DIRS["tables"]/"human_personalized_summary.csv",
    DIRS["tables"]/"dualadaptive_warm30_v6_overall_summary.csv",
    DIRS["tables"]/"dualadaptive_warm30_v6_paired_deltas.csv",
    DIRS["tables"]/"gee_hypothesis_tests.csv",
    DIRS["tables"]/"cluster_bootstrap_model_comparisons.csv",
    DIRS["tables"]/"q1_incremental_value_tests.csv",
    DIRS["tables"]/"direct_head_to_head_comparisons_v6.csv",
    DIRS["tables"]/"literature_contextual_comparison_v6.csv",
    DIRS["reports"]/"competitor_comparison_policy_v6.txt",
    DIRS["reports"]/"EXPERIMENT_METHOD_NOTE.txt",
    DIRS["reports"]/"v6_exports_done.json",
]
status = pd.DataFrame([{
    "artifact":str(p.relative_to(ROOT)),
    "exists":artifact_ok(p),
    "size_kb":round(p.stat().st_size/1024,2) if p.exists() else 0
} for p in required])
display(status)

if not status["exists"].all():
    print("WARNING: Some core artifacts are missing. Review earlier cells before using results in the paper.")
else:
    print("Core artifacts passed integrity check.")

# Save a readable inventory.
inventory = []
for folder in ["tables","figures","reports","models","processed","checkpoints"]:
    for p in DIRS[folder].rglob("*"):
        if p.is_file():
            inventory.append({
                "folder":folder,
                "file":str(p.relative_to(ROOT)),
                "size_kb":round(p.stat().st_size/1024,2),
                "sha256":sha256_file(p) if p.stat().st_size < 200*1024*1024 else "skipped_large_file"
            })
inv = pd.DataFrame(inventory)
save_df(inv, DIRS["reports"]/"output_inventory.csv")

# Package manuscript-facing outputs only (avoid duplicating very large raw/processed files).
if zip_path.exists() and FORCE:
    zip_path.unlink()
if not zip_path.exists() or FORCE:
    import zipfile
    with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for folder in ["tables","figures","reports","checkpoints"]:
            for p in DIRS[folder].rglob("*"):
                if p.is_file():
                    z.write(p,arcname=str(p.relative_to(ROOT)))
        z.write(ROOT/"config.json",arcname="config.json")
    print("[WRITE]",zip_path)
else:
    print("[LOAD] ZIP already exists:",zip_path)

print("\nDONE.")
print("Project root:",ROOT)
print("Results package:",zip_path)
print("Set FORCE=True in Cell 1 only when you intentionally want to recompute.")


,artifact,exists,size_kb
0,tables/phiusiil_risk_metrics.csv,True,0.69
1,tables/phiusiil_shortcut_ablation_metrics.csv,True,0.51
2,tables/lexical_risk_external_metrics.csv,True,0.76
3,tables/domainonly_external_zero_shot_v6_1.csv,True,1.94
4,tables/domainonly_external_recalibrated_v6_1.csv,True,2.01
5,tables/human_benchmark_summary.csv,True,1.60
6,tables/human_taskaware_summary.csv,True,0.83
7,tables/human_personalized_summary.csv,True,0.66
8,tables/dualadaptive_warm30_v6_overall_summary.csv,True,0.97
9,tables/dualadaptive_warm30_v6_paired_deltas.csv,True,1.19


Core artifacts passed integrity check.
[LOAD] ZIP already exists: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/CRB_Q1_Extension_Results_v6_1.zip

DONE.
Project root: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension
Results package: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/CRB_Q1_Extension_Results_v6_1.zip
Set FORCE=True in Cell 1 only when you intentionally want to recompute.


# CELL 13 — Published-competitor head-to-head benchmark (v7)

This final comparison cycle is intentionally split by **task layer**.

**Human calibration / unsafe-action prediction** is compared only with models trained on the same human dataset and the same participant-grouped protocols (Base CRB, Personalized CRB, Task-Aware CRB, and Dual-Adaptive CRB). Published human–AI studies that use different tasks/outcomes remain contextual comparisons.

**Technical phishing-risk estimation** adds published-method reproductions/adaptations on the same PhiUSIIL records and a common untouched domain-disjoint holdout.

Published methods represented here:

1. **Vajrobol, Gupta & Gaurav (2024), Cyber Security and Applications, DOI 10.1016/j.csa.2024.100044.**  
   Published method: Mutual-Information feature selection + Logistic Regression on PhiUSIIL. The paper explicitly reports a five-feature set: URLSimilarityIndex, LineOfCode, NoOfExternalRef, NoOfImage, and NoOfSelfRef. This notebook evaluates that published five-feature LR configuration on the common domain-disjoint holdout.

2. **Prasad & Chandra (2024), Computers & Security, DOI 10.1016/j.cose.2023.103545.**  
   Published method: PhiUSIIL similarity-index + incremental ensemble using BernoulliNB, PassiveAggressiveClassifier, and SGDClassifier with diverse security profiles. The same three incremental learners and the paper-reported hyperparameters are reproduced here. To preserve a common untouched test set, incremental training is performed only on the training partition; the BestSecurity majority decision is then evaluated on the holdout. The separate live URL-similarity gate is not reconstructed, so this is labelled a protocol-adapted reproduction rather than an exact end-to-end reproduction.

3. **Liu et al. (2026), DOI 10.3233/ATDE250870.**  
   Published study: PhiUSIIL feature compression and comparison of AdaBoost, SGD, ExtraTrees, and GaussianNB; ExtraTrees was the best reported model. This notebook evaluates the same classifier family with training-only feature selection to eight features and the same domain-disjoint holdout. Because the paper's exact intermediate engineered 16-feature pool is not fully available in PhiUSIIL, this is labelled a method adaptation.

4. **Misiek & Hyla / X-PHIDE (2026), IEEE Access, DOI 10.1109/ACCESS.2026.3672690.**  
   Published method: cross-dataset XGBoost using eight stable features and tuned XGBoost hyperparameters. Three of the eight features require DNS/WHOIS-derived information not present in the standard PhiUSIIL table. Therefore the notebook does **not** claim an exact X-PHIDE reproduction. It evaluates the published XGBoost hyperparameter configuration on the same leakage-resistant domain-only feature representation used in CRB v6.1. This is reported separately as a protocol-adapted classifier comparison.

**Interpretation rule:** only rows explicitly marked `same_protocol_direct` may support direct numerical comparison. Rows marked `adapted` must be described as method adaptations, not exact replications of the original publications.

### Interpretation rule
The published-method experiments below are separated by endpoint. URL-detector competitors are compared only inside the technical-risk layer. The human Dual-Adaptive CRB is compared only with human unsafe-action baselines under the same participant-grouped protocol. X-PHIDE is an **adapted implementation** because several of its DNS/WHOIS-derived features are unavailable in standard PhiUSIIL; its published numbers remain contextual only.


In [ ]:
# CELL 13A — Common competitor protocol, exact common holdout, and metric helpers
# Restart-safe. This cell DOES NOT alter any previous v6.1 results.

from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.linear_model import PassiveAggressiveClassifier, SGDClassifier, LogisticRegression
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef,
    balanced_accuracy_score, brier_score_loss, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve
)
from sklearn.model_selection import GroupShuffleSplit
from scipy.special import expit
import numpy as np, pandas as pd, joblib, time, math, json, os, sys, pkgutil, subprocess

FORCE_COMPETITORS = False
COMP_TAG = "published_competitors_v7"

if "ROOT" not in globals() or "DIRS" not in globals():
    raise RuntimeError("Please run CELL 1 first so ROOT/DIRS are initialized.")
if "phi_path" not in globals():
    phi_path = DIRS["raw"] / "phiusiil" / "phiusiil_full.parquet"
if not artifact_ok(phi_path):
    raise FileNotFoundError(f"PhiUSIIL file not found: {phi_path}. Run CELL 3 first.")

comp_metrics_path = DIRS["tables"] / f"{COMP_TAG}_fullfeature_metrics.csv"
comp_pred_path = DIRS["processed"] / f"{COMP_TAG}_fullfeature_predictions.parquet"
comp_features_path = DIRS["tables"] / f"{COMP_TAG}_selected_features.csv"
comp_protocol_path = DIRS["reports"] / f"{COMP_TAG}_protocol.json"

phi_comp = pd.read_parquet(phi_path).copy()

# Identify label exactly as previous cells and convert to 1=phishing.
target_candidates = [c for c in phi_comp.columns if str(c).lower() in {"label","class","target","phishing"}]
if not target_candidates:
    target_candidates = [c for c in phi_comp.columns if "label" in str(c).lower()]
assert target_candidates, "Could not identify PhiUSIIL target label."
target_col_comp = target_candidates[-1]
y_comp = (pd.to_numeric(phi_comp[target_col_comp], errors="coerce") == 0).astype(int)

# Domain grouping EXACTLY follows original CELL 4 so all full-feature competitors share the same holdout.
if "Domain" in phi_comp.columns:
    groups_comp = phi_comp["Domain"].astype(str).fillna("missing")
elif "URL" in phi_comp.columns:
    groups_comp = phi_comp["URL"].astype(str).str.extract(r"^(?:https?://)?([^/]+)", expand=False).fillna("missing")
else:
    groups_comp = pd.Series(np.arange(len(phi_comp)).astype(str), index=phi_comp.index)

drop_cols_comp = {target_col_comp, "FILENAME", "URL", "Domain", "TLD", "Title"}
numeric_cols_comp = [
    c for c in phi_comp.columns
    if c not in drop_cols_comp and pd.api.types.is_numeric_dtype(phi_comp[c])
]
X_comp = phi_comp[numeric_cols_comp].replace([np.inf, -np.inf], np.nan)

gss_comp = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_idx_comp, te_idx_comp = next(gss_comp.split(X_comp, y_comp, groups_comp))
Xtr_comp, Xte_comp = X_comp.iloc[tr_idx_comp].copy(), X_comp.iloc[te_idx_comp].copy()
ytr_comp, yte_comp = y_comp.iloc[tr_idx_comp].copy(), y_comp.iloc[te_idx_comp].copy()
gte_comp = groups_comp.iloc[te_idx_comp].astype(str).copy()

assert set(groups_comp.iloc[tr_idx_comp]).isdisjoint(set(groups_comp.iloc[te_idx_comp]))
print("Common PhiUSIIL competitor protocol")
print("  train rows:", len(tr_idx_comp))
print("  test rows :", len(te_idx_comp))
print("  numeric features:", len(numeric_cols_comp))
print("  train domains:", groups_comp.iloc[tr_idx_comp].nunique())
print("  test domains :", groups_comp.iloc[te_idx_comp].nunique())

def _ece_v7(ytrue, prob, bins=10):
    ytrue=np.asarray(ytrue).astype(int)
    prob=np.asarray(prob,dtype=float)
    edges=np.linspace(0,1,bins+1)
    out=0.0
    for a,b in zip(edges[:-1],edges[1:]):
        m=(prob>=a) & ((prob<b) if b<1 else (prob<=b))
        if m.sum():
            out += m.mean()*abs(ytrue[m].mean()-prob[m].mean())
    return float(out)

def _score_from_estimator(model, X):
    if hasattr(model, "predict_proba"):
        p=model.predict_proba(X)
        if p.ndim==2:
            return np.clip(p[:,1], 1e-8, 1-1e-8)
        return np.clip(p,1e-8,1-1e-8)
    if hasattr(model, "decision_function"):
        z=np.asarray(model.decision_function(X),dtype=float)
        return np.clip(expit(np.clip(z,-35,35)),1e-8,1-1e-8)
    pred=np.asarray(model.predict(X),dtype=float)
    return np.clip(pred,1e-8,1-1e-8)

def _metrics_v7(ytrue, prob, pred=None, threshold=.5):
    ytrue=np.asarray(ytrue).astype(int)
    prob=np.asarray(prob,dtype=float)
    if pred is None:
        pred=(prob>=threshold).astype(int)
    else:
        pred=np.asarray(pred).astype(int)
    tn,fp,fn,tp=confusion_matrix(ytrue,pred,labels=[0,1]).ravel()
    fpr,tpr,_=roc_curve(ytrue,prob)
    def tpr_at(max_fpr):
        ok=np.where(fpr<=max_fpr)[0]
        return float(tpr[ok[-1]]) if len(ok) else 0.0
    return {
        "Accuracy":accuracy_score(ytrue,pred),
        "Precision":precision_score(ytrue,pred,zero_division=0),
        "Recall_Sensitivity":recall_score(ytrue,pred,zero_division=0),
        "Specificity":tn/(tn+fp) if (tn+fp) else np.nan,
        "F1":f1_score(ytrue,pred,zero_division=0),
        "MCC":matthews_corrcoef(ytrue,pred),
        "BalancedAccuracy":balanced_accuracy_score(ytrue,pred),
        "AUROC":roc_auc_score(ytrue,prob),
        "AUPRC":average_precision_score(ytrue,prob),
        "Brier":brier_score_loss(ytrue,prob),
        "ECE":_ece_v7(ytrue,prob,10),
        "TPR_at_FPR_1pct":tpr_at(.01),
        "TPR_at_FPR_0_1pct":tpr_at(.001),
        "TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp),
    }

def _pred_frame_v7(name, prob, pred, reproduction_status, note):
    return pd.DataFrame({
        "row_id":phi_comp.index[te_idx_comp].to_numpy(),
        "domain_group":gte_comp.to_numpy(),
        "y_true":yte_comp.to_numpy(),
        "model":name,
        "prob_phishing":np.asarray(prob),
        "pred_phishing":np.asarray(pred).astype(int),
        "reproduction_status":reproduction_status,
        "note":note,
    })

save_json({
    "tag":COMP_TAG,
    "seed":SEED,
    "positive_class":"phishing=1",
    "split":"GroupShuffleSplit domain-disjoint, 20% untouched holdout",
    "train_rows":len(tr_idx_comp),
    "test_rows":len(te_idx_comp),
    "numeric_features":len(numeric_cols_comp),
    "rule":"No test labels used for feature selection or tuning in Cell 13.",
}, comp_protocol_path)

print("[OK] CELL 13A common protocol ready.")


Common PhiUSIIL competitor protocol
  train rows: 187218
  test rows : 48577
  numeric features: 50
  train domains: 176068
  test domains : 44018
[OK] CELL 13A common protocol ready.


In [ ]:
# CELL 13B — Published PhiUSIIL competitors on the SAME domain-disjoint holdout
#
# Methods:
#  A) Vajrobol et al. 2024 exact published 5-feature MI+LR configuration.
#  B) Prasad & Chandra / PhiUSIIL 2024 incremental ensemble, protocol-adapted.
#  C) Liu et al. 2026 model family (AdaBoost, SGD, ExtraTrees, GNB), protocol-adapted.
#  D) Fixed CRB technical-risk references under the exact same holdout.
#
# Output contains ALL requested metrics, not only AUPRC.

if all(artifact_ok(p) for p in [comp_metrics_path, comp_pred_path, comp_features_path]) and not FORCE_COMPETITORS:
    print("[LOAD] Published competitor full-feature benchmark")
    comp_metrics = pd.read_csv(comp_metrics_path)
    comp_predictions = pd.read_parquet(comp_pred_path)
    comp_selected_features = pd.read_csv(comp_features_path)
else:
    rows=[]
    preds=[]
    feat_records=[]

    # ------------------------------------------------------------
    # A) Vajrobol et al. 2024 — exact published five features + LR
    # ------------------------------------------------------------
    vaj5 = ["URLSimilarityIndex","LineOfCode","NoOfExternalRef","NoOfImage","NoOfSelfRef"]
    missing=[c for c in vaj5 if c not in phi_comp.columns]
    if missing:
        print("[WARN] Vajrobol five-feature reproduction skipped; missing:",missing)
    else:
        print("[RUN ] Vajrobol2024_MI5_LR")
        vaj_model=Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("scale",MinMaxScaler()),
            ("lr",LogisticRegression(max_iter=2000,random_state=SEED))
        ])
        t0=time.time()
        vaj_model.fit(phi_comp.iloc[tr_idx_comp][vaj5], ytr_comp)
        p=_score_from_estimator(vaj_model,phi_comp.iloc[te_idx_comp][vaj5])
        pred=(p>=.5).astype(int)
        met=_metrics_v7(yte_comp,p,pred)
        rows.append({
            "Method":"Vajrobol2024_MI5_LR",
            "Paper":"Vajrobol, Gupta & Gaurav (2024)",
            "DOI":"10.1016/j.csa.2024.100044",
            "Feature_Regime":"Published fixed 5 PhiUSIIL features",
            "Reproduction_Status":"same_protocol_direct",
            "TrainingSeconds":time.time()-t0,
            **met
        })
        preds.append(_pred_frame_v7(
            "Vajrobol2024_MI5_LR",p,pred,"same_protocol_direct",
            "Published five features; common domain-disjoint holdout."
        ))
        feat_records += [{"Method":"Vajrobol2024_MI5_LR","Feature":c,"Selection":"published_exact"} for c in vaj5]

    # ------------------------------------------------------------
    # B) PhiUSIIL 2024 incremental ensemble
    # Paper: feature-importance prefilter + MI top 85%, then
    # BernoulliNB, PassiveAggressive, SGDClassifier.
    # We reproduce feature selection on TRAINING DATA ONLY.
    # ------------------------------------------------------------
    print("[RUN ] PhiUSIIL2024 incremental ensemble adaptation")
    imp=SimpleImputer(strategy="median")
    Xtr_imp=imp.fit_transform(Xtr_comp)
    Xte_imp=imp.transform(Xte_comp)

    # Ensemble feature-importance prefilter as described in the paper.
    importance_vectors=[]
    fi_models=[
        RandomForestClassifier(n_estimators=160,n_jobs=-1,random_state=SEED),
        DecisionTreeClassifier(random_state=SEED),
    ]
    if HAVE_XGB:
        fi_models.append(XGBClassifier(
            n_estimators=160,max_depth=5,learning_rate=.08,
            subsample=.85,colsample_bytree=.85,eval_metric="logloss",
            random_state=SEED,n_jobs=-1
        ))
    # LightGBM is included when available, as used in the source paper.
    try:
        if pkgutil.find_loader("lightgbm") is None:
            subprocess.check_call([sys.executable,"-m","pip","install","-q","lightgbm"])
        from lightgbm import LGBMClassifier
        fi_models.append(LGBMClassifier(
            n_estimators=160,random_state=SEED,verbosity=-1,n_jobs=-1
        ))
        have_lgbm=True
    except Exception as e:
        print("[WARN] LightGBM unavailable; continuing with RF/DT/XGB importance ensemble:",e)
        have_lgbm=False

    for fm in fi_models:
        fm.fit(Xtr_imp,ytr_comp)
        v=np.asarray(fm.feature_importances_,dtype=float)
        if v.sum()>0: v=v/v.sum()
        importance_vectors.append(v)

    mean_imp=np.mean(np.vstack(importance_vectors),axis=0)
    positive_idx=np.where(mean_imp>0)[0]
    if len(positive_idx)<5:
        positive_idx=np.arange(Xtr_comp.shape[1])

    # MI top 85% among positive-importance features, fitted on training only.
    mi=mutual_info_classif(
        Xtr_imp[:,positive_idx],ytr_comp,
        discrete_features="auto",random_state=SEED
    )
    nkeep=max(5,int(math.ceil(.85*len(positive_idx))))
    local_order=np.argsort(mi)[::-1][:nkeep]
    sel_idx=positive_idx[local_order]
    phi_il_features=[numeric_cols_comp[i] for i in sel_idx]

    # Refit imputation/scaling only on the selected training columns.
    # This avoids any accidental test-side preprocessing and is robust to missing values.
    il_imp=SimpleImputer(strategy="median")
    il_scaler=MinMaxScaler()
    Xtr_il=il_scaler.fit_transform(il_imp.fit_transform(Xtr_comp[phi_il_features]))
    Xte_il=il_scaler.transform(il_imp.transform(Xte_comp[phi_il_features]))

    bnb=BernoulliNB(alpha=.01,binarize=0.0,class_prior=None,fit_prior=False)
    pa=PassiveAggressiveClassifier(
        C=5,fit_intercept=False,n_iter_no_change=50,n_jobs=-1,
        shuffle=True,random_state=SEED,max_iter=1000
    )
    sgd=SGDClassifier(
        alpha=.01,eta0=100,n_iter_no_change=50,n_jobs=1,
        random_state=SEED,max_iter=1000
    )

    # Incremental one-pass training, chunked, training partition only.
    order=np.random.default_rng(SEED+1302).permutation(len(ytr_comp))
    classes=np.array([0,1],dtype=int)
    chunk=5000
    t0=time.time()
    for start in range(0,len(order),chunk):
        ii=order[start:start+chunk]
        for model in (bnb,pa,sgd):
            model.partial_fit(Xtr_il[ii],ytr_comp.iloc[ii].to_numpy(),classes=classes)

    pred_b=bnb.predict(Xte_il).astype(int)
    pred_p=pa.predict(Xte_il).astype(int)
    pred_s=sgd.predict(Xte_il).astype(int)
    majority=((pred_b+pred_p+pred_s)>=2).astype(int)

    # Continuous score only for common rank/calibration metrics.
    pb=_score_from_estimator(bnb,Xte_il)
    pp=_score_from_estimator(pa,Xte_il)
    ps=_score_from_estimator(sgd,Xte_il)
    p_ens=np.clip((pb+pp+ps)/3,1e-8,1-1e-8)

    met=_metrics_v7(yte_comp,p_ens,majority)
    rows.append({
        "Method":"PhiUSIIL2024_Incremental_BestSecurity",
        "Paper":"Prasad & Chandra (2024)",
        "DOI":"10.1016/j.cose.2023.103545",
        "Feature_Regime":f"Importance prefilter + training-only MI top85% ({len(phi_il_features)} features)",
        "Reproduction_Status":"adapted",
        "TrainingSeconds":time.time()-t0,
        **met
    })
    preds.append(_pred_frame_v7(
        "PhiUSIIL2024_Incremental_BestSecurity",p_ens,majority,"adapted",
        "Three published incremental learners + majority BestSecurity; live USI gate omitted."
    ))
    feat_records += [{"Method":"PhiUSIIL2024_Incremental_BestSecurity","Feature":c,"Selection":"training_only_importance_plus_MI85"} for c in phi_il_features]

    # ------------------------------------------------------------
    # C) Liu et al. 2026 — same four classifier families, 8 selected features
    # Selection is fitted only on training data. Exact original intermediate
    # engineered 16-feature pool is unavailable, so this is method-adapted.
    # ------------------------------------------------------------
    print("[RUN ] Liu2026 classifier-family adaptation")
    selector=ExtraTreesClassifier(
        n_estimators=240,class_weight="balanced",n_jobs=-1,random_state=42
    )
    selector.fit(Xtr_imp,ytr_comp)
    top8_idx=np.argsort(selector.feature_importances_)[::-1][:8]
    liu8=[numeric_cols_comp[i] for i in top8_idx]
    Xtr8=Xtr_comp[liu8].copy()
    Xte8=Xte_comp[liu8].copy()

    liu_models={
        "Liu2026_AdaBoost_adapted":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",AdaBoostClassifier(n_estimators=100,random_state=42))
        ]),
        "Liu2026_SGD_adapted":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("sc",StandardScaler()),
            ("clf",SGDClassifier(loss="log_loss",max_iter=1500,random_state=42))
        ]),
        "Liu2026_ExtraTrees_adapted":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",ExtraTreesClassifier(
                n_estimators=350,class_weight="balanced",n_jobs=-1,random_state=42
            ))
        ]),
        "Liu2026_GNB_adapted":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",GaussianNB())
        ]),
    }
    for name,m in liu_models.items():
        print(" fitting",name)
        t0=time.time()
        m.fit(Xtr8,ytr_comp)
        p=_score_from_estimator(m,Xte8)
        pred=(p>=.5).astype(int)
        rows.append({
            "Method":name,
            "Paper":"Liu et al. (2026)",
            "DOI":"10.3233/ATDE250870",
            "Feature_Regime":"Training-only top-8 importance features",
            "Reproduction_Status":"adapted",
            "TrainingSeconds":time.time()-t0,
            **_metrics_v7(yte_comp,p,pred)
        })
        preds.append(_pred_frame_v7(
            name,p,pred,"adapted",
            "Published classifier family; top-8 feature selection reproduced on common training partition."
        ))
    feat_records += [{"Method":"Liu2026_family","Feature":c,"Selection":"training_only_top8_extratrees"} for c in liu8]

    # ------------------------------------------------------------
    # D) Fixed technical-risk reference configurations from CRB notebook
    # No holdout-driven winner selection here: every fixed model is reported.
    # ------------------------------------------------------------
    print("[RUN ] Fixed CRB technical-risk references")
    ref_models={
        "CRBTech_Logistic_fixed":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("sc",StandardScaler()),
            ("clf",LogisticRegression(max_iter=1500,class_weight="balanced",random_state=SEED))
        ]),
        "CRBTech_RF_fixed":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",RandomForestClassifier(
                n_estimators=350,min_samples_leaf=2,
                class_weight="balanced_subsample",n_jobs=-1,random_state=SEED
            ))
        ]),
        "CRBTech_HGB_fixed":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",HistGradientBoostingClassifier(
                max_iter=250,learning_rate=.06,random_state=SEED
            ))
        ]),
    }
    if HAVE_XGB:
        ref_models["CRBTech_XGB_fixed"]=Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",XGBClassifier(
                n_estimators=350,max_depth=6,learning_rate=.06,
                subsample=.85,colsample_bytree=.85,
                eval_metric="logloss",random_state=SEED,n_jobs=-1
            ))
        ])

    for name,m in ref_models.items():
        print(" fitting",name)
        t0=time.time()
        m.fit(Xtr_comp,ytr_comp)
        p=_score_from_estimator(m,Xte_comp)
        pred=(p>=.5).astype(int)
        rows.append({
            "Method":name,
            "Paper":"CRB technical-risk reference",
            "DOI":"",
            "Feature_Regime":"All available numeric PhiUSIIL features",
            "Reproduction_Status":"same_protocol_reference",
            "TrainingSeconds":time.time()-t0,
            **_metrics_v7(yte_comp,p,pred)
        })
        preds.append(_pred_frame_v7(
            name,p,pred,"same_protocol_reference",
            "Fixed a-priori CRB technical-risk configuration; no test-driven model selection."
        ))

    comp_metrics=pd.DataFrame(rows).sort_values(
        ["AUPRC","Brier"],ascending=[False,True]
    ).reset_index(drop=True)
    comp_predictions=pd.concat(preds,ignore_index=True)
    comp_selected_features=pd.DataFrame(feat_records)

    save_df(comp_metrics,comp_metrics_path)
    save_df(comp_predictions,comp_pred_path)
    save_df(comp_selected_features,comp_features_path)

display(comp_metrics[[
    "Method","Paper","Reproduction_Status","Accuracy","Precision","Recall_Sensitivity",
    "Specificity","F1","MCC","BalancedAccuracy","AUROC","AUPRC","Brier","ECE"
]])
print("[OK] CELL 13B published PhiUSIIL competitor benchmark complete.")


[LOAD] Published competitor full-feature benchmark


,Method,Paper,Reproduction_Status,Accuracy,Precision,Recall_Sensitivity,Specificity,F1,MCC,BalancedAccuracy,AUROC,AUPRC,Brier,ECE
0,CRBTech_RF_fixed,CRB technical-risk reference,same_protocol_reference,0.999979,1.000000,0.999954,1.000000,0.999977,0.999958,0.999977,1.000000,1.000000,6.091256e-05,0.000946
1,CRBTech_HGB_fixed,CRB technical-risk reference,same_protocol_reference,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.352218e-09,0.000031
2,CRBTech_XGB_fixed,CRB technical-risk reference,same_protocol_reference,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.165820e-06,0.000017
3,CRBTech_Logistic_fixed,CRB technical-risk reference,same_protocol_reference,0.999918,1.000000,0.999816,1.000000,0.999908,0.999833,0.999908,1.000000,1.000000,6.584402e-05,0.000252
4,Liu2026_ExtraTrees_adapted,Liu et al. (2026),adapted,0.999897,1.000000,0.999770,1.000000,0.999885,0.999792,0.999885,0.999999,0.999999,1.081390e-04,0.000124
5,Liu2026_AdaBoost_adapted,Liu et al. (2026),adapted,0.999897,1.000000,0.999770,1.000000,0.999885,0.999792,0.999885,0.999999,0.999998,6.107290e-02,0.236288
6,Vajrobol2024_MI5_LR,"Vajrobol, Gupta & Gaurav (2024)",same_protocol_direct,0.993721,1.000000,0.985958,1.000000,0.992930,0.987363,0.992979,0.999998,0.999998,5.182376e-03,0.006503
7,Liu2026_GNB_adapted,Liu et al. (2026),adapted,0.999897,1.000000,0.999770,1.000000,0.999885,0.999792,0.999885,0.999999,0.999998,1.026612e-04,0.000099
8,Liu2026_SGD_adapted,Liu et al. (2026),adapted,0.999527,1.000000,0.998941,1.000000,0.999470,0.999043,0.999471,0.999997,0.999996,4.645885e-04,0.001943
9,PhiUSIIL2024_Incremental_BestSecurity,Prasad & Chandra (2024),adapted,0.998703,0.999447,0.997652,0.999553,0.998548,0.997378,0.998603,0.999993,0.999992,8.010922e-03,0.068383


[OK] CELL 13B published PhiUSIIL competitor benchmark complete.


In [ ]:
# CELL 13C — X-PHIDE 2026 configuration on the common FAIR DOMAIN-ONLY protocol
#
# This is deliberately labelled a METHOD ADAPTATION, not an exact X-PHIDE reproduction.
# The published X-PHIDE pipeline uses eight features, including DNS/WHOIS-derived
# nameserver, SPF, and domain-age variables that are not contained in standard PhiUSIIL.
#
# What is reproduced here:
#   - the reported XGBoost configuration;
#   - the same source dataset;
#   - the same v6.1 fair hostname/domain-only representation;
#   - the exact same domain-disjoint split used by v6.1;
#   - the same full metric panel for every model.
#
# For a fair table, the v6.1 Domain Logistic/RF/HGB/XGB references are re-fitted here
# on that exact split so Accuracy/Precision/Recall/Specificity are available too.

xphide_metrics_path=DIRS["tables"]/"xphide2026_domainonly_adapted_v7.csv"
xphide_pred_path=DIRS["processed"]/"xphide2026_domainonly_adapted_predictions_v7.parquet"

if artifact_ok(xphide_metrics_path) and artifact_ok(xphide_pred_path) and not FORCE_COMPETITORS:
    print("[LOAD] X-PHIDE adapted domain-only comparison")
    xphide_metrics=pd.read_csv(xphide_metrics_path)
    xphide_predictions=pd.read_parquet(xphide_pred_path)
else:
    if "phi_dom" not in globals() or "dom_features" not in globals():
        dom_cache=DIRS["processed"]/"phiusiil_domainonly_features_v6.parquet"
        if not artifact_ok(dom_cache):
            raise RuntimeError("Run CELL 4C first so the fair domain-only feature cache exists.")
        phi_dom=pd.read_parquet(dom_cache)
        dom_features=[c for c in phi_dom.columns if c not in {"url","label","registrable_domain"}]

    phi_dom2=phi_dom.dropna(subset=["registrable_domain","label"]).copy()
    phi_dom2["registrable_domain"]=(
        phi_dom2["registrable_domain"].astype(str).str.lower().str.strip()
    )
    phi_dom2=phi_dom2[phi_dom2.registrable_domain.ne("")].copy()

    y_dom=pd.to_numeric(phi_dom2.label,errors="coerce").astype(int)
    g_dom=phi_dom2.registrable_domain.astype(str)
    X_dom=phi_dom2[dom_features].replace([np.inf,-np.inf],np.nan)

    # EXACT same internal domain split as v6.1 Cell 4C.
    gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED+606)
    tri,tei=next(gss.split(X_dom,y_dom,g_dom))
    Xtrd,Xted=X_dom.iloc[tri].copy(),X_dom.iloc[tei].copy()
    ytrd,yted=y_dom.iloc[tri].copy(),y_dom.iloc[tei].copy()
    gtrd,gted=g_dom.iloc[tri].copy(),g_dom.iloc[tei].copy()
    assert set(gtrd).isdisjoint(set(gted)), "Domain leakage in Cell 13C split."

    # The paper describes dynamic class weighting as 2x imbalance ratio.
    neg=(ytrd==0).sum()
    pos=(ytrd==1).sum()
    spw=2.0*(neg/max(pos,1))

    domain_models={
        "Domain_Logistic":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("sc",StandardScaler()),
            ("clf",LogisticRegression(max_iter=1500,class_weight="balanced",random_state=SEED))
        ]),
        "Domain_RF":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",RandomForestClassifier(
                n_estimators=450,min_samples_leaf=2,class_weight="balanced_subsample",
                n_jobs=-1,random_state=SEED
            ))
        ]),
        "Domain_HGB":Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",HistGradientBoostingClassifier(
                max_iter=300,learning_rate=.05,random_state=SEED
            ))
        ]),
    }
    if HAVE_XGB:
        domain_models["Domain_XGB"]=Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",XGBClassifier(
                n_estimators=450,max_depth=5,learning_rate=.05,
                subsample=.85,colsample_bytree=.85,eval_metric="logloss",
                random_state=SEED,n_jobs=-1
            ))
        ])
        domain_models["XPHIDE2026_XGB_config_domainonly"]=Pipeline([
            ("imp",SimpleImputer(strategy="median")),
            ("clf",XGBClassifier(
                n_estimators=300,max_depth=8,learning_rate=.05,
                gamma=.1,reg_alpha=.1,reg_lambda=.1,
                subsample=.9,colsample_bytree=.9,
                scale_pos_weight=spw,
                eval_metric="logloss",random_state=SEED,n_jobs=-1
            ))
        ])
    else:
        raise RuntimeError("XGBoost is required for the X-PHIDE configuration comparison.")

    rows=[]
    pred_frames=[]
    for name,m in domain_models.items():
        print(" fitting",name)
        t0=time.time()
        m.fit(Xtrd,ytrd)
        p=_score_from_estimator(m,Xted)
        pred=(p>=.5).astype(int)
        is_x=name.startswith("XPHIDE")
        rows.append({
            "Method":name,
            "Paper":"Misiek & Hyla (2026)" if is_x else "CRB v6.1 domain-only reference",
            "DOI":"10.1109/ACCESS.2026.3672690" if is_x else "",
            "Protocol":"PhiUSIIL domain-only domain-disjoint",
            "Reproduction_Status":"adapted" if is_x else "same_protocol_reference",
            "Feature_Regime":(
                "Common CRB v6.1 domain-only features; published X-PHIDE XGBoost configuration; "
                "original DNS/WHOIS feature set unavailable"
                if is_x else "Common CRB v6.1 domain-only features"
            ),
            "TrainingSeconds":time.time()-t0,
            **_metrics_v7(yted,p,pred)
        })
        pred_frames.append(pd.DataFrame({
            "row_id":phi_dom2.index[tei].to_numpy(),
            "domain_group":gted.to_numpy(),
            "y_true":yted.to_numpy(),
            "model":name,
            "prob_phishing":p,
            "pred_phishing":pred,
        }))

    xphide_metrics=pd.DataFrame(rows).sort_values(
        ["AUPRC","Brier"],ascending=[False,True]
    ).reset_index(drop=True)
    xphide_predictions=pd.concat(pred_frames,ignore_index=True)
    save_df(xphide_metrics,xphide_metrics_path)
    save_df(xphide_predictions,xphide_pred_path)

# Sanity check: v7 re-fitted reference metrics should closely reproduce v6.1 internal rows.
dom_int_file=DIRS["tables"]/"domainonly_internal_metrics_v6_1.csv"
if artifact_ok(dom_int_file):
    old_dom=pd.read_csv(dom_int_file)
    chk=xphide_metrics[xphide_metrics.Reproduction_Status.eq("same_protocol_reference")][
        ["Method","AUROC","AUPRC"]
    ].merge(
        old_dom[["Model","AUROC","AUPRC"]],left_on="Method",right_on="Model",
        suffixes=("_v7","_v6_1"),how="inner"
    )
    if len(chk):
        chk["AUROC_abs_diff"]=(chk.AUROC_v7-chk.AUROC_v6_1).abs()
        chk["AUPRC_abs_diff"]=(chk.AUPRC_v7-chk.AUPRC_v6_1).abs()
        display(chk)
        if (chk[["AUROC_abs_diff","AUPRC_abs_diff"]].to_numpy()>1e-8).any():
            print("[WARN] v7 domain references differ from cached v6.1 values; inspect package versions/data cache.")

# Published X-PHIDE values are kept as context only; they are not same-protocol scores.
published_xphide_context=pd.DataFrame([{
    "Paper":"Misiek & Hyla (2026)",
    "DOI":"10.1109/ACCESS.2026.3672690",
    "Published_Protocol":"Authors' custom dataset",
    "Published_Accuracy":0.907,
    "Published_F1":0.912,
    "Published_Average_CrossDataset_Accuracy":0.778,
    "Use":"context_only_not_direct"
}])
save_df(published_xphide_context,DIRS["tables"]/"xphide2026_published_context_v7.csv")

display(xphide_metrics[[
    "Method","Reproduction_Status","Accuracy","Precision","Recall_Sensitivity",
    "Specificity","F1","MCC","BalancedAccuracy","AUROC","AUPRC","Brier","ECE",
    "TPR_at_FPR_1pct","TPR_at_FPR_0_1pct"
]])
print("[OK] CELL 13C X-PHIDE adapted common-protocol comparison complete.")


[LOAD] X-PHIDE adapted domain-only comparison


,Method,AUROC_v7,AUPRC_v7,Model,AUROC_v6_1,AUPRC_v6_1,AUROC_abs_diff,AUPRC_abs_diff
0,Domain_XGB,0.876686,0.876914,Domain_XGB,0.876686,0.876914,0.000000e+00,0.0
1,Domain_HGB,0.874061,0.872934,Domain_HGB,0.874061,0.872934,0.000000e+00,0.0
2,Domain_RF,0.869769,0.872047,Domain_RF,0.869769,0.872047,0.000000e+00,0.0
3,Domain_Logistic,0.826369,0.833169,Domain_Logistic,0.826369,0.833169,1.110223e-16,0.0


,Method,Reproduction_Status,Accuracy,Precision,Recall_Sensitivity,Specificity,F1,MCC,BalancedAccuracy,AUROC,AUPRC,Brier,ECE,TPR_at_FPR_1pct,TPR_at_FPR_0_1pct
0,Domain_XGB,same_protocol_reference,0.845875,0.932869,0.645986,0.970919,0.763364,0.678907,0.808453,0.876686,0.876914,0.117254,0.021137,0.566840,0.496037
1,XPHIDE2026_XGB_config_domainonly,adapted,0.840302,0.818371,0.751892,0.895608,0.783724,0.658982,0.823750,0.876085,0.876770,0.133425,0.116545,0.579057,0.500864
2,Domain_HGB,same_protocol_reference,0.846219,0.936633,0.643960,0.972746,0.763200,0.680242,0.808353,0.874061,0.872934,0.118546,0.019886,0.562727,0.495977
3,Domain_RF,same_protocol_reference,0.848925,0.912698,0.671673,0.959809,0.773852,0.681939,0.815741,0.869769,0.872047,0.117188,0.028851,0.571786,0.458073
4,Domain_Logistic,same_protocol_reference,0.834522,0.919400,0.624769,0.965737,0.743976,0.653795,0.795253,0.826369,0.833169,0.141224,0.086457,0.414327,0.282913


[OK] CELL 13C X-PHIDE adapted common-protocol comparison complete.


In [ ]:

# Produces:
# 1) human CRB metrics across the robust 30-onboarding experiment,
# 2) technical published-competitor benchmark,
# 3) separate X-PHIDE protocol-adapted comparison,
# 4) one manuscript-facing summary with explicit comparison status.

human_all_path=DIRS["tables"]/"human_dualadaptive_all_metrics_v7.csv"
master_path=DIRS["tables"]/"complete_comparison_all_metrics_v7.csv"
master_tex=DIRS["tables"]/"tab_complete_comparison_all_metrics_v7.tex"
summary_note=DIRS["reports"]/"FINAL_COMPARISON_INTERPRETATION_v7.txt"

# Human metrics: these are already robust across 30 onboarding selections.
warm_file=DIRS["tables"]/"dualadaptive_warm30_v6_overall_summary.csv"
if artifact_ok(warm_file):
    human_all=pd.read_csv(warm_file)
elif "warm_overall" in globals():
    human_all=warm_overall.copy()
else:
    # fallback discover likely file name from current globals / folder
    candidates=list(DIRS["tables"].glob("*warm30*overall*.csv"))
    if not candidates:
        raise FileNotFoundError("Could not find 30-selection human robustness summary. Run CELL 7D.")
    human_all=pd.read_csv(candidates[0])

save_df(human_all,human_all_path)

# Verified author-reported values are kept in a SEPARATE contextual table.
# They are never mixed with the same-protocol rerun table because the original papers
# used different splits, feature availability, and sometimes different datasets.
published_context_path=DIRS["tables"]/"published_reported_metrics_context_v7.csv"
published_context=pd.DataFrame([
    {
        "Study":"Vajrobol, Gupta & Gaurav (2024)",
        "DOI":"10.1016/j.csa.2024.100044",
        "Reported_Setting":"Authors' original PhiUSIIL experiment",
        "Accuracy":0.9997,"AUROC":np.nan,"F1":np.nan,
        "Reported_Note":"Authors report 99.97% accuracy for MI + Logistic Regression.",
        "Comparison_Status":"context_only_original_protocol"
    },
    {
        "Study":"Prasad & Chandra / PhiUSIIL (2024)",
        "DOI":"10.1016/j.cose.2023.103545",
        "Reported_Setting":"Fully incremental / BestSecurity framework",
        "Accuracy":0.9924,"AUROC":np.nan,"F1":0.99338,
        "Reported_Note":"Paper reports 99.24% fully incremental accuracy; BestSecurity F1=0.99338.",
        "Comparison_Status":"context_only_original_protocol"
    },
    {
        "Study":"Prasad & Chandra / PhiUSIIL (2024)",
        "DOI":"10.1016/j.cose.2023.103545",
        "Reported_Setting":"90% pre-training / 10% incremental",
        "Accuracy":0.99792,"AUROC":np.nan,"F1":0.99817,
        "Reported_Note":"Highest pre-training row reported in the source paper.",
        "Comparison_Status":"context_only_original_protocol"
    },
    {
        "Study":"Liu et al. (2026)",
        "DOI":"10.3233/ATDE250870",
        "Reported_Setting":"Authors' original PhiUSIIL ExtraTrees test",
        "Accuracy":0.9574,"AUROC":0.99,"F1":0.96,
        "Reported_Note":"Best reported classifier in the paper; class-wise F1 differs slightly.",
        "Comparison_Status":"context_only_original_protocol"
    },
    {
        "Study":"Misiek & Hyla / X-PHIDE (2026)",
        "DOI":"10.1109/ACCESS.2026.3672690",
        "Reported_Setting":"Authors' custom dataset",
        "Accuracy":0.907,"AUROC":np.nan,"F1":0.912,
        "Reported_Note":"Paper also reports about 77.8% average accuracy across three independent test sets.",
        "Comparison_Status":"context_only_original_protocol"
    },
])
save_df(published_context,published_context_path)

human_rows=[]
for _,r in human_all.iterrows():
    human_rows.append({
        "Layer":"Human unsafe-action prediction",
        "Method":r["model"],
        "Paper_or_Source":"CRB direct same-protocol",
        "Comparison_Status":"same_protocol_direct",
        "Protocol":"30 onboarding selections; participant-grouped CV",
        "Accuracy":r.get("Accuracy_mean",np.nan),
        "Precision":r.get("Precision_mean",np.nan),
        "Recall_Sensitivity":r.get("Recall_mean",r.get("Sensitivity_mean",np.nan)),
        "Specificity":r.get("Specificity_mean",np.nan),
        "F1":r.get("F1_mean",np.nan),
        "MCC":r.get("MCC_mean",np.nan),
        "BalancedAccuracy":r.get("BalAcc_mean",r.get("BalancedAccuracy_mean",np.nan)),
        "AUROC":r.get("AUROC_mean",np.nan),
        "AUPRC":r.get("AUPRC_mean",np.nan),
        "Brier":r.get("Brier_mean",np.nan),
        "ECE":r.get("ECE_mean",np.nan),
        "Notes":"Primary robust human comparison; mean across 30 warm-up selections."
    })

tech_rows=[]
for _,r in comp_metrics.iterrows():
    tech_rows.append({
        "Layer":"Technical phishing-risk estimation",
        "Method":r["Method"],
        "Paper_or_Source":r["Paper"],
        "Comparison_Status":r["Reproduction_Status"],
        "Protocol":"Same PhiUSIIL domain-disjoint holdout",
        "Accuracy":r.get("Accuracy",np.nan),
        "Precision":r.get("Precision",np.nan),
        "Recall_Sensitivity":r.get("Recall_Sensitivity",np.nan),
        "Specificity":r.get("Specificity",np.nan),
        "F1":r.get("F1",np.nan),
        "MCC":r.get("MCC",np.nan),
        "BalancedAccuracy":r.get("BalancedAccuracy",np.nan),
        "AUROC":r.get("AUROC",np.nan),
        "AUPRC":r.get("AUPRC",np.nan),
        "Brier":r.get("Brier",np.nan),
        "ECE":r.get("ECE",np.nan),
        "Notes":r.get("Feature_Regime","")
    })

x_rows=[]
for _,r in xphide_metrics.iterrows():
    x_rows.append({
        "Layer":"Technical domain-only generalization",
        "Method":r["Method"],
        "Paper_or_Source":r["Paper"],
        "Comparison_Status":r["Reproduction_Status"],
        "Protocol":r["Protocol"],
        "Accuracy":r.get("Accuracy",np.nan),
        "Precision":r.get("Precision",np.nan),
        "Recall_Sensitivity":r.get("Recall_Sensitivity",np.nan),
        "Specificity":r.get("Specificity",np.nan),
        "F1":r.get("F1",np.nan),
        "MCC":r.get("MCC",np.nan),
        "BalancedAccuracy":r.get("BalancedAccuracy",np.nan),
        "AUROC":r.get("AUROC",np.nan),
        "AUPRC":r.get("AUPRC",np.nan),
        "Brier":r.get("Brier",np.nan),
        "ECE":r.get("ECE",np.nan),
        "Notes":r.get("Feature_Regime","")
    })

master=pd.DataFrame(human_rows+tech_rows+x_rows)
save_df(master,master_path)

# LaTeX export
cols=[
    "Layer","Method","Comparison_Status","Accuracy","Precision","Recall_Sensitivity",
    "Specificity","F1","MCC","BalancedAccuracy","AUROC","AUPRC","Brier","ECE"
]
master[cols].to_latex(
    master_tex,index=False,escape=True,
    float_format=lambda x:f"{x:.3f}" if isinstance(x,(float,np.floating)) else str(x),
    caption="Complete CRB comparison across human and technical layers. Direct numerical claims are restricted to methods evaluated under the same task and protocol.",
    label="tab:complete_comparison_v7"
)

note=f"""
FINAL COMPARISON INTERPRETATION — v7

HUMAN LAYER
- Primary result: DualAdaptiveCRB_LR is compared directly with CRB_Base_LR and PersonalizedCRB_LR
  under 30 independent onboarding selections and participant-grouped CV.
- Report AUROC, AUPRC, F1, MCC, Balanced Accuracy, Brier, and ECE from the robust summary.
- Do not compare human-layer scores numerically against URL-detector papers because the endpoint differs.

TECHNICAL LAYER
- Vajrobol2024_MI5_LR is the strongest same-dataset published configuration reproduced with its
  exact published five-feature set on the common domain-disjoint holdout.
- PhiUSIIL2024_Incremental_BestSecurity is a protocol-adapted reproduction: the three published
  incremental learners and paper-reported hyperparameters are used, but the standalone live URL
  similarity gate is omitted.
- Liu2026 models are method adaptations: the same classifier family is tested with training-only
  top-8 feature selection on the common holdout.
- XPHIDE2026_XGB_config_domainonly reproduces the paper's published XGBoost hyperparameters on
  the common domain-only feature representation; it is NOT the full X-PHIDE 8-feature pipeline
  because DNS/WHOIS-derived features are not present in standard PhiUSIIL.

CLAIM RULE
Use "outperformed" only inside a row group sharing the same task, dataset, split, and evaluation protocol.
Do NOT compare the numerical values of human unsafe-action prediction with URL-classification papers.
For method adaptations (e.g., X-PHIDE without its unavailable DNS/WHOIS features), say "adapted implementation"
and report the common-protocol result without implying an exact reproduction of the published pipeline.
Use "competitive with" / "compared with" for contextual published values.

Generated: {now()}
"""
summary_note.write_text(note,encoding="utf-8")

print("\n=== HUMAN ROBUST 30-SELECTION RESULTS ===")
display(pd.DataFrame(human_rows)[[
    "Method","AUROC","AUPRC","F1","MCC","BalancedAccuracy","Brier","ECE"
]].sort_values("AUPRC",ascending=False))

print("\n=== TECHNICAL PUBLISHED-COMPETITOR RESULTS ===")
display(pd.DataFrame(tech_rows)[[
    "Method","Comparison_Status","Accuracy","Precision","Recall_Sensitivity","Specificity",
    "F1","MCC","BalancedAccuracy","AUROC","AUPRC","Brier","ECE"
]].sort_values("AUPRC",ascending=False))

print("\n=== X-PHIDE / DOMAIN-ONLY COMMON-PROTOCOL RESULTS ===")
display(pd.DataFrame(x_rows)[[
    "Method","Comparison_Status","Accuracy","Precision","Recall_Sensitivity","Specificity",
    "F1","MCC","BalancedAccuracy","AUROC","AUPRC","Brier","ECE"
]].sort_values("AUPRC",ascending=False))

print("\n=== VERIFIED PUBLISHED VALUES — CONTEXT ONLY, NOT HEAD-TO-HEAD ===")
display(published_context)

print("[OK] CELL 13D complete all-metrics comparison exported.")
print("CSV:",master_path)
print("LaTeX:",master_tex)
print("Interpretation note:",summary_note)



=== HUMAN ROBUST 30-SELECTION RESULTS ===


,Method,AUROC,AUPRC,F1,MCC,BalancedAccuracy,Brier,ECE
1,DualAdaptiveCRB_LR,0.911174,0.777584,0.727087,0.621068,NaN,0.121133,0.106430
2,PersonalizedCRB_LR,0.908280,0.765574,0.726731,0.620852,NaN,0.123126,0.109631
0,CRB_Base_LR,0.887272,0.713797,0.692032,0.570886,NaN,0.138890,0.126146



=== TECHNICAL PUBLISHED-COMPETITOR RESULTS ===


,Method,Comparison_Status,Accuracy,Precision,Recall_Sensitivity,Specificity,F1,MCC,BalancedAccuracy,AUROC,AUPRC,Brier,ECE
0,CRBTech_RF_fixed,same_protocol_reference,0.999979,1.000000,0.999954,1.000000,0.999977,0.999958,0.999977,1.000000,1.000000,6.091256e-05,0.000946
1,CRBTech_HGB_fixed,same_protocol_reference,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.352218e-09,0.000031
2,CRBTech_XGB_fixed,same_protocol_reference,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.165820e-06,0.000017
3,CRBTech_Logistic_fixed,same_protocol_reference,0.999918,1.000000,0.999816,1.000000,0.999908,0.999833,0.999908,1.000000,1.000000,6.584402e-05,0.000252
4,Liu2026_ExtraTrees_adapted,adapted,0.999897,1.000000,0.999770,1.000000,0.999885,0.999792,0.999885,0.999999,0.999999,1.081390e-04,0.000124
5,Liu2026_AdaBoost_adapted,adapted,0.999897,1.000000,0.999770,1.000000,0.999885,0.999792,0.999885,0.999999,0.999998,6.107290e-02,0.236288
6,Vajrobol2024_MI5_LR,same_protocol_direct,0.993721,1.000000,0.985958,1.000000,0.992930,0.987363,0.992979,0.999998,0.999998,5.182376e-03,0.006503
7,Liu2026_GNB_adapted,adapted,0.999897,1.000000,0.999770,1.000000,0.999885,0.999792,0.999885,0.999999,0.999998,1.026612e-04,0.000099
8,Liu2026_SGD_adapted,adapted,0.999527,1.000000,0.998941,1.000000,0.999470,0.999043,0.999471,0.999997,0.999996,4.645885e-04,0.001943
9,PhiUSIIL2024_Incremental_BestSecurity,adapted,0.998703,0.999447,0.997652,0.999553,0.998548,0.997378,0.998603,0.999993,0.999992,8.010922e-03,0.068383



=== X-PHIDE / DOMAIN-ONLY COMMON-PROTOCOL RESULTS ===


,Method,Comparison_Status,Accuracy,Precision,Recall_Sensitivity,Specificity,F1,MCC,BalancedAccuracy,AUROC,AUPRC,Brier,ECE
0,Domain_XGB,same_protocol_reference,0.845875,0.932869,0.645986,0.970919,0.763364,0.678907,0.808453,0.876686,0.876914,0.117254,0.021137
1,XPHIDE2026_XGB_config_domainonly,adapted,0.840302,0.818371,0.751892,0.895608,0.783724,0.658982,0.823750,0.876085,0.876770,0.133425,0.116545
2,Domain_HGB,same_protocol_reference,0.846219,0.936633,0.643960,0.972746,0.763200,0.680242,0.808353,0.874061,0.872934,0.118546,0.019886
3,Domain_RF,same_protocol_reference,0.848925,0.912698,0.671673,0.959809,0.773852,0.681939,0.815741,0.869769,0.872047,0.117188,0.028851
4,Domain_Logistic,same_protocol_reference,0.834522,0.919400,0.624769,0.965737,0.743976,0.653795,0.795253,0.826369,0.833169,0.141224,0.086457



=== VERIFIED PUBLISHED VALUES — CONTEXT ONLY, NOT HEAD-TO-HEAD ===


,Study,DOI,Reported_Setting,Accuracy,AUROC,F1,Reported_Note,Comparison_Status
0,"Vajrobol, Gupta & Gaurav (2024)",10.1016/j.csa.2024.100044,Authors' original PhiUSIIL experiment,0.99970,NaN,NaN,Authors report 99.97% accuracy for MI + Logist...,context_only_original_protocol
1,Prasad & Chandra / PhiUSIIL (2024),10.1016/j.cose.2023.103545,Fully incremental / BestSecurity framework,0.99240,NaN,0.99338,Paper reports 99.24% fully incremental accurac...,context_only_original_protocol
2,Prasad & Chandra / PhiUSIIL (2024),10.1016/j.cose.2023.103545,90% pre-training / 10% incremental,0.99792,NaN,0.99817,Highest pre-training row reported in the sourc...,context_only_original_protocol
3,Liu et al. (2026),10.3233/ATDE250870,Authors' original PhiUSIIL ExtraTrees test,0.95740,0.99,0.96000,Best reported classifier in the paper; class-w...,context_only_original_protocol
4,Misiek & Hyla / X-PHIDE (2026),10.1109/ACCESS.2026.3672690,Authors' custom dataset,0.90700,NaN,0.91200,Paper also reports about 77.8% average accurac...,context_only_original_protocol


[OK] CELL 13D complete all-metrics comparison exported.
CSV: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/tables/complete_comparison_all_metrics_v7.csv
LaTeX: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/tables/tab_complete_comparison_all_metrics_v7.tex
Interpretation note: /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/reports/FINAL_COMPARISON_INTERPRETATION_v7.txt


In [ ]:
# CELL 13E — Final v7 competitor package
# Run after 13A–13D. Packages only manuscript-facing outputs; raw datasets are excluded.

zip_v7=ROOT/"CRB_Q1_Competitor_Comparison_v7.zip"
required_v7=[
    comp_metrics_path,
    comp_pred_path,
    comp_features_path,
    comp_protocol_path,
    xphide_metrics_path,
    xphide_pred_path,
    DIRS["tables"]/"xphide2026_published_context_v7.csv",
    human_all_path,
    DIRS["tables"]/"published_reported_metrics_context_v7.csv",
    master_path,
    master_tex,
    summary_note,
]

status_v7=pd.DataFrame([{
    "artifact":str(Path(p).relative_to(ROOT)),
    "exists":artifact_ok(p),
    "size_kb":round(Path(p).stat().st_size/1024,2) if Path(p).exists() else 0
} for p in required_v7])
display(status_v7)

if not status_v7["exists"].all():
    missing=status_v7.loc[~status_v7.exists,"artifact"].tolist()
    raise RuntimeError("Missing v7 competitor artifacts: "+", ".join(missing))

import zipfile
with zipfile.ZipFile(zip_v7,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in required_v7:
        p=Path(p)
        z.write(p,arcname=str(p.relative_to(ROOT)))

    # Also include the strongest existing human statistical outputs when present.
    extras=[
        DIRS["tables"]/"q1_incremental_value_tests.csv",
        DIRS["tables"]/"gee_hypothesis_tests.csv",
        DIRS["tables"]/"direct_head_to_head_comparisons_v6.csv",
        DIRS["tables"]/"literature_contextual_comparison_v6.csv",
        DIRS["tables"]/"domainonly_internal_metrics_v6_1.csv",
        DIRS["tables"]/"domainonly_external_zero_shot_v6_1.csv",
        DIRS["tables"]/"domainonly_external_recalibrated_v6_1.csv",
    ]
    for p in extras:
        if artifact_ok(p):
            z.write(p,arcname=str(p.relative_to(ROOT)))

print("\n[OK] FINAL v7 competitor package created.")
print(zip_v7)


,artifact,exists,size_kb
0,tables/published_competitors_v7_fullfeature_me...,True,3.55
1,processed/published_competitors_v7_fullfeature...,True,10826.59
2,tables/published_competitors_v7_selected_featu...,True,4.40
3,reports/published_competitors_v7_protocol.json,True,0.31
4,tables/xphide2026_domainonly_adapted_v7.csv,True,2.42
5,processed/xphide2026_domainonly_adapted_predic...,True,2286.15
6,tables/xphide2026_published_context_v7.csv,True,0.21
7,tables/human_dualadaptive_all_metrics_v7.csv,True,0.93
8,tables/published_reported_metrics_context_v7.csv,True,1.09
9,tables/complete_comparison_all_metrics_v7.csv,True,6.37



[OK] FINAL v7 competitor package created.
/content/drive/MyDrive/Cyber_Confidence_Q1_Extension/CRB_Q1_Competitor_Comparison_v7.zip


In [ ]:
# ============================================================
# CELL 14 — FINAL EFFICIENCY + DEPLOYMENT TRADE-OFF ANALYSIS
# ============================================================
#
# PURPOSE
# -------
# Adds deployment-oriented evidence to the v7 competitor comparison:
#
#   1. End-to-end training time
#   2. Model fitting time
#   3. Training-only feature-selection time
#   4. Median inference latency per 1,000 samples
#   5. Throughput (samples / second)
#   6. Number of deployed input features
#   7. Serialized deployment-model size (MB)
#   8. Existing AUROC / AUPRC / F1 / MCC / Brier / ECE
#   9. Explicit CRB Domain-XGB vs adapted X-PHIDE trade-off
#  10. Human Dual-Adaptive CRB relative improvements vs Base CRB
#
# IMPORTANT
# ---------
# - Does NOT overwrite previous v7 results.
# - Uses the SAME train/test partitions already established in Cells 13A/13C.
# - Full-feature and domain-only protocols remain separate.
# - Privacy is NOT claimed as an empirical advantage because privacy was
#   not experimentally benchmarked against the competing systems.
# ============================================================

import os
import gc
import time
import math
import json
import pickle
import tempfile
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression,
    PassiveAggressiveClassifier,
    SGDClassifier,
)
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import GroupShuffleSplit

print("=" * 78)
print("CELL 14 — EFFICIENCY + DEPLOYMENT TRADE-OFF ANALYSIS")
print("=" * 78)

# ------------------------------------------------------------
# 0. SAFETY / REQUIRED OBJECTS
# ------------------------------------------------------------

required_globals = [
    "DIRS", "ROOT", "SEED", "phi_comp",
    "Xtr_comp", "Xte_comp", "ytr_comp", "yte_comp",
    "numeric_cols_comp", "_score_from_estimator"
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(
        "Please run Cells 13A–13D first. Missing objects: "
        + ", ".join(missing_globals)
    )

EFF_TAG = "efficiency_v7"

eff_path = DIRS["tables"] / f"{EFF_TAG}_benchmark.csv"
tradeoff_path = DIRS["tables"] / f"{EFF_TAG}_xphide_tradeoff.csv"
human_gain_path = DIRS["tables"] / f"{EFF_TAG}_human_gain_summary.csv"
eff_tex_path = DIRS["tables"] / f"tab_{EFF_TAG}_benchmark.tex"
protocol_path = DIRS["reports"] / f"{EFF_TAG}_protocol.json"

# ------------------------------------------------------------
# 1. HELPERS
# ------------------------------------------------------------

def _serialized_size_mb(obj):
    """
    Compressed serialized deployment size.
    Same serialization rule is applied to every model/bundle.
    """
    tmp = None
    try:
        fd, tmp = tempfile.mkstemp(suffix=".joblib")
        os.close(fd)
        joblib.dump(obj, tmp, compress=3)
        return os.path.getsize(tmp) / (1024 ** 2)
    finally:
        if tmp and os.path.exists(tmp):
            os.remove(tmp)


def _slice_rows(X, n):
    n = min(n, len(X))
    if hasattr(X, "iloc"):
        return X.iloc[:n].copy()
    return X[:n]


def _benchmark_inference(score_function, X, n=10000, repeats=7):
    """
    Warm-up once, then return median and p95 latency.
    """
    Xb = _slice_rows(X, n)
    n_actual = len(Xb)

    # Warm-up
    _ = score_function(Xb)

    elapsed = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        _ = score_function(Xb)
        elapsed.append(time.perf_counter() - t0)

    elapsed = np.asarray(elapsed, dtype=float)

    med_sec = float(np.median(elapsed))
    p95_sec = float(np.quantile(elapsed, 0.95))

    # milliseconds required to process 1,000 observations
    ms_per_1000 = med_sec * 1000.0 * (1000.0 / n_actual)
    p95_ms_per_1000 = p95_sec * 1000.0 * (1000.0 / n_actual)

    return {
        "InferenceBenchmarkRows": n_actual,
        "Infer_ms_per_1000_median": ms_per_1000,
        "Infer_ms_per_1000_p95": p95_ms_per_1000,
        "Microseconds_per_sample": med_sec * 1e6 / n_actual,
        "Throughput_samples_per_sec": n_actual / max(med_sec, 1e-12),
    }


def _append_efficiency(
    rows,
    method,
    protocol,
    status,
    feature_regime,
    feature_count,
    selection_seconds,
    fit_seconds,
    deployment_object,
    score_function,
    X_test,
):
    inference = _benchmark_inference(score_function, X_test)

    rows.append({
        "Method": method,
        "Protocol": protocol,
        "Comparison_Status": status,
        "Feature_Regime": feature_regime,
        "FeatureCount": int(feature_count),
        "FeatureSelectionSeconds": float(selection_seconds),
        "ModelFitSeconds": float(fit_seconds),
        "EndToEndTrainSeconds": float(selection_seconds + fit_seconds),
        "ModelSizeMB": float(_serialized_size_mb(deployment_object)),
        **inference
    })


eff_rows = []

# ============================================================
# 2. FULL-FEATURE PHIUSIIL PROTOCOL
# ============================================================

print("\n[1/5] Benchmarking fixed CRB technical models...")

crb_models = {
    "CRBTech_Logistic_fixed": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=1500,
            class_weight="balanced",
            random_state=SEED
        ))
    ]),

    "CRBTech_RF_fixed": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=350,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=SEED
        ))
    ]),

    "CRBTech_HGB_fixed": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(
            max_iter=250,
            learning_rate=0.06,
            random_state=SEED
        ))
    ]),
}

if HAVE_XGB:
    crb_models["CRBTech_XGB_fixed"] = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=350,
            max_depth=6,
            learning_rate=0.06,
            subsample=0.85,
            colsample_bytree=0.85,
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1
        ))
    ])

for name, model in crb_models.items():
    print(" fitting", name)

    t0 = time.perf_counter()
    model.fit(Xtr_comp, ytr_comp)
    fit_seconds = time.perf_counter() - t0

    _append_efficiency(
        eff_rows,
        method=name,
        protocol="PhiUSIIL full-feature domain-disjoint",
        status="same_protocol_reference",
        feature_regime="All available numeric PhiUSIIL features",
        feature_count=len(numeric_cols_comp),
        selection_seconds=0.0,
        fit_seconds=fit_seconds,
        deployment_object=model,
        score_function=lambda X, m=model: _score_from_estimator(m, X),
        X_test=Xte_comp,
    )

    gc.collect()


# ============================================================
# 3. VAJROBOL 2024 — EXACT 5-FEATURE LR
# ============================================================

print("\n[2/5] Benchmarking Vajrobol et al. 2024...")

vaj5 = [
    "URLSimilarityIndex",
    "LineOfCode",
    "NoOfExternalRef",
    "NoOfImage",
    "NoOfSelfRef",
]

missing = [c for c in vaj5 if c not in phi_comp.columns]

if missing:
    print("[WARN] Vajrobol efficiency test skipped. Missing:", missing)

else:
    vaj_model_eff = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scale", MinMaxScaler()),
        ("lr", LogisticRegression(
            max_iter=2000,
            random_state=SEED
        ))
    ])

    Xtr_vaj = phi_comp.iloc[tr_idx_comp][vaj5]
    Xte_vaj = phi_comp.iloc[te_idx_comp][vaj5]

    t0 = time.perf_counter()
    vaj_model_eff.fit(Xtr_vaj, ytr_comp)
    fit_seconds = time.perf_counter() - t0

    _append_efficiency(
        eff_rows,
        method="Vajrobol2024_MI5_LR",
        protocol="PhiUSIIL full-feature domain-disjoint",
        status="same_protocol_direct",
        feature_regime="Published fixed five-feature configuration",
        feature_count=5,
        selection_seconds=0.0,
        fit_seconds=fit_seconds,
        deployment_object=vaj_model_eff,
        score_function=lambda X: _score_from_estimator(vaj_model_eff, X),
        X_test=Xte_vaj,
    )


# ============================================================
# 4. LIU 2026 — TRAINING-ONLY TOP-8 + CLASSIFIER FAMILY
# ============================================================

print("\n[3/5] Benchmarking Liu et al. 2026 adaptations...")

# Feature selection itself is part of training cost.
liu_select_start = time.perf_counter()

liu_imp_for_selection = SimpleImputer(strategy="median")
Xtr_liu_imp = liu_imp_for_selection.fit_transform(Xtr_comp)

liu_selector = ExtraTreesClassifier(
    n_estimators=240,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

liu_selector.fit(Xtr_liu_imp, ytr_comp)

top8_idx = np.argsort(liu_selector.feature_importances_)[::-1][:8]
liu8_eff = [numeric_cols_comp[i] for i in top8_idx]

liu_selection_seconds = time.perf_counter() - liu_select_start

Xtr8_eff = Xtr_comp[liu8_eff].copy()
Xte8_eff = Xte_comp[liu8_eff].copy()

liu_models_eff = {
    "Liu2026_AdaBoost_adapted": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", AdaBoostClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]),

    "Liu2026_SGD_adapted": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("clf", SGDClassifier(
            loss="log_loss",
            max_iter=1500,
            random_state=42
        ))
    ]),

    "Liu2026_ExtraTrees_adapted": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(
            n_estimators=350,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ))
    ]),

    "Liu2026_GNB_adapted": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", GaussianNB())
    ]),
}

for name, model in liu_models_eff.items():
    print(" fitting", name)

    t0 = time.perf_counter()
    model.fit(Xtr8_eff, ytr_comp)
    fit_seconds = time.perf_counter() - t0

    _append_efficiency(
        eff_rows,
        method=name,
        protocol="PhiUSIIL full-feature domain-disjoint",
        status="adapted",
        feature_regime="Training-only top-8 feature selection",
        feature_count=8,
        selection_seconds=liu_selection_seconds,
        fit_seconds=fit_seconds,
        deployment_object=model,
        score_function=lambda X, m=model: _score_from_estimator(m, X),
        X_test=Xte8_eff,
    )

    gc.collect()


# ============================================================
# 5. PHIUSIIL 2024 INCREMENTAL BESTSECURITY ADAPTATION
# ============================================================

print("\n[4/5] Benchmarking PhiUSIIL 2024 incremental adaptation...")

selection_start = time.perf_counter()

phi_eff_imp = SimpleImputer(strategy="median")
Xtr_phi_imp = phi_eff_imp.fit_transform(Xtr_comp)

importance_models = [
    RandomForestClassifier(
        n_estimators=160,
        n_jobs=-1,
        random_state=SEED
    ),
    DecisionTreeClassifier(random_state=SEED),
]

if HAVE_XGB:
    importance_models.append(
        XGBClassifier(
            n_estimators=160,
            max_depth=5,
            learning_rate=0.08,
            subsample=0.85,
            colsample_bytree=0.85,
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1
        )
    )

# Use LightGBM only if already available.
try:
    if pkgutil.find_loader("lightgbm") is not None:
        from lightgbm import LGBMClassifier

        importance_models.append(
            LGBMClassifier(
                n_estimators=160,
                random_state=SEED,
                verbosity=-1,
                n_jobs=-1
            )
        )
except Exception:
    pass

importance_vectors = []

for fm in importance_models:
    fm.fit(Xtr_phi_imp, ytr_comp)
    v = np.asarray(fm.feature_importances_, dtype=float)

    if v.sum() > 0:
        v = v / v.sum()

    importance_vectors.append(v)

mean_imp = np.mean(np.vstack(importance_vectors), axis=0)

positive_idx = np.where(mean_imp > 0)[0]

if len(positive_idx) < 5:
    positive_idx = np.arange(Xtr_comp.shape[1])

mi = mutual_info_classif(
    Xtr_phi_imp[:, positive_idx],
    ytr_comp,
    discrete_features="auto",
    random_state=SEED
)

nkeep = max(5, int(math.ceil(0.85 * len(positive_idx))))
local_order = np.argsort(mi)[::-1][:nkeep]
sel_idx = positive_idx[local_order]

phi_il_features_eff = [
    numeric_cols_comp[i] for i in sel_idx
]

phi_il_imp = SimpleImputer(strategy="median")
phi_il_scaler = MinMaxScaler()

Xtr_il_eff = phi_il_scaler.fit_transform(
    phi_il_imp.fit_transform(
        Xtr_comp[phi_il_features_eff]
    )
)

selection_seconds = time.perf_counter() - selection_start

bnb_eff = BernoulliNB(
    alpha=0.01,
    binarize=0.0,
    class_prior=None,
    fit_prior=False
)

pa_eff = PassiveAggressiveClassifier(
    C=5,
    fit_intercept=False,
    n_iter_no_change=50,
    n_jobs=-1,
    shuffle=True,
    random_state=SEED,
    max_iter=1000
)

sgd_eff = SGDClassifier(
    alpha=0.01,
    eta0=100,
    n_iter_no_change=50,
    n_jobs=1,
    random_state=SEED,
    max_iter=1000
)

order = np.random.default_rng(SEED + 1302).permutation(
    len(ytr_comp)
)

classes = np.array([0, 1], dtype=int)
chunk = 5000

fit_start = time.perf_counter()

for start in range(0, len(order), chunk):
    ii = order[start:start + chunk]

    yy = ytr_comp.iloc[ii].to_numpy()

    for model in (bnb_eff, pa_eff, sgd_eff):
        model.partial_fit(
            Xtr_il_eff[ii],
            yy,
            classes=classes
        )

fit_seconds = time.perf_counter() - fit_start

phi_bundle_eff = {
    "features": phi_il_features_eff,
    "imputer": phi_il_imp,
    "scaler": phi_il_scaler,
    "bnb": bnb_eff,
    "pa": pa_eff,
    "sgd": sgd_eff,
}


def _phi_incremental_score(X):
    Xt = phi_bundle_eff["scaler"].transform(
        phi_bundle_eff["imputer"].transform(
            X[phi_bundle_eff["features"]]
        )
    )

    pb = _score_from_estimator(phi_bundle_eff["bnb"], Xt)
    pp = _score_from_estimator(phi_bundle_eff["pa"], Xt)
    ps = _score_from_estimator(phi_bundle_eff["sgd"], Xt)

    return np.clip(
        (pb + pp + ps) / 3.0,
        1e-8,
        1 - 1e-8
    )


_append_efficiency(
    eff_rows,
    method="PhiUSIIL2024_Incremental_BestSecurity",
    protocol="PhiUSIIL full-feature domain-disjoint",
    status="adapted",
    feature_regime="Importance prefilter + training-only MI top-85%",
    feature_count=len(phi_il_features_eff),
    selection_seconds=selection_seconds,
    fit_seconds=fit_seconds,
    deployment_object=phi_bundle_eff,
    score_function=_phi_incremental_score,
    X_test=Xte_comp,
)

del Xtr_phi_imp, Xtr_il_eff, importance_models
gc.collect()


# ============================================================
# 6. DOMAIN-ONLY CRB vs X-PHIDE
# ============================================================

print("\n[5/5] Benchmarking domain-only CRB vs X-PHIDE...")

# Reconstruct exact v6.1 domain-only split if not currently in memory.
if "phi_dom" not in globals() or "dom_features" not in globals():
    dom_cache = (
        DIRS["processed"] /
        "phiusiil_domainonly_features_v6.parquet"
    )

    if not artifact_ok(dom_cache):
        raise RuntimeError(
            "Domain-only cache missing. Run Cell 4C first."
        )

    phi_dom = pd.read_parquet(dom_cache)

    dom_features = [
        c for c in phi_dom.columns
        if c not in {
            "url",
            "label",
            "registrable_domain"
        }
    ]

phi_dom_eff = phi_dom.dropna(
    subset=["registrable_domain", "label"]
).copy()

phi_dom_eff["registrable_domain"] = (
    phi_dom_eff["registrable_domain"]
    .astype(str)
    .str.lower()
    .str.strip()
)

phi_dom_eff = phi_dom_eff[
    phi_dom_eff["registrable_domain"].ne("")
].copy()

y_dom_eff = pd.to_numeric(
    phi_dom_eff["label"],
    errors="coerce"
).astype(int)

g_dom_eff = (
    phi_dom_eff["registrable_domain"]
    .astype(str)
)

X_dom_eff = (
    phi_dom_eff[dom_features]
    .replace([np.inf, -np.inf], np.nan)
)

gss_eff = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED + 606
)

tri_eff, tei_eff = next(
    gss_eff.split(
        X_dom_eff,
        y_dom_eff,
        g_dom_eff
    )
)

Xtrd_eff = X_dom_eff.iloc[tri_eff].copy()
Xted_eff = X_dom_eff.iloc[tei_eff].copy()

ytrd_eff = y_dom_eff.iloc[tri_eff].copy()
yted_eff = y_dom_eff.iloc[tei_eff].copy()

assert set(
    g_dom_eff.iloc[tri_eff]
).isdisjoint(
    set(g_dom_eff.iloc[tei_eff])
)

neg = int((ytrd_eff == 0).sum())
pos = int((ytrd_eff == 1).sum())

spw_eff = 2.0 * (neg / max(pos, 1))

domain_eff_models = {
    "Domain_Logistic": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=1500,
            class_weight="balanced",
            random_state=SEED
        ))
    ]),

    "Domain_RF": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=450,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=SEED
        ))
    ]),

    "Domain_HGB": Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            random_state=SEED
        ))
    ]),
}

if not HAVE_XGB:
    raise RuntimeError(
        "XGBoost is required for the final "
        "Domain-XGB / X-PHIDE comparison."
    )

domain_eff_models["Domain_XGB"] = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("clf", XGBClassifier(
        n_estimators=450,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1
    ))
])

domain_eff_models[
    "XPHIDE2026_XGB_config_domainonly"
] = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("clf", XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        scale_pos_weight=spw_eff,
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1
    ))
])

for name, model in domain_eff_models.items():
    print(" fitting", name)

    t0 = time.perf_counter()
    model.fit(Xtrd_eff, ytrd_eff)
    fit_seconds = time.perf_counter() - t0

    status = (
        "adapted"
        if name.startswith("XPHIDE")
        else "same_protocol_reference"
    )

    _append_efficiency(
        eff_rows,
        method=name,
        protocol="PhiUSIIL fair domain-only domain-disjoint",
        status=status,
        feature_regime="Common fair hostname/domain-only features",
        feature_count=len(dom_features),
        selection_seconds=0.0,
        fit_seconds=fit_seconds,
        deployment_object=model,
        score_function=lambda X, m=model: _score_from_estimator(m, X),
        X_test=Xted_eff,
    )

    gc.collect()


# ============================================================
# 7. MERGE WITH EXISTING PERFORMANCE RESULTS
# ============================================================

efficiency = pd.DataFrame(eff_rows)

performance_columns = [
    "Method",
    "Accuracy",
    "Precision",
    "Recall_Sensitivity",
    "Specificity",
    "F1",
    "MCC",
    "BalancedAccuracy",
    "AUROC",
    "AUPRC",
    "Brier",
    "ECE",
]

performance_frames = []

if "comp_metrics" in globals():
    cols = [
        c for c in performance_columns
        if c in comp_metrics.columns
    ]

    performance_frames.append(
        comp_metrics[cols].copy()
    )

if "xphide_metrics" in globals():
    cols = [
        c for c in performance_columns
        if c in xphide_metrics.columns
    ]

    performance_frames.append(
        xphide_metrics[cols].copy()
    )

if performance_frames:
    perf = pd.concat(
        performance_frames,
        ignore_index=True
    )

    perf = perf.drop_duplicates(
        subset=["Method"],
        keep="last"
    )

    efficiency = efficiency.merge(
        perf,
        on="Method",
        how="left"
    )


# Nice manuscript-facing order
ordered_columns = [
    "Protocol",
    "Method",
    "Comparison_Status",
    "FeatureCount",
    "EndToEndTrainSeconds",
    "FeatureSelectionSeconds",
    "ModelFitSeconds",
    "Infer_ms_per_1000_median",
    "Infer_ms_per_1000_p95",
    "Throughput_samples_per_sec",
    "ModelSizeMB",
    "Accuracy",
    "Precision",
    "Recall_Sensitivity",
    "Specificity",
    "F1",
    "MCC",
    "BalancedAccuracy",
    "AUROC",
    "AUPRC",
    "Brier",
    "ECE",
    "Feature_Regime",
]

ordered_columns = [
    c for c in ordered_columns
    if c in efficiency.columns
]

efficiency = efficiency[
    ordered_columns
].copy()

save_df(efficiency, eff_path)


# ============================================================
# 8. DOMAIN-XGB vs X-PHIDE: VALUE / TRADE-OFF SUMMARY
# ============================================================

if "xphide_metrics" in globals():

    xp = (
        xphide_metrics
        .set_index("Method")
    )

    required_pair = {
        "Domain_XGB",
        "XPHIDE2026_XGB_config_domainonly"
    }

    if required_pair.issubset(set(xp.index)):

        crb = xp.loc["Domain_XGB"]
        xph = xp.loc[
            "XPHIDE2026_XGB_config_domainonly"
        ]

        eff_pair = efficiency.set_index("Method")

        ce = eff_pair.loc["Domain_XGB"]
        xe = eff_pair.loc[
            "XPHIDE2026_XGB_config_domainonly"
        ]

        tradeoff = pd.DataFrame([{
            "Comparison":
                "Domain_XGB minus adapted X-PHIDE",

            "AUROC_CRB": crb["AUROC"],
            "AUROC_XPHIDE": xph["AUROC"],
            "AUROC_delta": (
                crb["AUROC"] - xph["AUROC"]
            ),

            "AUPRC_CRB": crb["AUPRC"],
            "AUPRC_XPHIDE": xph["AUPRC"],
            "AUPRC_delta": (
                crb["AUPRC"] - xph["AUPRC"]
            ),

            "Precision_CRB": crb["Precision"],
            "Precision_XPHIDE": xph["Precision"],
            "Precision_delta": (
                crb["Precision"] - xph["Precision"]
            ),

            "Recall_CRB":
                crb["Recall_Sensitivity"],
            "Recall_XPHIDE":
                xph["Recall_Sensitivity"],
            "Recall_delta": (
                crb["Recall_Sensitivity"]
                - xph["Recall_Sensitivity"]
            ),

            "F1_CRB": crb["F1"],
            "F1_XPHIDE": xph["F1"],
            "F1_delta": (
                crb["F1"] - xph["F1"]
            ),

            "Specificity_CRB": crb["Specificity"],
            "Specificity_XPHIDE": xph["Specificity"],
            "Specificity_delta": (
                crb["Specificity"]
                - xph["Specificity"]
            ),

            "Brier_CRB": crb["Brier"],
            "Brier_XPHIDE": xph["Brier"],
            "Brier_absolute_reduction": (
                xph["Brier"] - crb["Brier"]
            ),
            "Brier_relative_reduction_pct": (
                100.0
                * (xph["Brier"] - crb["Brier"])
                / max(xph["Brier"], 1e-12)
            ),

            "ECE_CRB": crb["ECE"],
            "ECE_XPHIDE": xph["ECE"],
            "ECE_absolute_reduction": (
                xph["ECE"] - crb["ECE"]
            ),
            "ECE_relative_reduction_pct": (
                100.0
                * (xph["ECE"] - crb["ECE"])
                / max(xph["ECE"], 1e-12)
            ),

            "TrainSeconds_CRB":
                ce["EndToEndTrainSeconds"],
            "TrainSeconds_XPHIDE":
                xe["EndToEndTrainSeconds"],

            # > 1 means CRB is faster
            "CRB_training_speedup_vs_XPHIDE": (
                xe["EndToEndTrainSeconds"]
                / max(
                    ce["EndToEndTrainSeconds"],
                    1e-12
                )
            ),

            "Inference_ms1000_CRB":
                ce["Infer_ms_per_1000_median"],
            "Inference_ms1000_XPHIDE":
                xe["Infer_ms_per_1000_median"],

            # > 1 means CRB inference is faster
            "CRB_inference_speedup_vs_XPHIDE": (
                xe["Infer_ms_per_1000_median"]
                / max(
                    ce["Infer_ms_per_1000_median"],
                    1e-12
                )
            ),

            "ModelSizeMB_CRB": ce["ModelSizeMB"],
            "ModelSizeMB_XPHIDE": xe["ModelSizeMB"],

            # > 1 means CRB is smaller
            "CRB_model_size_advantage_ratio": (
                xe["ModelSizeMB"]
                / max(
                    ce["ModelSizeMB"],
                    1e-12
                )
            ),
        }])

        save_df(tradeoff, tradeoff_path)

    else:
        tradeoff = pd.DataFrame()
        print(
            "[WARN] Domain-XGB/X-PHIDE pair "
            "not available for trade-off summary."
        )

else:
    tradeoff = pd.DataFrame()


# ============================================================
# 9. HUMAN DUAL-ADAPTIVE MULTI-METRIC GAIN SUMMARY
# ============================================================

human_file = (
    DIRS["tables"] /
    "dualadaptive_warm30_v6_overall_summary.csv"
)

if artifact_ok(human_file):

    h = pd.read_csv(human_file).set_index("model")

    if {
        "DualAdaptiveCRB_LR",
        "CRB_Base_LR"
    }.issubset(h.index):

        dual = h.loc["DualAdaptiveCRB_LR"]
        base = h.loc["CRB_Base_LR"]

        human_gain = pd.DataFrame([{
            "Comparison":
                "DualAdaptiveCRB_LR vs CRB_Base_LR",

            "AUROC_Base": base["AUROC_mean"],
            "AUROC_Dual": dual["AUROC_mean"],
            "AUROC_absolute_gain": (
                dual["AUROC_mean"]
                - base["AUROC_mean"]
            ),
            "AUROC_relative_gain_pct": (
                100.0
                * (
                    dual["AUROC_mean"]
                    - base["AUROC_mean"]
                )
                / base["AUROC_mean"]
            ),

            "AUPRC_Base": base["AUPRC_mean"],
            "AUPRC_Dual": dual["AUPRC_mean"],
            "AUPRC_absolute_gain": (
                dual["AUPRC_mean"]
                - base["AUPRC_mean"]
            ),
            "AUPRC_relative_gain_pct": (
                100.0
                * (
                    dual["AUPRC_mean"]
                    - base["AUPRC_mean"]
                )
                / base["AUPRC_mean"]
            ),

            "F1_Base": base["F1_mean"],
            "F1_Dual": dual["F1_mean"],
            "F1_absolute_gain": (
                dual["F1_mean"]
                - base["F1_mean"]
            ),
            "F1_relative_gain_pct": (
                100.0
                * (
                    dual["F1_mean"]
                    - base["F1_mean"]
                )
                / base["F1_mean"]
            ),

            "MCC_Base": base["MCC_mean"],
            "MCC_Dual": dual["MCC_mean"],
            "MCC_absolute_gain": (
                dual["MCC_mean"]
                - base["MCC_mean"]
            ),
            "MCC_relative_gain_pct": (
                100.0
                * (
                    dual["MCC_mean"]
                    - base["MCC_mean"]
                )
                / base["MCC_mean"]
            ),

            # Lower is better
            "Brier_Base": base["Brier_mean"],
            "Brier_Dual": dual["Brier_mean"],
            "Brier_relative_reduction_pct": (
                100.0
                * (
                    base["Brier_mean"]
                    - dual["Brier_mean"]
                )
                / base["Brier_mean"]
            ),

            # Lower is better
            "ECE_Base": base["ECE_mean"],
            "ECE_Dual": dual["ECE_mean"],
            "ECE_relative_reduction_pct": (
                100.0
                * (
                    base["ECE_mean"]
                    - dual["ECE_mean"]
                )
                / base["ECE_mean"]
            ),

            "WarmupSelections":
                int(dual.get(
                    "warmup_selections",
                    30
                )),
        }])

        save_df(
            human_gain,
            human_gain_path
        )

    else:
        human_gain = pd.DataFrame()

else:
    human_gain = pd.DataFrame()


# ============================================================
# 10. MANUSCRIPT-READY LATEX + PROTOCOL
# ============================================================

latex_columns = [
    "Method",
    "FeatureCount",
    "EndToEndTrainSeconds",
    "Infer_ms_per_1000_median",
    "Throughput_samples_per_sec",
    "ModelSizeMB",
    "AUROC",
    "AUPRC",
    "F1",
    "MCC",
    "Brier",
    "ECE",
]

latex_columns = [
    c for c in latex_columns
    if c in efficiency.columns
]

eff_tex_path.write_text(
    efficiency[latex_columns].to_latex(
        index=False,
        escape=True,
        float_format=lambda x: f"{x:.4f}"
    ),
    encoding="utf-8"
)

save_json({
    "generated_utc": now(),
    "protocol": (
        "Same v7 train/test partitions; "
        "wall-clock timing in one Colab runtime."
    ),
    "inference_benchmark_rows": 10000,
    "inference_repeats": 7,
    "inference_reported": (
        "median + p95 wall-clock latency"
    ),
    "model_size": (
        "joblib compressed serialized "
        "deployment object, compression level 3"
    ),
    "feature_selection_time": (
        "Included in end-to-end training time "
        "for training-selected competitor methods."
    ),
    "hardware_note": (
        "Timing is environment-dependent and "
        "should be reported together with "
        "the runtime environment."
    ),
    "runtime": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
    },
    "privacy_note": (
        "No empirical privacy superiority claim "
        "is made because privacy was not measured "
        "against competitor implementations."
    ),
}, protocol_path)


# ============================================================
# 11. DISPLAY FINAL OUTPUTS
# ============================================================

print("\n" + "=" * 78)
print("FINAL EFFICIENCY TABLE")
print("=" * 78)

display(
    efficiency.sort_values(
        ["Protocol", "AUPRC"],
        ascending=[True, False]
    )
)

if len(tradeoff):
    print("\n" + "=" * 78)
    print("DOMAIN-XGB vs X-PHIDE — PERFORMANCE + EFFICIENCY TRADE-OFF")
    print("=" * 78)
    display(tradeoff.T.rename(columns={0: "value"}))

if len(human_gain):
    print("\n" + "=" * 78)
    print("DUAL-ADAPTIVE CRB vs BASE CRB — MULTI-METRIC IMPROVEMENT")
    print("=" * 78)
    display(human_gain.T.rename(columns={0: "value"}))

print("\nSaved:")
print(" ", eff_path)
print(" ", tradeoff_path)
print(" ", human_gain_path)
print(" ", eff_tex_path)
print(" ", protocol_path)

print(
    "\n[OK] CELL 14 efficiency/deployment "
    "comparison completed."
)

CELL 14 — EFFICIENCY + DEPLOYMENT TRADE-OFF ANALYSIS

[1/5] Benchmarking fixed CRB technical models...
 fitting CRBTech_Logistic_fixed
 fitting CRBTech_RF_fixed
 fitting CRBTech_HGB_fixed
 fitting CRBTech_XGB_fixed

[2/5] Benchmarking Vajrobol et al. 2024...

[3/5] Benchmarking Liu et al. 2026 adaptations...
 fitting Liu2026_AdaBoost_adapted
 fitting Liu2026_SGD_adapted
 fitting Liu2026_ExtraTrees_adapted
 fitting Liu2026_GNB_adapted

[4/5] Benchmarking PhiUSIIL 2024 incremental adaptation...

[5/5] Benchmarking domain-only CRB vs X-PHIDE...
 fitting Domain_Logistic
 fitting Domain_RF
 fitting Domain_HGB
 fitting Domain_XGB
 fitting XPHIDE2026_XGB_config_domainonly

FINAL EFFICIENCY TABLE


,Protocol,Method,Comparison_Status,FeatureCount,EndToEndTrainSeconds,FeatureSelectionSeconds,ModelFitSeconds,Infer_ms_per_1000_median,Infer_ms_per_1000_p95,Throughput_samples_per_sec,...,Recall_Sensitivity,Specificity,F1,MCC,BalancedAccuracy,AUROC,AUPRC,Brier,ECE,Feature_Regime
13,PhiUSIIL fair domain-only domain-disjoint,Domain_XGB,same_protocol_reference,17,9.800529,0.000000,9.800529,8.127818,9.107550,1.230342e+05,...,0.645986,0.970919,0.763364,0.678907,0.808453,0.876686,0.876914,1.172538e-01,0.021137,Common fair hostname/domain-only features
14,PhiUSIIL fair domain-only domain-disjoint,XPHIDE2026_XGB_config_domainonly,adapted,17,9.493709,0.000000,9.493709,7.756226,7.983243,1.289287e+05,...,0.751892,0.895608,0.783724,0.658982,0.823750,0.876085,0.876770,1.334249e-01,0.116545,Common fair hostname/domain-only features
12,PhiUSIIL fair domain-only domain-disjoint,Domain_HGB,same_protocol_reference,17,11.631443,0.000000,11.631443,34.828332,35.565238,2.871226e+04,...,0.643960,0.972746,0.763200,0.680242,0.808353,0.874061,0.872934,1.185459e-01,0.019886,Common fair hostname/domain-only features
11,PhiUSIIL fair domain-only domain-disjoint,Domain_RF,same_protocol_reference,17,96.528933,0.000000,96.528933,68.384386,118.168167,1.462322e+04,...,0.671673,0.959809,0.773852,0.681939,0.815741,0.869769,0.872047,1.171877e-01,0.028851,Common fair hostname/domain-only features
10,PhiUSIIL fair domain-only domain-disjoint,Domain_Logistic,same_protocol_reference,17,2.280386,0.000000,2.280386,0.499817,0.523788,2.000732e+06,...,0.624769,0.965737,0.743976,0.653795,0.795253,0.826369,0.833169,1.412241e-01,0.086457,Common fair hostname/domain-only features
1,PhiUSIIL full-feature domain-disjoint,CRBTech_RF_fixed,same_protocol_reference,50,77.956009,0.000000,77.956009,19.132302,19.910150,5.226763e+04,...,0.999954,1.000000,0.999977,0.999958,0.999977,1.000000,1.000000,6.091256e-05,0.000946,All available numeric PhiUSIIL features
2,PhiUSIIL full-feature domain-disjoint,CRBTech_HGB_fixed,same_protocol_reference,50,12.086438,0.000000,12.086438,17.223084,17.813501,5.806161e+04,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.352218e-09,0.000031,All available numeric PhiUSIIL features
3,PhiUSIIL full-feature domain-disjoint,CRBTech_XGB_fixed,same_protocol_reference,50,11.865737,0.000000,11.865737,4.176792,4.219751,2.394182e+05,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.165820e-06,0.000017,All available numeric PhiUSIIL features
0,PhiUSIIL full-feature domain-disjoint,CRBTech_Logistic_fixed,same_protocol_reference,50,2.221641,0.000000,2.221641,1.092033,2.298166,9.157232e+05,...,0.999816,1.000000,0.999908,0.999833,0.999908,1.000000,1.000000,6.584402e-05,0.000252,All available numeric PhiUSIIL features
7,PhiUSIIL full-feature domain-disjoint,Liu2026_ExtraTrees_adapted,adapted,8,55.944763,40.056599,15.888163,20.262369,20.762167,4.935257e+04,...,0.999770,1.000000,0.999885,0.999792,0.999885,0.999999,0.999999,1.081390e-04,0.000124,Training-only top-8 feature selection



DOMAIN-XGB vs X-PHIDE — PERFORMANCE + EFFICIENCY TRADE-OFF


,value
Comparison,Domain_XGB minus adapted X-PHIDE
AUROC_CRB,0.876686
AUROC_XPHIDE,0.876085
AUROC_delta,0.0006
AUPRC_CRB,0.876914
AUPRC_XPHIDE,0.87677
AUPRC_delta,0.000144
Precision_CRB,0.932869
Precision_XPHIDE,0.818371
Precision_delta,0.114498



DUAL-ADAPTIVE CRB vs BASE CRB — MULTI-METRIC IMPROVEMENT


,value
Comparison,DualAdaptiveCRB_LR vs CRB_Base_LR
AUROC_Base,0.887272
AUROC_Dual,0.911174
AUROC_absolute_gain,0.023902
AUROC_relative_gain_pct,2.693821
AUPRC_Base,0.713797
AUPRC_Dual,0.777584
AUPRC_absolute_gain,0.063787
AUPRC_relative_gain_pct,8.936292
F1_Base,0.692032



Saved:
  /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/tables/efficiency_v7_benchmark.csv
  /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/tables/efficiency_v7_xphide_tradeoff.csv
  /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/tables/efficiency_v7_human_gain_summary.csv
  /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/tables/tab_efficiency_v7_benchmark.tex
  /content/drive/MyDrive/Cyber_Confidence_Q1_Extension/reports/efficiency_v7_protocol.json

[OK] CELL 14 efficiency/deployment comparison completed.
